# Clinical multiview analysis

This notebook prepares clinical data, loads pipeline outputs, and runs the paper's post-analysis and figures.

In [ ]:
# Resolve shared multiclust imports when this notebook is run from notebooks/<project>/.
from pathlib import Path
import os
import sys

_candidates = []
if os.environ.get("MULTICLUST_ROOT"):
    _candidates.append(Path(os.environ["MULTICLUST_ROOT"]).expanduser())
_candidates.extend([Path.cwd(), *Path.cwd().parents])
for _candidate in _candidates:
    if (_candidate / "Utils.py").exists() and (_candidate / "full_pipeline.py").exists():
        MULTICLUST_ROOT = _candidate.resolve()
        break
else:
    raise RuntimeError("Could not find multiclust root. Set MULTICLUST_ROOT to the Code/multiclust folder.")
if str(MULTICLUST_ROOT) not in sys.path:
    sys.path.insert(0, str(MULTICLUST_ROOT))
print(f"Using multiclust root: {MULTICLUST_ROOT}")


In [ ]:
# Import theme for plots
import theme
theme.apply_all()


# Data Preparation

### Load all requirements

In [ ]:
from Utils import *
register_notebook_context(globals())
from pathlib import Path
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import pickle
import dill
from SVM import *

#Import own functions
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list
import scipy.cluster.hierarchy as hierarchy

import seaborn as sns
from scipy.stats import f_oneway, kruskal, shapiro, levene, chi2_contingency, fisher_exact

from parea_classes import KMeansPyrea, ModelBasedClusteringPyrea, HierarchicalClusteringPyrea, EnsembleClusteringPyrea

# Save every Matplotlib figure in both raster and vector formats.



In [ ]:
# If need to reload after change
#importlib.reload(Utils)

### Load in data and metatable

In [ ]:
data = pd.read_csv("path/to/multiclust_data/merged_data.csv")
meta = pd.read_csv("path/to/multiclust_data/merged_meta.csv")



Change colums to character type when categorical.

In [ ]:
# Load the dictionary from an Excel file
dictionary_DIR = "path/to/project/Feature selection/Complete_dictionary.xlsx"
dict_df = pd.read_excel(dictionary_DIR)
ignore_keys = {-900, -300}

# Infer variables with explicit value-label mappings in the dictionary.
# Some continuous variables have numeric notes too, so this list is only a candidate list.

dict_df["dictionary_codes"] = dict_df["Notes"].apply(clinical_analysis_extract_dictionary_codes)
dict_df["InferredDataType"] = pd.Series(pd.NA, index=dict_df.index, dtype="object")
dict_df.loc[dict_df["dictionary_codes"].apply(len) >= 2, "InferredDataType"] = "Categorical"

cat_vars = dict_df.loc[dict_df["InferredDataType"] == "Categorical", "ElementName"].tolist()

# Only coerce truly discrete categorical variables to object/string.
# High-cardinality numeric variables, such as continuous cognition scores, must stay numeric;
# otherwise preprocessing will one-hot encode every observed value and create columns like cnb_*_nan.

cols_to_convert = [
    col for col in cat_vars
    if col in data.columns and clinical_analysis_should_convert_to_categorical(data[col])
]
cols_left_numeric = [col for col in cat_vars if col in data.columns and col not in cols_to_convert]

# Use object dtype so real NaN values remain missing rather than becoming the literal string "nan".
data[cols_to_convert] = data[cols_to_convert].astype("object")

print(f"Converted {len(cols_to_convert)} dictionary-categorical columns to object for encoding.")
print(f"Kept {len(cols_left_numeric)} dictionary-flagged columns numeric because they look continuous/high-cardinality.")
print("Examples kept numeric:", cols_left_numeric[:20])


### Remove community controls

In [ ]:

# Remove all rows where the 'phenotype' column has the value 'HC'
data_CHR = data[data['phenotype'] != 'HC'].reset_index(drop=True)
data_CC = data[data['phenotype'] == 'HC'].reset_index(drop=True)

data_all = data.copy()


### Split into discovery and test

In [ ]:
# 1. Load the Prescient ID list once:
prescient = pd.read_csv(
    "path/to/"
    "project/"
    "Data/Prescient_Client List_ DPACC ID with RA Name (ALL sites)_20_03_2025.txt",
    sep="\t"
)
prescient.columns = ["fkLocationID", "pkclientid", "src_subject_id", "RAname"]
prescient_ids = set(prescient["src_subject_id"])

# 2. Suppose your dict of modalities is called `data_dict`:
#    e.g. data_dict = {'fmri': fmri_df, 'eeg': eeg_df, ...}

# 3. Split it:
discovery_data, test_data = split_by_network(data_CHR, prescient_ids)
discovery_data_CC, test_data_CC = split_by_network(data_CC, prescient_ids)
discovery_data_all, test_data_all = split_by_network(data_all, prescient_ids)

# Now `discovery_dict['fmri']` has only Prescient rows for fMRI, 
# and `test_dict['fmri']` only the Pronet rows, etc.


In [ ]:
test_data_all

### Data cleaning

Handle missing values. 
First remove columns with more than 50% missing and then removing rows that have more than 50% missing. 

In [ ]:
cleaned_discovery, cleaned_test = remove_high_missing_data_split(
    discovery_data, test_data,
    subject_id_column='src_subject_id',
    col_threshold=0.5,
    row_threshold=0.5
)

display(cleaned_discovery)

cleaned_discovery_CC, cleaned_test_CC = remove_high_missing_data_split(
    discovery_data_CC, test_data_CC,
    subject_id_column='src_subject_id',
    col_threshold=0.5,
    row_threshold=0.5
)

cleaned_discovery_all, cleaned_test_all = remove_high_missing_data_split(
    discovery_data_all, test_data_all,
    subject_id_column='src_subject_id',
    col_threshold=0.5,
    row_threshold=0.5
)


In [ ]:
# Only keep relevant modalities
modalities_keep = ["Psychoticism", "Detachment", "Functioning", "Internalising", "Cognition"]

vars_to_keep = meta["ElementName"][meta["Modality"].isin(modalities_keep)].tolist()
# Filter the data if the columns are present in cleaned_discovery. Also include src_subject_id
vars_to_keep.append("src_subject_id")
vars_to_keep.append("phenotype")
cleaned_discovery = cleaned_discovery[cleaned_discovery.columns.intersection(vars_to_keep)]

cleaned_discovery_CC = cleaned_discovery_CC[cleaned_discovery_CC.columns.intersection(vars_to_keep)]
cleaned_discovery_all = cleaned_discovery_all[cleaned_discovery_all.columns.intersection(vars_to_keep)]


In [ ]:
cleaned_discovery_CC

### Collinearity 


This diagnostic first applies the same pipeline-style preprocessing used for clustering: dummy/ordinal coding for categorical variables, imputation, and scaling within each modality. It then checks collinearity on the resulting numeric modality matrices, so categorical predictors are included through their encoded features. It does not remove variables from the analysis.


#### check within clinical domains

In [ ]:



# Use the shared assess domain collinearity helper.
from Utils import clinical_analysis_assess_domain_collinearity


_, collinearity_subject_ids, collinearity_processed_modalities = preprocessing(
    cleaned_discovery,
    meta,
    subject_id_column="src_subject_id",
    col_threshold=0.5,
    row_threshold=0.5,
    skew_threshold=0.75,
    scaler_type="robust",
    modalities=modalities_keep,
    dummy_code_modalities=modalities_keep,
    mixed_categorical_modalities=[],
)

collinearity_results = clinical_analysis_assess_domain_collinearity(
    cleaned_discovery,
    meta=meta,
    preprocessed_modalities=collinearity_processed_modalities,
    domain_col="Modality",
    variable_col="ElementName",
    correlation_methods=("pearson", "spearman"),
    high_corr_threshold=0.90,
    moderate_corr_threshold=0.70,
)

collinearity_summary = collinearity_results["summary"]
domain_collinearity_correlation_pairs = collinearity_results["correlation_pairs"]
domain_collinearity_vif = collinearity_results["vif"]
domain_collinearity_condition_index = collinearity_results["condition_index"]
domain_collinearity_skipped_variables = collinearity_results["skipped_variables"]

collinearity_summary_path = "path/to/multiclust_data/domain_collinearity_summary.csv"
domain_collinearity_correlation_pairs_path = "path/to/multiclust_data/domain_collinearity_correlation_pairs.csv"
domain_collinearity_vif_path = "path/to/multiclust_data/domain_collinearity_vif_tolerance.csv"
domain_collinearity_condition_index_path = "path/to/multiclust_data/domain_collinearity_condition_index.csv"
domain_collinearity_skipped_variables_path = "path/to/multiclust_data/domain_collinearity_skipped_variables.csv"

collinearity_summary.to_csv(collinearity_summary_path, index=False)
domain_collinearity_correlation_pairs.to_csv(domain_collinearity_correlation_pairs_path, index=False)
domain_collinearity_vif.to_csv(domain_collinearity_vif_path, index=False)
domain_collinearity_condition_index.to_csv(domain_collinearity_condition_index_path, index=False)
domain_collinearity_skipped_variables.to_csv(domain_collinearity_skipped_variables_path, index=False)

print(f"Saved collinearity summary to: {collinearity_summary_path}")
print(f"Saved correlation-pair diagnostics to: {domain_collinearity_correlation_pairs_path}")
print(f"Saved VIF/tolerance diagnostics to: {domain_collinearity_vif_path}")
print(f"Saved condition-index diagnostics to: {domain_collinearity_condition_index_path}")
print(f"Saved skipped-variable log to: {domain_collinearity_skipped_variables_path}")

print("Table: domain-level collinearity summary")
display(collinearity_summary)

print("Table: pairwise correlation diagnostics, Pearson and Spearman, showing pairs with |r| >= 0.70")
display(domain_collinearity_correlation_pairs)

print("Table: variable-level VIF and tolerance diagnostics within each domain")
display(domain_collinearity_vif)

print("Table: domain-level condition index diagnostics")
display(domain_collinearity_condition_index)


## Save data

In [ ]:
# Save the cleaned dataframes to a CSV file
cleaned_discovery.to_csv(
    "path/to/"
    "multiclust_data/cleaned_discovery_data.csv",
    index=False
)

cleaned_test.to_csv(
    "path/to/multiclust_data/cleaned_test_data.csv",
    index=False
)

cleaned_discovery_all.to_csv(
    "path/to/multiclust_data/cleaned_discovery_data_all.csv",
    index=False
)


# Full pipeline results

## Load in results

The actual clustering pipeline is run on Spartan using the full pipeline scripts.

In [ ]:

# Load in the metrics.pkl from each training fold 
# Results_noDim_4opt_5fold_FIXED_highmut_onlyagree_min50_SVM_ARI
base = 'path/to/results/study_1/release/FinalRun_PCA99_extended100-100'
# Define results path
results_dir = os.path.join(base, "results/")
#intermediates_dir = os.path.join(base, "intermediates/")

# Make and define directory for plots etc.
plots_dir = os.path.join(base, "plots/")
os.makedirs(plots_dir, exist_ok=True)
# Find all fold directories that contain a metrics.pkl
metrics_files = sorted(glob.glob(os.path.join(results_dir, 'fold*/metrics.pkl')))

# Load metrics dynamically
metrics = {}

for metrics_file in metrics_files:
    fold_name = os.path.basename(os.path.dirname(metrics_file))  # e.g., 'fold0'
    with open(metrics_file, 'rb') as f:
        metrics[fold_name] = pickle.load(f)

# Access your metrics like this:
# metrics['fold0'], metrics['fold1'], etc.
print(f"Loaded metrics for folds: {list(metrics.keys())}")



In [ ]:

# Print best_params for each fold
for fold_name, data in metrics.items():
    print(f"{fold_name}: {data['best_params']}")


In [ ]:

# Collect parameters from folds
param_list = [data["best_params"] for fold_name, data in metrics.items()]
param_df = pd.DataFrame(param_list)

reconstructed = {}

for col in param_df.columns:
    s = param_df[col]

    # Column contains lists/tuples → per-position mode
    if s.apply(lambda x: isinstance(x, (list, tuple))).any():
        # turn each list into a row of a small DF
        tmp = pd.DataFrame(s.tolist())
        # mode per column, take first mode; convert back to list
        reconstructed[col] = tmp.mode().iloc[0].tolist()
    else:
        # scalar column → simple mode
        m = s.mode()
        reconstructed[col] = m.iloc[0] if not m.empty else None

print(reconstructed)


## Check quality of clusters

### Stability of best individual in each fold

In [ ]:
# Print best_fitness for each fold
for fold_name, data in metrics.items():
    print(f"Best individual fitness in {fold_name}: {data['best_fitness']}")


# Final on all data results

## Load in results and check quality

In [ ]:
# Load in the metrics.pkl from each training fold 
# Find all fold directories that contain a metrics.pkl
final_metric_file = sorted(glob.glob(os.path.join(results_dir, 'final/final_metrics.pkl')))

with open(final_metric_file[0], 'rb') as f:
    final_metrics = pickle.load(f)

print(f"Loaded metrics from: {final_metric_file[0]}")


In [ ]:
# Print final params

print("Final parameters used in the model:", final_metrics['final_params'])


In [ ]:
final_metrics.keys()


In [ ]:
# Print the final metrics

print("Final quality per view:", final_metrics['view_scores_per_view'])
print("Final mean quality across views:", final_metrics['view_quality_mean'])
print("Final quality of integrated cluster:", final_metrics['final_quality'])

print("Final cluster stability per view:", final_metrics['per_view_stabilities'])
print("Final mean cluster stability across views:", final_metrics['mean_view_stability'])
print("Final integrated cluster stability:", final_metrics['final_stability'])

cluster_pvalues = final_metrics.get('cluster_pvalues', {})
quality_pvalues = cluster_pvalues.get('pvalues_raw', {})
quality_pvalues_fdr = cluster_pvalues.get('pvalues_fdr', {})
ari_pvalues = cluster_pvalues.get('ari_stability', {}).get('pvalues_raw', {})
ari_pvalues_fdr = cluster_pvalues.get('ari_stability', {}).get('pvalues_fdr', {})

print("\nQuality p-values (raw):")
print("  Per modality:", quality_pvalues.get('modalities'))
print("  Final integrated:", quality_pvalues.get('final'))
print("Quality p-values (FDR):")
print("  Per modality:", quality_pvalues_fdr.get('modalities'))
print("  Final integrated:", quality_pvalues_fdr.get('with_final'))

print("\nARI stability p-values (raw):")
print("  Per modality:", ari_pvalues.get('modalities'))
print("  Final integrated:", ari_pvalues.get('final'))
print("ARI stability p-values (FDR):")
print("  Per modality:", ari_pvalues_fdr.get('modalities'))
print("  Final integrated:", ari_pvalues_fdr.get('with_final'))


In [ ]:
final_metrics['cluster_pvalues']


In [ ]:
print("Final mean cluster stability across views (MAT_CCC):", final_metrics['mean_view_stability_MAT_CCC'])
print("Final mean cluster stability across views (MAT_PAC):", final_metrics['mean_view_stability_MAT_PAC'])

print("Final integrated cluster stability (CCC):", final_metrics['final_stability_SUM_MAT_full']['CCC'])
print("Final integrated cluster stability (PAC):", final_metrics['final_stability_SUM_MAT_full']['PAC'])


In [ ]:
for i in range(0, len(final_metrics['per_view_stabilities_SUM_MAT_full'])):
    modality = final_metrics['per_view_stabilities_SUM_MAT_full'][i]
    print(f"Final cluster stability for modality {i} (CCC):", modality['CCC'])
    print(f"Final cluster stability for modality {i} (PAC):", modality['PAC'])


In [ ]:
final_metrics['final_reporting']


## Cluster validation sensitivity analyses

This section reports the optional no-cluster / continuum sensitivity analyses when they were enabled for the run profile and the exported sensitivity files are present.

In [ ]:
# Optional cluster-validation / continuum sensitivity analyses
from pathlib import Path
import json
import os
import re

import numpy as np
import pandas as pd


_repo_root = find_multiclust_repository_root(Path.cwd())
_profile_name = "clinical_paper"
_sensitivity_enabled, _profile_path = get_cluster_sensitivity_profile_setting(_repo_root, _profile_name)

if _sensitivity_enabled is False:
    print(
        "Cluster validation sensitivity analyses were not enabled for "
        f"profile '{_profile_name}' ({_profile_path})."
    )
else:
    _base_results_dir = Path(results_dir) if "results_dir" in globals() else None
    _sensitivity_dir = _base_results_dir / "final" / "cluster_validation_sensitivity" if _base_results_dir else None
    _summary_csv = _sensitivity_dir / "cluster_validation_sensitivity_summary.csv" if _sensitivity_dir else None
    _results_json = _sensitivity_dir / "cluster_validation_sensitivity_results.json" if _sensitivity_dir else None

    if _sensitivity_enabled is None:
        print("Could not infer whether the run profile enabled cluster validation sensitivity analyses.")
    elif _sensitivity_enabled:
        print(f"Cluster validation sensitivity analyses were enabled for profile '{_profile_name}'.")

    if not _summary_csv or not _summary_csv.exists():
        print(
            "No cluster validation sensitivity summary was found. "
            "Expected file:",
            _summary_csv,
        )
    else:
        cluster_validation_sensitivity_summary = pd.read_csv(_summary_csv)
        print("Cluster validation sensitivity summary:", _summary_csv)
        display_notebook_result(cluster_validation_sensitivity_summary)

        if _results_json and _results_json.exists():
            with open(_results_json, "r") as f:
                cluster_validation_sensitivity_results = json.load(f)

            print("Included sensitivity tests:")
            for _test in cluster_validation_sensitivity_results.get("included_tests", []):
                print("-", _test)

            cluster_validation_sensitivity_details = summarize_cluster_sensitivity_results(
                cluster_validation_sensitivity_results
            )
            print("Detailed sensitivity metrics:")
            display_notebook_result(cluster_validation_sensitivity_details)
        else:
            print("Detailed sensitivity JSON not found:", _results_json)

        _gap_tables = sorted((_sensitivity_dir / "tables").glob("*/*_gap_statistic.csv"))
        cluster_validation_gap_tables = {}
        if _gap_tables:
            print("Gap-statistic tables:")
            for _gap_table in _gap_tables:
                _gap_name = _gap_table.parent.name
                cluster_validation_gap_tables[_gap_name] = pd.read_csv(_gap_table)
                print("-", _gap_table)
                display_notebook_result(cluster_validation_gap_tables[_gap_name])

        _plot_files = sorted((_sensitivity_dir / "plots").glob("*.pdf")) + sorted((_sensitivity_dir / "plots").glob("*.png"))
        if _plot_files:
            print("Sensitivity plot files:")
            for _plot_file in _plot_files:
                print("-", _plot_file)

## Matrix plots

#### Matrix plots final labels

In [ ]:
# Grab consensus
M = final_metrics['final_stability_SUM_MAT_full']['consensus']
M = np.asarray(M, dtype=float)

# Safety checks
assert M.ndim == 2 and M.shape[0] == M.shape[1], "Consensus must be square."
M = (M + M.T) / 2.0           # enforce symmetry (just in case)
np.fill_diagonal(M, 1.0)      # conventional

# Distance from consensus
D = 1.0 - M
dvec = squareform(D, checks=False)

# Hierarchical clustering (match your CCC linkage_method if you want)
Z = linkage(dvec, method="average")

# Leaf order and reordered matrix
order = leaves_list(Z)
M_ord = M[np.ix_(order, order)]

# Plot
plt.figure()
plt.imshow(M_ord, aspect='auto')
#plt.title("Consensus matrix (reordered by hierarchical clustering)")
plt.xlabel("Subjects")
plt.ylabel("Subjects")
plt.colorbar(label="Consensus")
clinical_analysis_save_figure_png_pdf(
    plt.gcf(),
    os.path.join(plots_dir, "consensus_matrices", "final_consensus_hierarchical.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

diag = final_metrics['final_stability_SUM_MAT_full']
M = np.asarray(diag['consensus'], dtype=float)
M = (M + M.T) / 2.0
np.fill_diagonal(M, 1.0)

union_ids = np.asarray(diag['union_ids'])

# --- you need the IDs that match final_labels order ---
final_labels = np.asarray(final_metrics['final_labels'])

# Use the first retained modality; all final-metric modality frames share subject order.
reference_modality = next(iter(final_metrics['data']))
final_ids = np.asarray(final_metrics['data'][reference_modality]['src_subject_id'])

# Map id -> label
id2lab = dict(zip(final_ids, final_labels))

# Align labels to consensus matrix order
labels_aligned = np.array([id2lab[sid] for sid in union_ids])

# Now sort consensus by aligned labels
order = np.lexsort((np.arange(len(labels_aligned)), labels_aligned))
M_ord = M[np.ix_(order, order)]
labels_ord = labels_aligned[order]

plt.figure()
plt.imshow(M_ord, aspect='auto')
plt.title("Consensus matrix (sorted by final labels, aligned to union_ids)")
plt.colorbar(label="Consensus")
clinical_analysis_save_figure_png_pdf(
    plt.gcf(),
    os.path.join(plots_dir, "consensus_matrices", "final_consensus_sorted_by_labels.png"),
    dpi=300,
    bbox_inches="tight",
)
plt.show()


In [ ]:
M = final_metrics['final_stability_SUM_MAT_full']['consensus']
M = np.asarray(M, float)
iu = np.triu_indices_from(M, k=1)
vals = M[iu]

print("fraction <0.1:", np.mean(vals < 0.1))
print("fraction >0.9:", np.mean(vals > 0.9))
print("PAC (0.1..0.9):", np.mean((vals > 0.1) & (vals < 0.9)))
print("mean:", np.mean(vals), "median:", np.median(vals))


#### Matrix plots per view

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, leaves_list, cut_tree
from scipy.cluster import hierarchy


indiv_labels_TEST = []
for i in range(0, len(final_metrics['per_view_stabilities_SUM_MAT_full'])):
    modality = final_metrics['per_view_stabilities_SUM_MAT_full'][i]
    mod_name = list(final_metrics['data'].keys())[i]
    print(f"Processing modality {mod_name} with stability metrics: CCC={modality['CCC']}, PAC={modality['PAC']}")
    M = np.asarray(modality['consensus'], dtype=float)

    # Safety checks
    assert M.ndim == 2 and M.shape[0] == M.shape[1], "Consensus must be square."
    M = (M + M.T) / 2.0           # enforce symmetry (just in case)
    np.fill_diagonal(M, 1.0)      # conventional

    # Distance from consensus
    D = 1.0 - M
    dvec = squareform(D, checks=False)

    # Hierarchical clustering (match your CCC linkage_method if you want)
    Z = linkage(dvec, method="average")

    indiv_labels_TEST.append(mod_name)


    # Leaf order and reordered matrix
    order = leaves_list(Z)
    M_ord = M[np.ix_(order, order)]

    # Plot
    plt.figure()
    plt.imshow(M_ord, aspect='auto')
    #plt.title(f"Consensus matrix for {mod_name} (reordered by hierarchical clustering)")
    plt.xlabel("Subjects")
    plt.ylabel("Subjects")
    #plt.colorbar(label="Consensus")
    clinical_analysis_save_figure_png_pdf(
        plt.gcf(),
        os.path.join(plots_dir, "consensus_matrices", f"{mod_name}_consensus_hierarchical.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
for i in range(0, len(final_metrics['per_view_stabilities_SUM_MAT_full'])):
    modality = final_metrics['per_view_stabilities_SUM_MAT_full'][i]
    mod_name = list(final_metrics['data'].keys())[i]
    M = np.asarray(modality['consensus'], dtype=float)
    M = (M + M.T) / 2.0
    np.fill_diagonal(M, 1.0)

    labels = final_metrics['individual_labels'][i]

    # Align these labels to this modality's consensus-matrix ID order.
    final_ids = np.asarray(final_metrics['data'][mod_name]['src_subject_id'])

    # Map id -> label
    id2lab = dict(zip(final_ids, labels))

    # Align labels to this modality's consensus matrix order.
    union_ids_view = np.asarray(modality['union_ids'])
    labels_aligned = np.array([id2lab[sid] for sid in union_ids_view])

    # Now sort consensus by aligned labels
    order = np.lexsort((np.arange(len(labels_aligned)), labels_aligned))
    M_ord = M[np.ix_(order, order)]
    labels_ord = labels_aligned[order]


    plt.figure()
    plt.imshow(M_ord, aspect='auto')
    plt.title(f"Consensus matrix for {mod_name} (reordered by modality  cluster labels)")
    plt.xlabel("Subjects (sorted by label)")
    plt.ylabel("Subjects (sorted by label)")
    plt.colorbar(label="Consensus")
    clinical_analysis_save_figure_png_pdf(
        plt.gcf(),
        os.path.join(plots_dir, "consensus_matrices", f"{mod_name}_consensus_sorted_by_modality_labels.png"),
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


## scatter plots

### Individual labels

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np
import umap  # pip install umap-learn

#Create directory for final latent space plots
os.makedirs(os.path.join(plots_dir, "merged_tsne/"), exist_ok=True)

data = final_metrics['ae_res']                     # dict: modality -> DataFrame
labels_list = final_metrics['individual_labels']  # list of label vectors

modality_names = list(data.keys())

if len(labels_list) != len(modality_names):
    raise ValueError(
        f"labels_list has {len(labels_list)} items but there are "
        f"{len(modality_names)} modalities for {fold_name}."
    )

for i, modality in enumerate(modality_names):
    print(f"  Modality: {modality}")
    df = data[modality]
    X = np.asarray(df['final_latent'])
    y = np.asarray(labels_list[i])

    if len(y) != len(X):
        raise ValueError(
            f"Label vector length ({len(y)}) does not match samples ({len(X)}) "
            f"for {fold_name}, modality {modality}."
        )

    # ----- color mapping -----
    classes = np.unique(y)
    palette = sns.color_palette(n_colors=len(classes))  # <-- uses current theme palette
    color_map = {cls: palette[j] for j, cls in enumerate(classes)}
    colors = [color_map[cls] for cls in y]

    # ----- dimensionality reductions -----
    pca = PCA(n_components=2)
    pca_proj = pca.fit_transform(X)

    tsne_2d = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_proj_2d = tsne_2d.fit_transform(X)

    umap_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
    umap_proj_2d = umap_2d.fit_transform(X)

    # ----- plots -----
    fig = plt.figure(figsize=(22, 5))

    ax1 = fig.add_subplot(1, 4, 1)
    ax1.scatter(pca_proj[:, 0], pca_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
    ax1.set_title(f'PCA (2D) — {modality}')

    ax2 = fig.add_subplot(1, 4, 2)
    ax2.scatter(tsne_proj_2d[:, 0], tsne_proj_2d[:, 1], c=colors, alpha=0.6, edgecolors="none")
    ax2.set_title(f't-SNE (2D) — {modality}')

    ax4 = fig.add_subplot(1, 4, 3)
    ax4.scatter(umap_proj_2d[:, 0], umap_proj_2d[:, 1], c=colors, alpha=0.6, edgecolors="none")
    ax4.set_title(f'UMAP (2D) — {modality}')

    # unified legend
    handles = [Line2D([0], [0], marker='o', linestyle='', color=color_map[cls], label=str(cls))
                for cls in classes]
    for ax in [ax1, ax2, ax4]:
        ax.legend(handles=handles, title='Label', loc='best', frameon=True)

    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(plots_dir, "merged_tsne/", f"{fold_name}_{modality}_merged_individual_tsne.png"))
    plt.show()


### Final labels

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np


data = final_metrics['ae_res']                       # dict: modality -> DataFrame
y = np.asarray(final_metrics['final_labels'])  # one label vector for the whole fold

# Build a stable color map for this fold (consistent colors across all modalities)
classes = np.unique(y)
palette = sns.color_palette(n_colors=len(classes))  # <-- uses current theme palette
color_map = {cls: palette[j] for j, cls in enumerate(classes)}
colors = [color_map[cls] for cls in y]

modality_names = list(data.keys())

for modality in modality_names:
    print(f"  Modality: {modality}")
    df = data[modality]
    X = np.asarray(df['final_latent'])

    if len(y) != len(X):
        raise ValueError(
            f"Label vector length ({len(y)}) does not match samples ({len(X)}) "
            f"for {fold_name}, modality {modality}. Ensure correct alignment."
        )

    colors = [color_map[cls] for cls in y]

    # --- Dimensionality reductions ---
    pca = PCA(n_components=2)
    pca_proj = pca.fit_transform(X)

    tsne = TSNE(n_components=2, perplexity=30, random_state=42)
    tsne_proj = tsne.fit_transform(X)

    # --- Plot side-by-side ---
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].scatter(pca_proj[:, 0], pca_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
    axes[0].set_title(f'PCA projection — {modality} ({fold_name})')

    axes[1].scatter(tsne_proj[:, 0], tsne_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
    axes[1].set_title(f't-SNE projection — {modality} ({fold_name})')

    # Legend
    handles = [
        Line2D([0], [0], marker='o', linestyle='', color=color_map[cls], label=str(cls))
        for cls in classes
    ]
    for ax in axes:
        ax.legend(handles=handles, title='Label', loc='best', frameon=True)

    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(plots_dir, "merged_tsne/", f"{modality}_merged_final_tsne.png"))
    plt.show()


### Plot final labels across modalities

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import numpy as np


data = final_metrics['ae_res']                       # modality -> dict with 'final_latent'
y = np.asarray(final_metrics['final_labels'])  # labels aligned to subjects

modality_names = list(data.keys())
latent_blocks = []

for modality in modality_names:
    X_mod = np.asarray(data[modality]['final_latent'])
    if X_mod.shape[0] != len(y):
        raise ValueError(
            f"Label vector length ({len(y)}) != samples ({X_mod.shape[0]}) "
            f"for {fold_name}, modality {modality}"
        )
    latent_blocks.append(X_mod)

# concatenate [n_samples, sum(latent_dims)]
X_all = np.hstack(latent_blocks)

classes = np.unique(y)
palette = sns.color_palette(n_colors=len(classes))  # <-- uses current theme palette
color_map = {cls: palette[j] for j, cls in enumerate(classes)}
colors = [color_map[cls] for cls in y]

# --- PCA projection over all modalities ---
pca = PCA(n_components=2)
pca_proj = pca.fit_transform(X_all)

# --- t-SNE projection over all modalities ---
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_proj = tsne.fit_transform(X_all)

# --- UMAP projection over all modalities ---
umap_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
umap_proj = umap_2d.fit_transform(X_all)

# --- Plot side-by-side ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(pca_proj[:, 0], pca_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
axes[0].set_title(f"PCA — all modalities ({fold_name})")

axes[1].scatter(tsne_proj[:, 0], tsne_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
axes[1].set_title(f"t-SNE — all modalities ({fold_name})")

axes[2].scatter(umap_proj[:, 0], umap_proj[:, 1], c=colors, alpha=0.6, edgecolors="none")
axes[2].set_title(f"UMAP — all modalities ({fold_name})")

handles = [
    Line2D([0], [0], marker='o', linestyle='', color=color_map[cls], label=str(cls))
    for cls in classes
]
for ax in axes:
    ax.legend(handles=handles, title='Label', loc='best', frameon=True)

plt.tight_layout()
clinical_analysis_save_figure_png_pdf(fig, os.path.join(plots_dir, "merged_tsne/", f"merged_final_alldata_tsne.png"))
plt.show()




### Show the final quality and stability metrics

In [ ]:
print('Mean quality across modalities',final_metrics['view_quality_mean'])
print('Final integrated cluster quality',final_metrics['final_quality'])
print('Mean stability across modalities',final_metrics['mean_view_stability'])
print('Final integrated cluster stability',final_metrics['final_stability'])


## Differences features

### Differences in original features based on modality labels

In [ ]:

from sklearn.feature_selection import f_classif
from Utils import display_feature_name
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

out_dir = os.path.join(plots_dir, "merged_feature_differences")
os.makedirs(out_dir, exist_ok=True)

top_k = 30
sample_name = "discovery"
label_scope = "individual_labels"

for mod_num, (modality, df) in enumerate(final_metrics['data'].items()):
    print(f"\n=== {sample_name} / {label_scope}: {modality} ===")

    feature_df = df.drop(columns=['src_subject_id'])
    X = feature_df.values
    clusters = np.asarray(final_metrics['individual_labels'][mod_num])
    feature_names = feature_df.columns

    if len(clusters) != len(df):
        raise ValueError(f"{modality}: label length ({len(clusters)}) != data rows ({len(df)})")
    if len(pd.unique(clusters)) < 2:
        print(f"Skipping {modality}: fewer than two individual labels.")
        continue

    f_vals, _ = f_classif(X, clusters)
    f_df = (
        pd.DataFrame({'feature': feature_names, 'f_value': f_vals})
        .assign(display_feature=lambda d: d['feature'].map(display_feature_name))
        .sort_values('f_value', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x='f_value', y='display_feature', data=f_df.head(top_k), palette=sns.color_palette(n_colors=top_k), ax=ax)
    ax.set_title(f"{modality} — Top {top_k} Discriminative Features (ANOVA F-value; discovery individual labels)")
    ax.set_xlabel("F-value")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{sample_name}_{label_scope}_{modality}_top_features_bar.png"))
    plt.show()

    top_features = f_df['feature'].head(top_k).tolist()
    max_cols = 3
    n_rows = int(np.ceil(len(top_features) / max_cols))
    fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
    axes = np.array(axes).reshape(-1)

    cluster_order = [str(x) for x in np.unique(clusters)]
    df_cc = None
    cc_available = df_cc is not None
    if cc_available:
        df_cc = df_cc.copy().reindex(columns=df.columns, fill_value=np.nan)
    group_order = cluster_order + (["CC"] if cc_available else [])
    pal = sns.color_palette(n_colors=len(group_order))

    for i, feat in enumerate(top_features):
        ax = axes[i]
        plot_df = pd.DataFrame({
            'group': pd.Series(clusters).astype(str),
            'value': df[feat].values,
        }).dropna()
        if cc_available and feat in df_cc.columns:
            cc_plot_df = pd.DataFrame({'group': 'CC', 'value': df_cc[feat].values}).dropna()
            plot_df = pd.concat([plot_df, cc_plot_df], ignore_index=True)
        sns.boxplot(data=plot_df, x='group', y='value', order=group_order, palette=pal, ax=ax)
        sns.stripplot(data=plot_df, x='group', y='value', order=group_order, color='black', size=4, jitter=True, alpha=0.5, ax=ax)
        ax.set_title(display_feature_name(feat), fontsize=14)
        ax.set_xlabel("", fontsize=22)
        ax.set_ylabel("", fontsize=22)
        ax.tick_params(axis='both', labelsize=14)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{modality} — Distribution of Top Features by Discovery Individual Labels\n(with individual data points)",
        y=1.02, fontsize=18
    )
    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{sample_name}_{label_scope}_{modality}_top_features_with_cc.png"))
    plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from Utils import display_feature_name

# =========================
# PCA aggregated plots per modality + variance explained (first 5 PCs)
# - Respects your existing global theme (NO sns.set_theme here)
# - PC1 by cluster: violin (quartiles) + jitter + median marker + n labels
# - Variance plot: explained variance ratio for PC1..PC5
# - Optional: PC1 loadings (top +/- contributors)
# =========================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

loadings_dir = os.path.join(out_dir, "pc1_loadings/")
os.makedirs(loadings_dir, exist_ok=True)

variance_dir = os.path.join(out_dir, "variance/")
os.makedirs(variance_dir, exist_ok=True)

mod_num = 0
for modality, df in final_metrics["data"].items():
    print(f"\n=== PCA Aggregated Plots — Modality: {modality} ===")

    # --- Extract X and cluster labels ---
    feature_df = df.drop(columns=["src_subject_id"])
    feature_names = feature_df.columns.to_list()

    X = feature_df.values
    clusters = np.asarray(final_metrics["individual_labels"][mod_num])

    # Stable cluster order (customize if you want a specific ordering)
    cluster_order = np.sort(pd.unique(clusters))

    # --- Standardize ---
    Xz = StandardScaler().fit_transform(X)

    # --- PCA for variance (first 5 components) ---
    n_pcs = min(5, Xz.shape[1])  # cannot exceed number of features
    pca_var = PCA(n_components=n_pcs, random_state=0)
    pca_var.fit(Xz)

    evr = pca_var.explained_variance_ratio_
    cum_evr = np.cumsum(evr)

    # --- Also compute PC scores (at least PC1) ---
    pc_scores = pca_var.transform(Xz)  # shape: (n_samples, n_pcs)
    pc1 = pc_scores[:, 0]
    evr1 = float(evr[0])
    print(f"PC1 EVR: {evr1:.2%}")

    plot_df = pd.DataFrame({"cluster": clusters, "PC1": pc1})

    # -------------------------
    # 1) PC1 by cluster (publication-ready, respects global theme)
    # -------------------------
    fig, ax = plt.subplots(figsize=(10.5, 6.5))


    sns.violinplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        palette=sns.color_palette(n_colors=len(cluster_order)),
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Median marker per cluster
    medians = plot_df.groupby("cluster")["PC1"].median()
    x_positions = np.arange(len(cluster_order))
    ax.scatter(
        x=x_positions,
        y=[medians.loc[c] for c in cluster_order],
        s=180,
        marker="_",
        linewidths=3
    )

    # Annotate n per cluster near the bottom
    counts = plot_df["cluster"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, c in enumerate(cluster_order):
        ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="top", fontsize=12)

    ax.set_xlabel("Cluster", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)

    # Light grid for readability; remove if your global theme already handles grids
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{modality}_PC1_violin_pubready.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 2) Variance explained (PC1..PC5): bar + cumulative line
    # -------------------------
    figv, axv = plt.subplots(figsize=(10, 5.5))

    pcs_idx = np.arange(1, n_pcs + 1)
    axv.bar(pcs_idx, evr)  # uses your global matplotlib color cycle
    axv.plot(pcs_idx, cum_evr, marker="o")

    axv.set_xticks(pcs_idx)
    axv.set_xlabel("Principal Component")
    axv.set_ylabel("Explained variance ratio")
    axv.set_title(f"{modality} — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

    axv.set_ylim(0, max(0.25, evr.max() * 1.2))  # keeps plot readable if EVR is small/large
    axv.grid(axis="y", alpha=0.15)
    sns.despine(ax=axv)

    figv.tight_layout()
    clinical_analysis_save_figure_png_pdf(figv, os.path.join(variance_dir, f"{modality}_variance_top{n_pcs}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 3) Optional: PC1 feature loadings (top +/- contributors)
    # -------------------------
    loadings = pca_var.components_[0]  # PC1 loadings
    load_df = pd.DataFrame({"feature": feature_names, "loading": loadings})
    load_df["display_feature"] = load_df["feature"].map(display_feature_name)

    top_n = min(15, len(feature_names) // 2) if len(feature_names) >= 2 else 1
    top_pos = load_df.sort_values("loading", ascending=False).head(top_n)
    top_neg = load_df.sort_values("loading", ascending=True).head(top_n)
    load_plot_df = pd.concat([top_neg, top_pos], axis=0)

    fig2, ax2 = plt.subplots(figsize=(10.5, 7.5))
    sns.barplot(data=load_plot_df, x="loading", y="display_feature", ax=ax2)
    ax2.axvline(0, linewidth=1)

    ax2.set_title(f"{modality} — PC1 Feature Loadings (Top ±{top_n})", pad=12)
    ax2.set_xlabel("PC1 loading")
    ax2.set_ylabel("")

    ax2.grid(axis="x", alpha=0.15)
    sns.despine(ax=ax2)

    fig2.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig2, os.path.join(loadings_dir, f"{modality}_PC1_loadings_top_pm{top_n}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    mod_num += 1


#### Plot with CC

In [ ]:
# Apply the fitted discovery CHR preprocessing to discovery CC.
# This keeps CHR-vs-CC comparisons in the same reference feature space.
preproc_discovery = final_metrics['preprocessing_details']

cc_df_discovery = cleaned_discovery_CC.copy()
ae_data_cc_discovery, subject_id_list_cc_discovery, dict_final_cc = apply_preprocessing_to_new_data(
    cc_df_discovery,
    meta,
    preproc_discovery,
    subject_id_column="src_subject_id",
    imputation_mode="reference",
)


print("Discovery CC preprocessing complete:", {k: v.shape for k, v in dict_final_cc.items()})


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import Utils
Utils = importlib.reload(Utils)
from Utils import build_group_palette

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from Utils import display_feature_name

# =========================
# PCA aggregated plots per modality: CHR clusters + CC
# - Each modality can have its own manual colors
# - Fits scaler+PCA on CHR only, projects CC into same space
# - Black jittered points always in front of violins
# =========================

# Required inputs:
# final_metrics  -> CHR final metrics dict
# dict_final_cc  -> output of apply_preprocessing_to_new_data(...): discovery CC per modality
# plots_dir      -> base directory for plots

out_dir = os.path.join(plots_dir, "merged_feature_pca_chr_vs_cc")
os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------
# Per-modality colours come from Utils.MODALITY_CLUSTER_PALETTES, the same
# palette already used for the t-SNE/PCA latent-space plots elsewhere in
# this notebook. CC always uses the shared reference colour from
# theme.CC_COLOR (baked into MODALITY_CLUSTER_PALETTES as _THEME_CC).
# --------------------------------------------------


mod_num = 0
for modality, df_chr in final_metrics["data"].items():
    print(f"\n=== PCA Aggregated Plots — Modality: {modality} ===")

    # CHR features + labels
    X_chr_df = df_chr.drop(columns=["src_subject_id"]).copy()
    clusters_chr = np.asarray(final_metrics["individual_labels"][mod_num])

    # CC features
    df_cc = dict_final_cc[modality]
    X_cc_df = df_cc.drop(columns=["src_subject_id"]).copy()

    # Strict feature alignment to CHR
    X_cc_df = X_cc_df.reindex(columns=X_chr_df.columns)

    # Optional diagnostics
    print("CHR shape:", X_chr_df.shape, "CC shape:", X_cc_df.shape)
    print("NaNs in CC feature matrix:", int(X_cc_df.isna().sum().sum()))

    # Convert to arrays
    X_chr = X_chr_df.values
    X_cc = X_cc_df.values

    # Fit scaler + PCA on CHR only
    scaler = StandardScaler()
    X_chr_z = scaler.fit_transform(X_chr)
    X_cc_z = scaler.transform(X_cc)

    n_pcs = min(5, X_chr_z.shape[1])
    pca = PCA(n_components=n_pcs, random_state=0)
    PC_chr = pca.fit_transform(X_chr_z)
    PC_cc = pca.transform(X_cc_z)

    pc1_chr = PC_chr[:, 0]
    pc1_cc = PC_cc[:, 0]

    # Plot dataframe
    plot_chr = pd.DataFrame({
        "group": clusters_chr.astype(str),
        "PC1": pc1_chr,
        "cohort": "CHR"
    })
    plot_cc = pd.DataFrame({
        "group": "CC",
        "PC1": pc1_cc,
        "cohort": "CC"
    })
    plot_df = pd.concat([plot_chr, plot_cc], ignore_index=True)

    cluster_order = sorted(
        plot_chr["group"].unique(),
        key=lambda x: int(x) if str(x).isdigit() else str(x)
    )
    group_order = cluster_order + ["CC"]

    # ---------------------------------------------
    # Pick palette for this modality (shared with the rest of the notebook)
    # ---------------------------------------------
    group_palette = build_group_palette(modality, group_order)

    fig, ax = plt.subplots(figsize=(10.5, 6.5))

    # Violin by group color
    sns.violinplot(
        data=plot_df,
        x="group",
        y="PC1",
        hue="group",
        order=group_order,
        hue_order=group_order,
        palette=group_palette,
        dodge=False,
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        legend=False,
        ax=ax
    )

    # Black points on top
    sns.stripplot(
        data=plot_df,
        x="group",
        y="PC1",
        order=group_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Bring points forward
    for c in ax.collections:
        c.set_zorder(2)
    for l in ax.lines:
        l.set_zorder(3)

    # Median marker
    medians = plot_df.groupby("group")["PC1"].median()
    x_positions = np.arange(len(group_order))
    med_vals = [medians.loc[g] for g in group_order]
    ax.scatter(
        x_positions, med_vals,
        marker="_", s=180, linewidths=3,
        color="black", zorder=4
    )

    # n labels
    counts = plot_df["group"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, g in enumerate(group_order):
        ax.text(i, y_annot, f"n={int(counts.get(g, 0))}",
                ha="center", va="top", fontsize=14)

    ax.set_xlabel("CHR clusters + CC", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.set_title(f"{modality}: CHR cluster PC1 vs CC", fontsize=16)
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    clinical_analysis_save_figure_png_pdf(
        fig,
        os.path.join(out_dir, f"{modality}_PC1_CHR_vs_CC.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    mod_num += 1


### Differences in original features based on final labels

In [ ]:

from sklearn.feature_selection import f_classif
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from Utils import display_feature_name

out_dir = os.path.join(plots_dir, "merged_feature_differences")
os.makedirs(out_dir, exist_ok=True)

top_k = 10
sample_name = "discovery"
label_scope = "final_labels"

for modality, df in final_metrics['data'].items():
    print(f"\n=== {sample_name} / {label_scope}: {modality} ===")

    feature_df = df.drop(columns=['src_subject_id'])
    X = feature_df.values
    clusters = np.asarray(final_metrics['final_labels'])
    feature_names = feature_df.columns

    if len(clusters) != len(df):
        raise ValueError(f"{modality}: final label length ({len(clusters)}) != data rows ({len(df)})")
    if len(pd.unique(clusters)) < 2:
        print(f"Skipping {modality}: fewer than two final labels.")
        continue

    f_vals, _ = f_classif(X, clusters)
    f_df = (
        pd.DataFrame({'feature': feature_names, 'f_value': f_vals})
        .assign(display_feature=lambda d: d['feature'].map(display_feature_name))
        .sort_values('f_value', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x='f_value', y='display_feature', data=f_df.head(top_k), palette=sns.color_palette(n_colors=top_k), ax=ax)
    ax.set_title(f"{modality} — Top {top_k} Discriminative Features (ANOVA F-value; discovery final labels)")
    ax.set_xlabel("F-value")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{sample_name}_{label_scope}_{modality}_top_features_bar.png"))
    plt.show()

    top_features = f_df['feature'].head(top_k).tolist()
    max_cols = 3
    n_rows = int(np.ceil(len(top_features) / max_cols))
    fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
    axes = np.array(axes).reshape(-1)

    cluster_order = [str(x) for x in np.unique(clusters)]
    df_cc = dict_final_cc.get(modality)
    cc_available = df_cc is not None
    if cc_available:
        df_cc = df_cc.copy().reindex(columns=df.columns, fill_value=np.nan)
    group_order = cluster_order + (["CC"] if cc_available else [])
    pal = sns.color_palette(n_colors=len(group_order))

    for i, feat in enumerate(top_features):
        ax = axes[i]
        plot_df = pd.DataFrame({
            'group': pd.Series(clusters).astype(str),
            'value': df[feat].values,
        }).dropna()
        if cc_available and feat in df_cc.columns:
            cc_plot_df = pd.DataFrame({'group': 'CC', 'value': df_cc[feat].values}).dropna()
            plot_df = pd.concat([plot_df, cc_plot_df], ignore_index=True)
        sns.boxplot(data=plot_df, x='group', y='value', order=group_order, palette=pal, ax=ax)
        sns.stripplot(data=plot_df, x='group', y='value', order=group_order, color='black', size=4, jitter=True, alpha=0.5, ax=ax)
        ax.set_title(display_feature_name(feat), fontsize=14)
        ax.set_xlabel("", fontsize=22)
        ax.set_ylabel("", fontsize=22)
        ax.tick_params(axis='both', labelsize=14)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{modality} — Distribution of Top Features by Discovery Final Labels and CC\n(with individual data points)",
        y=1.02, fontsize=18
    )
    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{sample_name}_{label_scope}_{modality}_top_features_with_cc.png"))
    plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import f_classif
from Utils import display_feature_name

# -----------------------------
# Settings
# -----------------------------
top_k = 15                    # change as you like
max_cols = 3                  # grid columns for feature distributions
out_dir = os.path.join(plots_dir, "merged_feature_differences")
os.makedirs(out_dir, exist_ok=True)


# -----------------------------
# 1) Build ONE merged feature table across ALL modalities
#    aligned by src_subject_id, with modality-prefixed feature names
# -----------------------------
# Use first modality as "reference" ordering (assumes labels align to this order)
modalities = list(final_metrics["data"].keys())
ref_mod = modalities[0]
ref_df = final_metrics["data"][ref_mod].copy()

# Keep reference subject ordering
ref_ids = ref_df["src_subject_id"].astype(str)
clusters = np.asarray(final_metrics["final_labels"])
if len(clusters) != len(ref_df):
    raise ValueError(
        f"final_labels length ({len(clusters)}) != number of subjects in reference modality "
        f"{ref_mod} ({len(ref_df)}). You likely need to align labels by src_subject_id."
    )

# Start merged dataframe with IDs
merged = pd.DataFrame({"src_subject_id": ref_ids.values})

# Add each modality's features, aligned by src_subject_id, modality-prefixed
for modality, df in final_metrics["data"].items():
    tmp = df.copy()
    tmp["src_subject_id"] = tmp["src_subject_id"].astype(str)

    # Drop duplicates and set index for alignment
    tmp = tmp.drop_duplicates("src_subject_id").set_index("src_subject_id")

    # Align to reference IDs (inner join behavior -> missing subjects become NaN)
    tmp = tmp.reindex(ref_ids.values)

    # Prefix feature names to avoid collisions across modalities
    feat_cols = tmp.columns
    tmp = tmp.rename(columns={c: f"{modality}::{c}" for c in feat_cols})

    merged = pd.concat([merged, tmp.reset_index(drop=True)], axis=1)

# Option A: drop any subjects with missing modality data (strict intersection)
merged_clean = merged.dropna(axis=0).reset_index(drop=True)

# Make sure labels match rows after dropping NaNs (keep only subjects that remain)
keep_mask = merged.notna().all(axis=1).values
clusters_clean = clusters[keep_mask]

# Feature matrix
X = merged_clean.drop(columns=["src_subject_id"]).values
feature_names = merged_clean.drop(columns=["src_subject_id"]).columns

# -----------------------------
# 2) Global feature ranking (ANOVA F-test) using integrated labels
# -----------------------------
f_vals, _ = f_classif(X, clusters_clean)
f_df = (
    pd.DataFrame({"feature": feature_names, "f_value": f_vals})
    .assign(display_feature=lambda d: d["feature"].map(clinical_analysis_display_merged_feature_name))
      .sort_values("f_value", ascending=False)
)

# -----------------------------
# 3) Plot: Top-K contributing features ACROSS ALL modalities
# -----------------------------
# --- Barplot of Top Features ---
plt.figure(figsize=(12, 7))
sns.barplot(
    x="f_value", y="display_feature",
    data=f_df.head(top_k),
    palette=sns.color_palette(n_colors=top_k)
)
plt.title(f"All modalities — Top {top_k} Discriminative Features (ANOVA F-value)")
plt.xlabel("F-value")
plt.ylabel("Feature (modality::feature)")
plt.tight_layout()
clinical_analysis_save_figure_png_pdf(plt.gcf(), os.path.join(out_dir, "ALL_modalities_top_features_bar.png"), dpi=200)
plt.show()

# --- Boxplots + Scatter Points of Top Features ---
top_features = f_df["feature"].head(top_k).tolist()

n_feats = len(top_features)
n_rows = int(np.ceil(n_feats / max_cols))
fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
axes = np.array(axes).reshape(-1)

cluster_order = np.unique(clusters_clean)
pal = sns.color_palette(n_colors=len(cluster_order))

# Build a plotting df where each feature is a column
plot_df = merged_clean.copy()
plot_df["cluster"] = clusters_clean

for i, feat in enumerate(top_features):
    ax = axes[i]
    sns.boxplot(x="cluster", y=feat, data=plot_df, palette=pal, ax=ax)
    sns.stripplot(
        x="cluster", y=feat, data=plot_df,
        color="black", size=4, jitter=True, alpha=0.5, ax=ax
    )
    ax.set_title(clinical_analysis_display_merged_feature_name(feat), fontsize=12)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="both", labelsize=12)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f"All modalities — Distribution of Top {top_k} Features by Integrated Cluster\n(with individual data points)",
    y=1.02, fontsize=16
)
plt.tight_layout()
clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, "ALL_modalities_top_features_box.png"), dpi=200, bbox_inches="tight")
plt.show()

print("Saved:")
print(" -", os.path.join(out_dir, "ALL_modalities_top_features_bar.png"))
print(" -", os.path.join(out_dir, "ALL_modalities_top_features_box.png"))


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from Utils import display_feature_name

# ==========================================================
# 1) SINGLE PLOT: PC1 (per-modality PCA) distributions by modality (hue=integrated cluster)
# 2) ADDITIONAL "GLOBAL" DIFFERENCE: one shared PC1 computed from ALL features across ALL modalities
#    -> a separate global violin plot + optional printout of EVR
#
# IMPORTANT: Test integrated labels are aligned to src_subject_id via
# test_final_labels_by_subject_id, created from the test predictions.
# ==========================================================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

# --------------------------
# A) Per-modality PC1 plot (your current single plot)
# --------------------------
rows = []
for modality, df in final_metrics["data"].items():
    feature_df = df.drop(columns=["src_subject_id"])
    X = feature_df.values
    clusters = np.asarray(final_metrics["final_labels"])

    Xz = StandardScaler().fit_transform(X)
    pca = PCA(n_components=1, random_state=0).fit(Xz)
    pc1 = pca.transform(Xz)[:, 0]
    evr1 = float(pca.explained_variance_ratio_[0])

    tmp = pd.DataFrame({"modality": modality, "cluster": clusters, "PC1": pc1})
    tmp["PC1_EVR"] = evr1
    rows.append(tmp)

    print(f"{modality}: PC1 EVR={evr1:.2%}")

plot_df = pd.concat(rows, ignore_index=True)

modality_order = list(final_metrics["data"].keys())
cluster_order = np.sort(plot_df["cluster"].unique())

fig_w = max(18, 2.2 * len(modality_order))
fig_h = 9
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.violinplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="cluster",
    order=modality_order,
    hue_order=cluster_order,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    dodge=True,
    width=0.65,
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="cluster",
    order=modality_order,
    hue_order=cluster_order,
    dodge=True,
    jitter=0.18,
    size=2.2,
    alpha=0.18,
    color="black",
    ax=ax
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:len(cluster_order)], labels[:len(cluster_order)], title="Cluster", frameon=False, loc="upper right")

for x in np.arange(0.5, len(modality_order), 1.0):
    ax.axvline(x, linewidth=0.8, alpha=0.25)

ax.set_xlabel("Modality", fontsize=16)
ax.set_ylabel("PCA Component 1 score", fontsize=16)
ax.set_title("PC1 Distributions by Modality (Integrated Cluster Differences)", fontsize=18, pad=14)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=13)
plt.setp(ax.get_yticklabels(), fontsize=13)

fig.tight_layout()
clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, "ALL_modalities_PC1_singleplot_violin_big.png"), dpi=300, bbox_inches="tight")
plt.show()


# --------------------------
# B) GLOBAL PCA PC1 across ALL modalities/features (shared PC axis)
# --------------------------

# 1) Merge all modalities into one wide table keyed by src_subject_id
dfs = []
for modality, df in final_metrics["data"].items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})  # avoid name collisions
    dfs.append(tmp)

merged = dfs[0]
for d in dfs[1:]:
    merged = merged.merge(d, on="src_subject_id", how="inner")  # subjects present in ALL modalities

print(f"\n[GLOBAL PCA] Subjects after inner-join: {merged.shape[0]}")
print(f"[GLOBAL PCA] Total merged features: {merged.shape[1] - 1}")

# 2) Align integrated labels to merged subject IDs
label_map = final_metrics.get("final_labels_by_subject_id", None)

if label_map is not None:
    merged_clusters = merged["src_subject_id"].map(label_map).to_numpy()
    if np.any(pd.isna(merged_clusters)):
        missing = merged.loc[pd.isna(merged_clusters), "src_subject_id"].head(5).tolist()
        raise ValueError(
            "Some merged subjects are missing labels in final_labels_by_subject_id. "
            f"Examples: {missing}"
        )
else:
    # Fallback assumption: final_labels already correspond exactly to rows in the merged table.
    # This is ONLY valid if your data are already aligned and the merge didn't drop/reorder subjects.
    if len(final_metrics["final_labels"]) != merged.shape[0]:
        raise ValueError(
            f"final_labels length ({len(final_metrics['final_labels'])}) != merged subjects ({merged.shape[0]}).\n"
            "To do this safely, provide:\n"
            "  final_metrics['final_labels_by_subject_id'] = {src_subject_id: label, ...}\n"
            "so labels can be aligned after the merge."
        )
    merged_clusters = np.asarray(final_metrics["final_labels"])

# 3) Global PCA (PC1)
X_global = merged.drop(columns=["src_subject_id"]).values
Xg_z = StandardScaler().fit_transform(X_global)

pca_global = PCA(n_components=1, random_state=0).fit(Xg_z)
pc1_global = pca_global.transform(Xg_z)[:, 0]
evr1_global = float(pca_global.explained_variance_ratio_[0])
print(f"[GLOBAL PCA] PC1 EVR: {evr1_global:.2%}")

global_df = pd.DataFrame({
    "cluster": merged_clusters,
    "PC1_global": pc1_global
})

global_cluster_order = np.sort(global_df["cluster"].unique())

# 4) Plot global PC1 difference by integrated cluster
fig2, ax2 = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=global_df,
    x="cluster",
    y="PC1_global",
    order=global_cluster_order,
    palette=sns.color_palette(n_colors=len(global_cluster_order)),
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    ax=ax2
)

sns.stripplot(
    data=global_df,
    x="cluster",
    y="PC1_global",
    order=global_cluster_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.30,
    ax=ax2
)

# Median markers
med = global_df.groupby("cluster")["PC1_global"].median()
xpos = np.arange(len(global_cluster_order))
ax2.scatter(
    xpos,
    [med.loc[c] for c in global_cluster_order],
    s=180,
    marker="_",
    linewidths=3
)

# n labels
counts = global_df["cluster"].value_counts()
ymin, ymax = ax2.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
#for i, c in enumerate(global_cluster_order):
#    ax2.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="top", fontsize=16)

ax2.set_xlabel("Integrated cluster", fontsize=20)
ax2.set_ylabel("Global PC1 score (all modalities/features)", fontsize=20)
ax2.set_title(f"Global PC1 Across All Features and Modalities", fontsize=20, pad=14)
ax2.grid(axis="y", alpha=0.15)
ax2.set_xticklabels(ax2.get_xticklabels(), fontsize=16)
ax2.set_yticklabels(ax2.get_yticklabels(), fontsize=16)
sns.despine(ax=ax2)

fig2.tight_layout()
clinical_analysis_save_figure_png_pdf(fig2, os.path.join(out_dir, "GLOBAL_all_modalities_PC1_violin.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from Utils import display_feature_name

# ==========================================================
# GLOBAL PCA (ALL modalities merged) -> PC1 by final clusters
# - Merges modalities on src_subject_id (inner-join by default)
# - Standardizes all features
# - PCA -> PC1
# - Publication-ready violin (quartiles) + jitter + median + n
# - Also saves a variance plot (top 5 PCs) for the merged space
# ==========================================================

global_out_dir = os.path.join(plots_dir, "global_pca_all_modalities/")
os.makedirs(global_out_dir, exist_ok=True)

# ---- 1) Merge all modalities into one wide dataframe ----
dfs = []
for modality, df in final_metrics["data"].items():
    tmp = df.copy()

    # Prefix feature names with modality to avoid collisions
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})

    dfs.append(tmp)

# Inner join across modalities by subject id (keeps subjects present in ALL modalities)
merged = dfs[0]
for d in dfs[1:]:
    merged = merged.merge(d, on="src_subject_id", how="inner")

print(f"[GLOBAL PCA] Subjects after inner-join across modalities: {merged.shape[0]}")
print(f"[GLOBAL PCA] Total merged features: {merged.shape[1] - 1}")

# ---- 2) Align final labels to the merged subject IDs ----
clusters = np.asarray(final_metrics["final_labels"])

# If your final_labels are already in the same row-order as each modality df,
# then this should match the *intersection* order only if you kept the same subjects.
# Safest is to align by subject ID if you have a mapping.
#
# Try to auto-detect a mapping dict if provided:
label_map = final_metrics.get("final_labels_by_subject_id", None)

if label_map is not None:
    # label_map should be {src_subject_id: label}
    merged_clusters = merged["src_subject_id"].map(label_map).to_numpy()
    if np.any(pd.isna(merged_clusters)):
        raise ValueError("Some merged subjects are missing labels in final_labels_by_subject_id.")
else:
    # Fallback: assume labels are already aligned to the rows of EACH modality df.
    # This is ONLY safe if all modality dfs share the same subject order and
    # inner-joining didn't change it. If you see mismatches, create label_map above.
    if len(clusters) != merged.shape[0]:
        raise ValueError(
            f"final_labels length ({len(clusters)}) != merged subjects ({merged.shape[0]}). "
            "Provide final_metrics['final_labels_by_subject_id'] to align labels safely."
        )
    merged_clusters = clusters

# ---- 3) PCA on all features ----
X = merged.drop(columns=["src_subject_id"]).values

# Standardize before PCA
Xz = StandardScaler().fit_transform(X)

# Fit PCA (enough components to report variance for first 5, but at least 2 if possible)
n_pcs = min(5, Xz.shape[1])
pca = PCA(n_components=n_pcs, random_state=0)
scores = pca.fit_transform(Xz)

pc1 = scores[:, 0]
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)

print(f"[GLOBAL PCA] PC1 EVR: {evr[0]:.2%}")

# ---- 4) Plot PC1 by cluster (global / all modalities) ----
plot_df = pd.DataFrame({"cluster": merged_clusters, "PC1": pc1})
cluster_order = np.sort(pd.unique(plot_df["cluster"]))

fig, ax = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=plot_df,
    x="cluster",
    y="PC1",
    order=cluster_order,
    palette=sns.color_palette(n_colors=len(cluster_order)),
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="cluster",
    y="PC1",
    order=cluster_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.35,
    ax=ax
)

# Median marker per cluster
medians = plot_df.groupby("cluster")["PC1"].median()
x_positions = np.arange(len(cluster_order))
ax.scatter(
    x=x_positions,
    y=[medians.loc[c] for c in cluster_order],
    s=180,
    marker="_",
    linewidths=3
)

# Annotate n per cluster near bottom
counts = plot_df["cluster"].value_counts()
ymin, ymax = ax.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
for i, c in enumerate(cluster_order):
    ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="bottom", fontsize=12)

ax.set_xlabel("Cluster", fontsize=16)
ax.set_ylabel("PCA Component 1 score", fontsize=16)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

fig.tight_layout()
clinical_analysis_save_figure_png_pdf(fig, os.path.join(global_out_dir, "GLOBAL_all_modalities_PC1_violin.png"), dpi=300, bbox_inches="tight")
plt.show()

# ---- 5) Variance explained plot (Top PCs) ----
figv, axv = plt.subplots(figsize=(10, 5.5))
pcs_idx = np.arange(1, n_pcs + 1)

axv.bar(pcs_idx, evr)
axv.plot(pcs_idx, cum_evr, marker="o")

axv.set_xticks(pcs_idx)
axv.set_xlabel("Principal Component")
axv.set_ylabel("Explained variance ratio")
axv.set_title(f"GLOBAL (All Modalities) — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

axv.grid(axis="y", alpha=0.15)
sns.despine(ax=axv)

figv.tight_layout()
clinical_analysis_save_figure_png_pdf(figv, os.path.join(global_out_dir, f"GLOBAL_all_modalities_variance_top{n_pcs}.png"),
             dpi=300, bbox_inches="tight")
plt.show()


### Add CC to PCA integrated results plots

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import Utils
Utils = importlib.reload(Utils)
from Utils import modality_cluster_palette

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# 1) SINGLE PLOT: PC1 (per-modality PCA) distributions by modality
#    - CHR integrated clusters + CC group
# 2) GLOBAL PCA across ALL modalities/features
#    - CHR integrated clusters + CC group
# ==========================================================

out_dir = os.path.join(plots_dir, "merged_feature_pca_chr_vs_cc/")
os.makedirs(out_dir, exist_ok=True)

# --------------------------
# A) Per-modality PC1 plot: CHR clusters + CC
# --------------------------
rows = []
final_labels_chr = np.asarray(final_metrics["final_labels"]).astype(str)

for modality, df_chr in final_metrics["data"].items():
    # CHR
    X_chr_df = df_chr.drop(columns=["src_subject_id"]).copy()
    if len(final_labels_chr) != len(X_chr_df):
        raise ValueError(f"{modality}: final_labels length != CHR rows. Use final_labels_by_subject_id alignment if needed.")
    grp_chr = final_labels_chr

    # CC (already transformed via apply_preprocessing_to_new_data)
    df_cc = dict_final_cc[modality]
    X_cc_df = df_cc.drop(columns=["src_subject_id"]).copy()
    X_cc_df = X_cc_df.reindex(columns=X_chr_df.columns)  # strict CHR feature order

    # Fit scaler+PCA on CHR only
    scaler = StandardScaler()
    X_chr_z = scaler.fit_transform(X_chr_df.values)
    X_cc_z = scaler.transform(X_cc_df.values)

    pca = PCA(n_components=1, random_state=0).fit(X_chr_z)
    pc1_chr = pca.transform(X_chr_z)[:, 0]
    pc1_cc = pca.transform(X_cc_z)[:, 0]
    evr1 = float(pca.explained_variance_ratio_[0])

    print(f"{modality}: CHR-fitted PC1 EVR={evr1:.2%}")

    tmp_chr = pd.DataFrame({
        "modality": modality,
        "group": grp_chr,
        "PC1": pc1_chr,
        "cohort": "CHR"
    })
    tmp_cc = pd.DataFrame({
        "modality": modality,
        "group": "CC",
        "PC1": pc1_cc,
        "cohort": "CC"
    })
    tmp = pd.concat([tmp_chr, tmp_cc], ignore_index=True)
    tmp["PC1_EVR_chrfit"] = evr1
    rows.append(tmp)

plot_df = pd.concat(rows, ignore_index=True)

modality_order = list(final_metrics["data"].keys())
chr_cluster_order = sorted(pd.unique(final_labels_chr), key=lambda x: int(x) if str(x).isdigit() else str(x))
group_order = chr_cluster_order + ["CC"]

# Shared palette (same helper used for the per-modality CHR-vs-CC plots above)
group_palette = modality_cluster_palette(group_order)

fig_w = max(18, 2.2 * len(modality_order))
fig_h = 6
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.violinplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="group",
    order=modality_order,
    hue_order=group_order,
    palette=group_palette,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    dodge=True,
    width=0.8,
    ax=ax
)

ax.set_xlim(-0.55, len(modality_order) - 0.45)

# --- manual black points centered inside each dodged violin ---
rng = np.random.default_rng(0)

n_hue = len(group_order)
violin_width = 0.8                      # must match sns.violinplot(width=0.65)
sub_width = violin_width / n_hue         # width allotted to each group within a modality
jitter_scale = sub_width * 0.28          # small jitter within each subgroup

for i, modality in enumerate(modality_order):
    for j, group in enumerate(group_order):
        vals = plot_df.loc[
            (plot_df["modality"] == modality) & (plot_df["group"] == group),
            "PC1"
        ].to_numpy()

        if len(vals) == 0:
            continue

        # exact center of this group's violin within this modality
        center = i - violin_width / 2 + (j + 0.5) * sub_width

        # small symmetric jitter around that center
        x = center + rng.uniform(-jitter_scale, jitter_scale, size=len(vals))

        ax.scatter(
            x,
            vals,
            color="black",
            s=10,
            alpha=0.22,
            zorder=3,
            linewidths=0
        )

# Legend cleanup (keep one)
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:len(group_order)],
    labels[:len(group_order)],
    title="Group",
    frameon=False,
    loc="upper right"
)

for x in np.arange(0.5, len(modality_order), 1.0):
    ax.axvline(x, linewidth=0.8, alpha=0.25)

ax.set_xlabel("Modality", fontsize=16)
ax.set_ylabel("PC1 score (CHR-fitted PCA)", fontsize=16)
ax.set_title("PC1 Distributions by Modality (CHR integrated clusters + CC)", fontsize=18, pad=14)
ax.tick_params(axis="both", labelsize=14)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=13)
plt.setp(ax.get_yticklabels(), fontsize=13)

fig.tight_layout()
clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, "ALL_modalities_PC1_singleplot_violin_CHR_vs_CC.png"), dpi=300, bbox_inches="tight")
plt.show()

# --------------------------
# B) GLOBAL PCA PC1 across ALL modalities/features: CHR clusters + CC
# --------------------------

# CHR merged wide
dfs_chr = []
for modality, df in final_metrics["data"].items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})
    dfs_chr.append(tmp)

merged_chr = dfs_chr[0]
for d in dfs_chr[1:]:
    merged_chr = merged_chr.merge(d, on="src_subject_id", how="inner")

print(f"\n[GLOBAL PCA] CHR subjects after inner-join: {merged_chr.shape[0]}")
print(f"[GLOBAL PCA] CHR merged features: {merged_chr.shape[1] - 1}")

# CC merged wide
dfs_cc = []
for modality, df in dict_final_cc.items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})
    dfs_cc.append(tmp)

merged_cc = dfs_cc[0]
for d in dfs_cc[1:]:
    merged_cc = merged_cc.merge(d, on="src_subject_id", how="inner")

print(f"[GLOBAL PCA] CC subjects after inner-join: {merged_cc.shape[0]}")
print(f"[GLOBAL PCA] CC merged features: {merged_cc.shape[1] - 1}")

# Align CHR labels to merged CHR IDs
label_map = final_metrics.get("final_labels_by_subject_id", None)
if label_map is not None:
    chr_clusters = merged_chr["src_subject_id"].map(label_map).astype(str).to_numpy()
    if np.any(pd.isna(chr_clusters)):
        missing = merged_chr.loc[pd.isna(chr_clusters), "src_subject_id"].head(5).tolist()
        raise ValueError(f"Missing labels for merged CHR IDs. Examples: {missing}")
else:
    if len(final_labels_chr) != merged_chr.shape[0]:
        raise ValueError(
            f"final_labels length ({len(final_labels_chr)}) != merged CHR subjects ({merged_chr.shape[0]}). "
            "Provide final_labels_by_subject_id for safe alignment."
        )
    chr_clusters = final_labels_chr

# Strict CC feature alignment to CHR merged features
chr_feature_cols = [c for c in merged_chr.columns if c != "src_subject_id"]
cc_feature_cols = [c for c in merged_cc.columns if c != "src_subject_id"]
missing_in_cc = [c for c in chr_feature_cols if c not in cc_feature_cols]
if missing_in_cc:
    raise ValueError(f"CC missing {len(missing_in_cc)} global features. Example: {missing_in_cc[:10]}")

X_chr_global = merged_chr[chr_feature_cols].values
X_cc_global = merged_cc.reindex(columns=chr_feature_cols).values

# CHR-fitted global PCA
Xg_scaler = StandardScaler()
Xg_chr_z = Xg_scaler.fit_transform(X_chr_global)
Xg_cc_z = Xg_scaler.transform(X_cc_global)

pca_global = PCA(n_components=1, random_state=0).fit(Xg_chr_z)
pc1_chr_global = pca_global.transform(Xg_chr_z)[:, 0]
pc1_cc_global = pca_global.transform(Xg_cc_z)[:, 0]
evr1_global = float(pca_global.explained_variance_ratio_[0])
print(f"[GLOBAL PCA] CHR-fitted PC1 EVR: {evr1_global:.2%}")

global_df = pd.concat([
    pd.DataFrame({"group": chr_clusters, "PC1_global": pc1_chr_global, "cohort": "CHR"}),
    pd.DataFrame({"group": "CC", "PC1_global": pc1_cc_global, "cohort": "CC"})
], ignore_index=True)

global_group_order = chr_cluster_order + ["CC"]

fig2, ax2 = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=global_df,
    x="group",
    y="PC1_global",
    hue="group",
    order=global_group_order,
    hue_order=global_group_order,
    palette=group_palette,
    dodge=False,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    legend=False,
    ax=ax2
)

sns.stripplot(
    data=global_df,
    x="group",
    y="PC1_global",
    order=global_group_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.30,
    ax=ax2
)

med = global_df.groupby("group")["PC1_global"].median()
xpos = np.arange(len(global_group_order))
ax2.scatter(xpos, [med.loc[g] for g in global_group_order], s=180, marker="_", linewidths=3, color="black", zorder=4)

counts = global_df["group"].value_counts()
ymin, ymax = ax2.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
for i, g in enumerate(global_group_order):
    ax2.text(i, y_annot, f"n={int(counts.get(g, 0))}", ha="center", va="top", fontsize=16)

ax2.set_xlabel("Group", fontsize=20)
ax2.set_ylabel("Global PC1 score (CHR-fitted)", fontsize=20)
ax2.set_title("Global PC1 Across All Features and Modalities (CHR clusters + CC)", fontsize=20, pad=14)
ax2.grid(axis="y", alpha=0.15)
ax2.tick_params(axis="x", labelsize=16)
ax2.tick_params(axis="y", labelsize=16)
sns.despine(ax=ax2)

fig2.tight_layout()
clinical_analysis_save_figure_png_pdf(fig2, os.path.join(out_dir, "GLOBAL_all_modalities_PC1_violin_CHR_vs_CC.png"), dpi=300, bbox_inches="tight")
plt.show()


### Differences categorical with individual labels

In [ ]:
# Use the shared add metadata and clusters helper.
from Utils import clinical_analysis_add_metadata_and_clusters_individual_labels

# Use the shared chi square comparison helper.
from Utils import clinical_analysis_chi_square_comparison_individual_labels

mod_num = 0
for modality in final_metrics['data'].keys():
    print(f"\n=== Analyzing categorical differences for modality: {modality} ===")

    # Merge cluster labels into full data
    df = clinical_analysis_add_metadata_and_clusters_individual_labels(final_metrics, discovery_data, mod_num)

    # Compare by site
    if 'Site' in df.columns:
        clinical_analysis_chi_square_comparison_individual_labels(
            df=df,
            group_col='Cluster',
            label_col='Site',
            title_prefix=f"Comparison of Site Distribution per Subgroup",
        )

    # Optional: extend for other categorical variables
    for col in ['sips_bips_scr_lifetime', 'sips_aps_scr_lifetime', 'sips_grd_scr_lifetime']:
        if col in df.columns:
            clinical_analysis_chi_square_comparison_individual_labels(
                df=df,
                group_col='Cluster',
                label_col=col,
                title_prefix=f"Comparison of {col} per Subgroup",
            )
    mod_num = mod_num + 1




### Differences categorical with final labels

In [ ]:
## Create directory for categorical differences plots
os.makedirs(os.path.join(plots_dir, "merged_cat_diff/"), exist_ok=True)

# Use the shared add metadata and clusters helper.
from Utils import clinical_analysis_add_metadata_and_clusters_final_labels


# Use the shared chi square comparison helper.
from Utils import clinical_analysis_chi_square_comparison_final_labels


# Merge cluster labels into full data
df = clinical_analysis_add_metadata_and_clusters_final_labels(final_metrics, discovery_data)


# Compare by site
if 'Site' in df.columns:
    clinical_analysis_chi_square_comparison_final_labels(
        df=df,
        group_col='Cluster',
        label_col='Site',
        title_prefix=f"Comparison of Site Distribution per Subgroup",
        save_path=os.path.join(plots_dir,'merged_cat_diff/', f"merged_Site_by_subgroup.png")
    )

# Optional: extend for other categorical variables
for col in ['sips_bips_scr_lifetime', 'sips_aps_scr_lifetime', 'sips_grd_scr_lifetime']:
    if col in df.columns:
        clinical_analysis_chi_square_comparison_final_labels(
            df=df,
            group_col='Cluster',
            label_col=col,
            title_prefix=f"Comparison of {col} per Subgroup",
            save_path=os.path.join(plots_dir,'merged_cat_diff/', f"merged_{col}_by_subgroup.png")
        )




## Overlap between modalities (in labels)

In [ ]:
# Rename labels for clarity
# The cluster with generally high scores should be called 'high' and the cluster with generally low scores should be called 'low' for all modalities

new_labels_list = []
modality_names = list(final_metrics["data"].keys())
for i, modality in enumerate(modality_names):
    print(f"Processing modality: {modality}")

    labels = final_metrics['individual_labels'][i]
    df = final_metrics['data'][modality]

    # Skip for modality if num_cluster<2
    if len(np.unique(labels)) < 2:
        continue
    
    # Calculate mean score per cluster
    cluster_means = {}
    for cluster in np.unique(labels):
        cluster_means[cluster] = df.drop(columns=['src_subject_id']).mean().mean()

    # Determine which cluster is 'high' and which is 'low'
    sorted_clusters = sorted(cluster_means, key=cluster_means.get, reverse=True)
    high_cluster = sorted_clusters[0]
    low_cluster = sorted_clusters[1]
    
    if modality == "Functioning" or modality == "Cognition":
        # For Functioning, reverse the assignment
        new_labels = ['low_severity' if lbl == high_cluster else 'high_severity' for lbl in labels]
    else:
        # Create new labels
        new_labels = ['high_severity' if lbl == high_cluster else 'low_severity' for lbl in labels]

    new_labels_list.append(new_labels)


In [ ]:
# pip install upsetplot pandas matplotlib
from collections import Counter
import pandas as pd
from upsetplot import UpSet

modality_names = list(final_metrics["data"].keys())

labels = new_labels_list
M = len(labels)
N = len(labels[0])
assert all(len(x) == N for x in labels)

# majority label per sample
majority = [Counter(labels[m][k] for m in range(M)).most_common(1)[0][0] for k in range(N)]

# boolean matrix: modality agrees with majority?
agree = pd.DataFrame({f"{modality_names[m]}"    : [labels[m][k] == majority[k] for k in range(N)] for m in range(M)})

# UpSet expects a MultiIndex of booleans
counts = agree.value_counts()
UpSet(counts, show_counts=True, sort_by="cardinality").plot()


In [ ]:
import pandas as pd

labels = new_labels_list
M = len(labels)
N = len(labels[0])

all_agree = sum(len({labels[m][k] for m in range(M)}) == 1 for k in range(N))
print("All-4 agree:", all_agree, "/", N, "=", all_agree / N)

# 1 means full agreement, 2 means split (high vs low), since only two labels exist
unique_per_sample = [len({labels[m][k] for m in range(M)}) for k in range(N)]
print("Unique-labels-per-sample distribution:")
print(pd.Series(unique_per_sample).value_counts().sort_index())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

labels = new_labels_list
modality_names = list(final_metrics["data"].keys())

names = modality_names
N = len(labels[0])

mat = np.zeros((len(labels), len(labels)), dtype=float)
for i in range(len(labels)):
    for j in range(len(labels)):
        mat[i, j] = sum(labels[i][k] == labels[j][k] for k in range(N)) / N

df = pd.DataFrame(mat, index=names, columns=names)
print(df)

plt.figure()
plt.imshow(df.values)
plt.xticks(range(len(names)), names)
plt.yticks(range(len(names)), names)
plt.title("Pairwise agreement rate")
plt.colorbar()
plt.show()


In [ ]:

final = final_metrics["final_labels"]
mods = new_labels_list  # list of 4 lists, length 665 each


assert all(len(m) == len(final) for m in mods)

for name, m in zip(names, mods):
    ct = pd.crosstab(
        pd.Series(m, name=f"{name}_label"),
        pd.Series(final, name="final_label")
    )
    print(f"\n=== {name}: counts (rows=mod label, cols=final label) ===")
    print(ct)

    # Row-normalized: P(final | modality_label)
    ct_row = ct.div(ct.sum(axis=1), axis=0).fillna(0)
    print(f"\n=== {name}: row-normalized (P(final | mod label)) ===")
    print(ct_row.round(3))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

final = pd.Series(final_metrics["final_labels"], name="final")
mods = new_labels_list

rows = []
for name, m in zip(names, mods):
    m = pd.Series(m, name="mod")
    ct = pd.crosstab(m, final, normalize="index").fillna(0)  # P(final | mod_label)
    # make rows like "mod0:0", "mod0:1"
    for mod_label in ct.index:
        rowname = f"{name}:{mod_label}"
        rows.append(pd.Series(ct.loc[mod_label], name=rowname))

mat = pd.DataFrame(rows)  # rows=modality:label, cols=final labels

plt.figure(figsize=(6, 6))
plt.imshow(mat.values)
plt.xticks(range(mat.shape[1]), mat.columns)
plt.yticks(range(mat.shape[0]), mat.index)
plt.title("P(final label | modality label)")
plt.xlabel("final label")
plt.colorbar()
plt.tight_layout()
plt.show()


In [ ]:

final = pd.Series(final_metrics["final_labels"], name="final")
mods = [pd.Series(m, name=f"mod{i}") for i, m in enumerate(new_labels_list)]
df = pd.concat([final] + mods, axis=1)

mod_names = names

for col in df.columns[1:]:
    # distribution of modality labels within each final label
    dist = pd.crosstab(df["final"], df[col], normalize="index")
    name = mod_names[df.columns.get_loc(col)-1]
    print(f"\n=== {name}: distribution of {col} within each final label (rows sum to 1) ===")
    print(dist.round(3))


In [ ]:
new_labels_by_modality = {}
modality_names = list(final_metrics["data"].keys())

for i, modality in enumerate(modality_names):
    labels = final_metrics['individual_labels'][i]
    df = final_metrics['data'][modality]

    if len(np.unique(labels)) < 2:
        continue

    cluster_means = {}
    for cluster in np.unique(labels):
        cluster_means[cluster] = df.drop(columns=['src_subject_id']).mean().mean()

    sorted_clusters = sorted(cluster_means, key=cluster_means.get, reverse=True)
    high_cluster = sorted_clusters[0]
    low_cluster = sorted_clusters[1]

    if modality in ("Functioning", "Cognition"):
        new_labels = ['low_severity' if lbl == high_cluster else 'high_severity' for lbl in labels]
    else:
        new_labels = ['high_severity' if lbl == high_cluster else 'low_severity' for lbl in labels]

    new_labels_by_modality[modality] = new_labels


In [ ]:
import pandas as pd

stage_order = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"]

# Pick subject_id vector from one modality (must share row order with labels)
subject_ids = final_metrics["data"][stage_order[0]]["src_subject_id"].astype(str).reset_index(drop=True)

# Build the path dataframe
df_paths = pd.DataFrame({"src_subject_id": subject_ids})
for stage in stage_order:
    df_paths[stage] = pd.Series(new_labels_by_modality[stage]).astype(str).reset_index(drop=True)

df_paths["final"] = pd.Series(final_metrics["final_labels"]).astype(str).reset_index(drop=True)

# Sanity checks
N = len(df_paths)
assert all(len(new_labels_by_modality[s]) == N for s in stage_order), "Label lengths don't match subject_ids length"
assert len(final_metrics["final_labels"]) == N, "Final labels length doesn't match subject_ids length"

df_paths.head()


In [ ]:
# Use the shared summarize streams helper.
from Utils import clinical_analysis_summarize_streams

stream_summary = clinical_analysis_summarize_streams(df_paths, stage_order, top_k=100, sample_ids=12)
stream_summary


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

stages = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"]
df = df_paths.copy()

fig, axes = plt.subplots(1, len(stages), figsize=(4*len(stages), 4), sharey=True)
for ax, s in zip(axes, stages):
    ct = pd.crosstab(df[s], df["final"])
    ct.plot(kind="bar", ax=ax)
    ax.set_title(s)
    ax.set_xlabel("")
    ax.legend(title="final")
plt.tight_layout()
plt.show()


In [ ]:
# General k-safe parallel-categories helper.
# The implementation lives in Utils.py so the same behavior is used across notebooks and reports.
Utils = importlib.reload(Utils)
from Utils import domain_map

In [ ]:
stage_order = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"]

domain_map(
    new_labels_by_modality=new_labels_by_modality,
    final_labels=final_metrics["final_labels"],
    stage_order=["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"],
    final_name="final",
    top_token="high_severity",          # now HIGH is on top
    bottom_token="low_severity",
    invert_final=False,
    color_for_top_final="#38699A",      # top-like final ribbons
    color_for_bottom_final="#B36F9C",   # bottom-like final ribbons
    add_gap_in_final=True,
    gap_weight=20,
    plots_dir=plots_dir,
    save_file_name = "Parcats_by_final.pdf"
)


## SVM results

In [ ]:
Utils=importlib.reload(Utils)
from Utils import *

### Final labels

#### Accuracy

In [ ]:
svm_plots_dir = Path(plots_dir) / "SVM"
svm_plots_dir.mkdir(parents=True, exist_ok=True)


for metric in final_metrics['svm_results']['mean_metrics']:
    print(f"SVM Mean {metric}: {final_metrics['svm_results']['mean_metrics'][metric]}")


#### Uncertainty

In [ ]:
final_metrics['svm_results']['oof_uncertainty']


In [ ]:
# Find most confident mistakes 
df = final_metrics['svm_results']['oof_uncertainty'] 

df_bad = df[(df["y_true"] != df["y_pred"]) & (df["confidence"] > 0.9)]
df_bad.sort_values("confidence", ascending=False).head(20)


In [ ]:
# Find most uncertain predictions
df_uncertain = df.sort_values(["confidence", "entropy"], ascending=[True, False])
df_uncertain.head(20)


In [ ]:
correct = df["y_true"] == df["y_pred"]

fig = plt.figure()
plt.hist(df.loc[correct, "confidence"].dropna(), bins=30, alpha=0.7, label="correct")
plt.hist(df.loc[~correct, "confidence"].dropna(), bins=30, alpha=0.7, label="wrong")
plt.xlabel("Confidence (max predicted probability)")
plt.ylabel("Count")
plt.title("Confidence distribution: correct vs wrong")
plt.legend()
plt.tight_layout()
clinical_analysis_save_svm_figure_png_svg(fig, "integrated_confidence_distribution", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
df2 = df.dropna(subset=["confidence"]).copy()
bins = np.linspace(0, 1, 11)

df2["bin"] = pd.cut(df2["confidence"], bins=bins, include_lowest=True)
grp = df2.groupby("bin", observed=True)

acc = grp.apply(lambda g: (g["y_true"] == g["y_pred"]).mean())
cnt = grp.size()

# Midpoints only for bins that exist
x = np.array([interval.mid for interval in acc.index])

fig = plt.figure()
plt.plot(x, acc.values, marker="o")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Confidence bin midpoint")
plt.ylabel("Observed accuracy")
plt.title("Accuracy vs confidence (OOF)")
plt.tight_layout()
clinical_analysis_save_svm_figure_png_svg(fig, "integrated_accuracy_vs_confidence", dpi=300, bbox_inches="tight")
plt.show()

print(pd.DataFrame({"bin": acc.index.astype(str), "bin_mid": x, "accuracy": acc.values, "n": cnt.values}))


In [ ]:
df2 = df.dropna(subset=["confidence"]).copy()
df2["correct"] = (df2["y_true"] == df2["y_pred"]).astype(int)

thresholds = np.linspace(0, 1, 101)
coverage = []
error_rate = []

for t in thresholds:
    kept = df2[df2["confidence"] >= t]
    if len(kept) == 0:
        coverage.append(0.0)
        error_rate.append(np.nan)
        continue
    coverage.append(len(kept) / len(df2))
    error_rate.append(1 - kept["correct"].mean())

fig = plt.figure()
plt.plot(coverage, error_rate)
plt.xlabel("Coverage (fraction kept)")
plt.ylabel("Error rate among kept samples")
plt.title("Reject option: trade coverage for accuracy")
plt.tight_layout()
clinical_analysis_save_svm_figure_png_svg(fig, "integrated_reject_option_coverage_error", dpi=300, bbox_inches="tight")
plt.show()


#### Variable contribution

In [ ]:
from Utils import display_feature_name, original_feature_importance_from_svm

meta_svm = final_metrics['svm_results']['feature_importance_meta']
print(f"Model information: {meta_svm}")

feat_imp_mean = final_metrics['svm_results']['feature_importance_mean']
feat_imp_std  = final_metrics['svm_results']['feature_importance_std']

# If loaded from packed results (dict), convert to Series
if isinstance(feat_imp_mean, dict):
    feat_imp_mean = pd.Series(feat_imp_mean, dtype=float)
if isinstance(feat_imp_std, dict):
    feat_imp_std = pd.Series(feat_imp_std, dtype=float)

# Direct trained-SVM feature contributions. In PCA/reduced runs these are latent
# features, because the assignment SVM is trained in the clustering feature space.
df_imp = (
    pd.DataFrame({
        "feature": feat_imp_mean.index,
        "importance_mean": feat_imp_mean.values,
        "importance_std": feat_imp_std.reindex(feat_imp_mean.index).values if feat_imp_std is not None else None,
    })
    .assign(display_feature=lambda d: d["feature"].map(display_feature_name))
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

# Original/preprocessed variable interpretation. Latent/PCA contributions are
# allocated back to source variables using fitted loadings when available, or
# latent-original correlations for non-linear encoders.
df_imp_original = original_feature_importance_from_svm(final_metrics)
df_imp_original = df_imp_original.assign(
    display_feature=lambda d: d["feature"].map(display_feature_name)
)
print("Original-variable importance mapping:", df_imp_original["importance_source"].value_counts().to_dict())

df_imp_original.head(20)

In [ ]:
import matplotlib.pyplot as plt
from Utils import plot_svm_feature_contributions

# Plot source-variable interpretation rather than latent PCA dimensions.
top_n = 25
source_label = df_imp_original["importance_source"].iloc[0] if not df_imp_original.empty else "original-variable mapping"
fig, ax, plot_df = plot_svm_feature_contributions(
    df_imp_original,
    feature_col="feature",
    importance_col="importance_mean",
    std_col="importance_std",
    display_col="display_feature",
    top_n=top_n,
    xlabel="Original-variable contribution (mean)",
    ylabel="Feature",
    title=f"Top {top_n} original-variable SVM contributions ({source_label})",
)
clinical_analysis_save_svm_figure_png_svg(fig, "integrated_original_variable_svm_contributions", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
fig = plt.figure()
plt.hist(df_imp["importance_mean"].values, bins=50)
plt.xlabel("Feature contribution (mean)")
plt.ylabel("Count")
plt.title("Distribution of feature contributions")
plt.tight_layout()
clinical_analysis_save_svm_figure_png_svg(fig, "integrated_svm_contribution_distribution", dpi=300, bbox_inches="tight")
plt.show()


### Individual labels

#### Accuracy

In [ ]:
modality_names = list(final_metrics["data"].keys())
svm_results_raw = final_metrics.get("svm_results_modalities", [])

from Utils import display_feature_name


svm_results_by_modality = list(clinical_analysis_iter_discovery_svm_results(final_metrics))

for mod_name, res in svm_results_by_modality:
    print(f"SVM results for modality: {mod_name}")

    mean_metrics = (res or {}).get("mean_metrics")
    if mean_metrics:
        for metric, value in mean_metrics.items():
            print(f"SVM Mean {metric}: {value}")
    else:
        print("SVM Mean metrics: None")

    print()  # empty line between modalities

#### Uncertainty

In [ ]:
for mod_name, res in svm_results_by_modality:
    print(f"SVM results for modality: {mod_name}")

    if not res:
        print("No SVM result for this modality.")
        continue

    oof_uncertainty_mod = (res or {}).get("oof_uncertainty")
    if oof_uncertainty_mod is None or oof_uncertainty_mod.empty:
        print(f"No OOF uncertainty results found for modality {mod_name}.")
        continue

    # Find most confident mistakes 
    df = oof_uncertainty_mod

    df_bad = df[(df["y_true"] != df["y_pred"]) & (df["confidence"] > 0.9)]
    df_bad.sort_values("confidence", ascending=False).head(20)
    if df_bad is None or df_bad.empty:
        print(f"No confident mistakes found for modality {mod_name}.")
    else:
        print(df_bad)

    # Find most uncertain predictions
    df_uncertain = df.sort_values(["confidence", "entropy"], ascending=[True, False])
    print(df_uncertain.head(10))

    correct = df["y_true"] == df["y_pred"]

    fig = plt.figure()
    plt.hist(df.loc[correct, "confidence"].dropna(), bins=30, alpha=0.7, label="correct")
    plt.hist(df.loc[~correct, "confidence"].dropna(), bins=30, alpha=0.7, label="wrong")
    plt.xlabel("Confidence (max predicted probability)")
    plt.ylabel("Count")
    plt.title("Confidence distribution: correct vs wrong")
    plt.legend()
    plt.tight_layout()
    clinical_analysis_save_svm_figure_png_svg(fig, f"{clinical_analysis_safe_svm_plot_name(mod_name)}_confidence_distribution", dpi=300, bbox_inches="tight")
    plt.show()

    df2 = df.dropna(subset=["confidence"]).copy()
    bins = np.linspace(0, 1, 11)

    df2["bin"] = pd.cut(df2["confidence"], bins=bins, include_lowest=True)
    grp = df2.groupby("bin", observed=True)

    acc = grp.apply(lambda g: (g["y_true"] == g["y_pred"]).mean())
    cnt = grp.size()

    # Midpoints only for bins that exist
    x = np.array([interval.mid for interval in acc.index])

    fig = plt.figure()
    plt.plot(x, acc.values, marker="o")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Confidence bin midpoint")
    plt.ylabel("Observed accuracy")
    plt.title("Accuracy vs confidence (OOF)")
    plt.tight_layout()
    clinical_analysis_save_svm_figure_png_svg(fig, f"{clinical_analysis_safe_svm_plot_name(mod_name)}_accuracy_vs_confidence", dpi=300, bbox_inches="tight")
    plt.show()

    print(pd.DataFrame({"bin": acc.index.astype(str), "bin_mid": x, "accuracy": acc.values, "n": cnt.values}))

    df2 = df.dropna(subset=["confidence"]).copy()
    df2["correct"] = (df2["y_true"] == df2["y_pred"]).astype(int)

    thresholds = np.linspace(0, 1, 101)
    coverage = []
    error_rate = []

    for t in thresholds:
        kept = df2[df2["confidence"] >= t]
        if len(kept) == 0:
            coverage.append(0.0)
            error_rate.append(np.nan)
            continue
        coverage.append(len(kept) / len(df2))
        error_rate.append(1 - kept["correct"].mean())

    fig = plt.figure()
    plt.plot(coverage, error_rate)
    plt.xlabel("Coverage (fraction kept)")
    plt.ylabel("Error rate among kept samples")
    plt.title("Reject option: trade coverage for accuracy")
    plt.tight_layout()
    clinical_analysis_save_svm_figure_png_svg(fig, f"{clinical_analysis_safe_svm_plot_name(mod_name)}_reject_option_coverage_error", dpi=300, bbox_inches="tight")
    plt.show()

#### Variable contribution

In [ ]:
from Utils import display_feature_name, original_feature_importance_from_svm, plot_svm_feature_contributions
for mod_name, res in svm_results_by_modality:
    print(f"SVM results for modality: {mod_name}")

    if not res:
        print("No SVM result for this modality.")
        continue

    meta_svm = (res or {}).get("feature_importance_meta")
    print(f"Model information: {meta_svm}")

    try:
        df_imp_mod_original = original_feature_importance_from_svm(
            final_metrics,
            svm_result=res,
            top_n=None,
        )
        if "modality" in df_imp_mod_original.columns:
            modality_mask = df_imp_mod_original["modality"].isna() | (df_imp_mod_original["modality"].astype(str) == str(mod_name))
            filtered = df_imp_mod_original.loc[modality_mask].copy()
            if not filtered.empty:
                df_imp_mod_original = filtered
            else:
                print(
                    f"No modality-tagged original-variable rows matched {mod_name}; "
                    "using all rows returned for this modality-specific SVM result."
                )
        df_imp_mod_original = df_imp_mod_original.assign(
            display_feature=lambda d: d["feature"].map(display_feature_name)
        )
    except Exception as exc:
        print(f"Could not derive original-variable importance for {mod_name}: {exc}")
        continue

    if df_imp_mod_original.empty:
        print(f"No original-variable importance rows to plot for {mod_name}.")
        continue

    top_n = 25
    source_label = df_imp_mod_original["importance_source"].iloc[0] if not df_imp_mod_original.empty else "original-variable mapping"
    fig, ax, plot_df = plot_svm_feature_contributions(
        df_imp_mod_original,
        feature_col="feature",
        importance_col="importance_mean",
        std_col="importance_std",
        display_col="display_feature",
        top_n=top_n,
        xlabel="Original-variable contribution (mean)",
        ylabel="Feature",
        title=f"Top original-variable contributions for {mod_name} ({source_label})",
    )
    clinical_analysis_save_svm_figure_png_svg(fig, f"{clinical_analysis_safe_svm_plot_name(mod_name)}_original_variable_svm_contributions", dpi=300, bbox_inches="tight")
    plt.show()

# Apply to test set

## Prepare data

In [ ]:
# Only keep relevant modalities
modalities_keep = ["Psychoticism", "Detachment", "Functioning", "Internalising", "Cognition"]

vars_to_keep = meta["ElementName"][meta["Modality"].isin(modalities_keep)].tolist()
# Filter the data if the columns are present in cleaned_discovery. Also include src_subject_id
vars_to_keep.append("src_subject_id")
vars_to_keep.append("phenotype")
cleaned_test = cleaned_test[cleaned_test.columns.intersection(vars_to_keep)]


## Apply SVM models

Get the test data

In [ ]:
Test_df = pd.read_csv(
    "path/to/multiclust_data/cleaned_test_data.csv"
)


Apply preprocessing

In [ ]:
from Utils import *
#from full_pipeline import preprocessing

# Set parameters
subject_id_column = 'src_subject_id'
col_threshold = 0.5
row_threshold = 0.5
skew_threshold = 0.75
scaler_type = 'robust'  # Options: 'standard', 'minmax', 'robust'
modalities = list(final_metrics["data"].keys())


In [ ]:
import importlib
import Utils
Utils = importlib.reload(Utils)
from Utils import apply_dimensionality_reduction_to_new_data, apply_preprocessing_to_new_data

_validation_preprocessing_details = final_metrics.get("preprocessing_details")
if not isinstance(_validation_preprocessing_details, dict):
    raise KeyError("final_metrics['preprocessing_details'] is required for deployment-style validation preprocessing.")

_pipeline_preproc_params = _validation_preprocessing_details.get("preprocessing_parameters", {}) or {}
_validation_dummy_code_modalities = _pipeline_preproc_params.get("dummy_code_modalities", [])
_validation_mixed_categorical_modalities = _pipeline_preproc_params.get("mixed_categorical_modalities", [])

print("Validation preprocessing choices from discovery pipeline:")
print("  dummy_code_modalities:", _validation_dummy_code_modalities)
print("  mixed_categorical_modalities:", _validation_mixed_categorical_modalities)
print("  imputation_mode: reference (fit on discovery preprocessing reference, transform validation)")

# Apply the fitted discovery preprocessing to the independent validation sample.
# This keeps validation in the same deployment-style feature space as the trained
# discovery SVM/profile-assignment model without fitting preprocessing on validation.
ae_data, subject_id_list, dict_final = apply_preprocessing_to_new_data(
    Test_df,
    meta,
    _validation_preprocessing_details,
    subject_id_column=subject_id_column,
    imputation_mode="reference",
)

test_preprocessing_details = _validation_preprocessing_details  # compatibility for later cells
validation_preprocessing_details = _validation_preprocessing_details
dict_final_test = dict_final

# Assert identical subject order across modalities after preprocessing.
base_ids = dict_final[modalities[0]][subject_id_column].tolist()
for m in modalities[1:]:
    assert dict_final[m][subject_id_column].tolist() == base_ids, (
        f"Subject-ID order mismatch between {modalities[0]} and {m} after preprocessing"
    )

# Project the discovery-preprocessed validation sample into the exact SVM feature
# space learned in discovery. This reuses fitted discovery reducers such as PCA.
ae_res_test, X_test, X_test_by_modality = apply_dimensionality_reduction_to_new_data(
    dict_final_new=dict_final,
    final_metrics=final_metrics,
    modalities=modalities,
    subject_id_column=subject_id_column,
)

ae_res = ae_res_test
_context = final_metrics.get("final_reporting", {}).get("compute_context", {}) if isinstance(final_metrics, dict) else {}
_dimred_by_modality = _context.get("dim_reduction_by_modality", {}) if isinstance(_context, dict) else {}
_dimred_default = _context.get("dim_reduction", "none") if isinstance(_context, dict) else "none"
_validation_dimred_methods = {
    mod: str(_dimred_by_modality.get(mod, _dimred_default)).strip().lower()
    for mod in modalities
}
print("Validation dimensionality reduction methods:", _validation_dimred_methods)
print("Validation dimensionality reduction complete:", {k: v["final_latent"].shape for k, v in ae_res_test.items()})
print("Integrated SVM validation matrix:", X_test.shape)

# Feature-space diagnostics for SVM prediction.
svm_feat = final_metrics.get("svm_feature_names")
if svm_feat is not None:
    missing = [c for c in svm_feat if c not in X_test.columns]
    extra = [c for c in X_test.columns if c not in svm_feat]
    print(f"Integrated SVM feature check: expected={len(svm_feat)}, got={X_test.shape[1]}, missing={len(missing)}, extra={len(extra)}")
    if missing:
        print("Missing integrated feature examples:", missing[:10])
    if extra:
        print("Extra integrated feature examples:", extra[:10])

feat_mods = final_metrics.get("svm_feature_names_modalities", [None] * len(modalities))
for i, mod in enumerate(modalities):
    feat_i = feat_mods[i] if feat_mods is not None and i < len(feat_mods) else None
    cols_i = list(X_test_by_modality[mod].columns)
    uses_latent = bool(cols_i) and all(str(c).startswith(f"{mod}__latent_") for c in cols_i)
    print(f"{mod}: SVM input shape={X_test_by_modality[mod].shape}, latent_features={uses_latent}")
    if feat_i is not None:
        missing_i = [c for c in feat_i if c not in cols_i]
        extra_i = [c for c in cols_i if c not in feat_i]
        print(f"  modality SVM feature check: expected={len(feat_i)}, got={len(cols_i)}, missing={len(missing_i)}, extra={len(extra_i)}")
        if missing_i:
            print("  missing examples:", missing_i[:10])
        if extra_i:
            print("  extra examples:", extra_i[:10])


In [ ]:
## Apply discovery-fitted preprocessing to CC in the validation/test sample

# CC is transformed through the same fitted discovery preprocessing as validation CHR
# so CHR-vs-CC plots are in the same deployment-style feature space.
preproc_test = validation_preprocessing_details

cc_df_test = cleaned_test_CC.copy()
ae_data_cc_test, subject_id_list_cc_test, dict_final_cc_test = apply_preprocessing_to_new_data(
    cc_df_test,
    meta,
    preproc_test,
    subject_id_column="src_subject_id",
    imputation_mode="reference",
)

print("Test CC preprocessing complete:", {k: v.shape for k, v in dict_final_cc_test.items()})


Run model

In [ ]:
svm_final = final_metrics['svm_final_model']
svm_modalities = final_metrics['svm_final_models_modalities']


Integrated clusters

In [ ]:
import pandas as pd
import numpy as np

svm_final = final_metrics["svm_final_model"]

# Align columns to the training feature order and fail loudly if the validation
# representation is not the same SVM feature space.
feat = final_metrics.get("svm_feature_names", None)
if feat is not None and isinstance(X_test, pd.DataFrame):
    missing = [c for c in feat if c not in X_test.columns]
    extra = [c for c in X_test.columns if c not in feat]
    if missing:
        raise ValueError(f"Validation SVM matrix is missing {len(missing)} trained features; examples: {missing[:10]}")
    if extra:
        print(f"Dropping {len(extra)} validation features not used by the integrated SVM; examples: {extra[:10]}")
    X_test_aligned = X_test.loc[:, feat]
else:
    X_test_aligned = X_test

y_pred, proba, confidence, entropy, margin = svm_predict_with_uncertainty(svm_final, X_test_aligned)

pred_final = pd.DataFrame({
    "y_pred": y_pred,
    "confidence": confidence,
    "entropy": entropy,
    "margin": margin,
})

# Optional: add per-class probabilities for debugging.
if proba is not None:
    proba_df = pd.DataFrame(proba, columns=[f"p_{c}" for c in svm_final.classes_])
    pred_final = pd.concat([pred_final, proba_df], axis=1)

test_subject_ids = dict_final[modalities[0]]["src_subject_id"].astype(str).reset_index(drop=True)
if len(test_subject_ids) != len(pred_final):
    raise ValueError(
        f"Test subject ID count ({len(test_subject_ids)}) does not match integrated predictions ({len(pred_final)})."
    )

pred_final.insert(0, "src_subject_id", test_subject_ids)
labels_test_final = pred_final["y_pred"].to_numpy()
test_final_labels_by_subject_id = dict(zip(test_subject_ids, pred_final["y_pred"]))

final_counts = pred_final["y_pred"].astype(str).value_counts().sort_index()
print("Integrated SVM predicted classes:", final_counts.to_dict())
prob_cols = [c for c in pred_final.columns if str(c).startswith("p_")]
if prob_cols:
    prob_summary = pred_final[prob_cols].describe(percentiles=[0.05, 0.5, 0.95]).loc[["mean", "5%", "50%", "95%"]]
    print("Integrated SVM probability summary:")
    display(prob_summary)
if len(final_counts) < 2:
    print("WARNING: Integrated SVM predicted a single class. Check feature-space diagnostics above before interpreting subgroup plots.")

pred_final.head(10)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# pred_final: DataFrame with columns
# ['y_pred', 'confidence', 'entropy', 'margin', 'p_0', 'p_1']

df = pred_final.copy()

# --- (Optional) sanity / recompute checks ---
# If you want to ensure these are consistent with p_0/p_1, uncomment:
# df["confidence_chk"] = np.maximum(df["p_0"], df["p_1"])
# df["margin_chk"] = np.abs(df["p_1"] - df["p_0"])
# eps = 1e-12
# df["entropy_chk"] = -(df["p_0"] * np.log(df["p_0"] + eps) + df["p_1"] * np.log(df["p_1"] + eps))

# --- 1) Histograms ---
fig = plt.figure()
plt.hist(df["p_1"].astype(float), bins=30)
plt.xlabel("Predicted probability p(class=1)")
plt.ylabel("Count")
plt.title("p_1 distribution (unlabeled hold-out)")
clinical_analysis_save_svm_figure_png_svg(fig, "holdout_integrated_p1_distribution", dpi=300, bbox_inches="tight")
plt.show()

fig = plt.figure()
plt.hist(df["confidence"].astype(float), bins=30)
plt.xlabel("Confidence = max(p_0, p_1)")
plt.ylabel("Count")
plt.title("Confidence distribution (unlabeled hold-out)")
clinical_analysis_save_svm_figure_png_svg(fig, "holdout_integrated_confidence_distribution", dpi=300, bbox_inches="tight")
plt.show()

fig = plt.figure()
plt.hist(df["entropy"].astype(float), bins=30)
plt.xlabel("Entropy (higher = more uncertain)")
plt.ylabel("Count")
plt.title("Entropy distribution (unlabeled hold-out)")
clinical_analysis_save_svm_figure_png_svg(fig, "holdout_integrated_entropy_distribution", dpi=300, bbox_inches="tight")
plt.show()

fig = plt.figure()
plt.hist(df["margin"].astype(float), bins=30)
plt.xlabel("Margin |p_1 - p_0| (lower = more uncertain)")
plt.ylabel("Count")
plt.title("Margin distribution (unlabeled hold-out)")
clinical_analysis_save_svm_figure_png_svg(fig, "holdout_integrated_margin_distribution", dpi=300, bbox_inches="tight")
plt.show()

# --- 2) Scatter plots ---
fig = plt.figure()
plt.scatter(df["confidence"].astype(float), df["entropy"].astype(float), s=10)
plt.xlabel("Confidence")
plt.ylabel("Entropy")
plt.title("Confidence vs entropy")
clinical_analysis_save_svm_figure_png_svg(fig, "holdout_integrated_confidence_vs_entropy", dpi=300, bbox_inches="tight")
plt.show()

fig = plt.figure()
plt.scatter(df["confidence"].astype(float), df["margin"].astype(float), s=10)
plt.xlabel("Confidence")
plt.ylabel("Margin")
plt.title("Confidence vs margin")
clinical_analysis_save_svm_figure_png_svg(fig, "holdout_integrated_confidence_vs_margin", dpi=300, bbox_inches="tight")
plt.show()

# --- 3) Quick numeric summaries (handy for write-up) ---
conf = df["confidence"].astype(float).to_numpy()
print("\n=== Summary (unlabeled) ===")
print(f"N = {len(df)}")
print(f"Confidence mean={conf.mean():.3f}, median={np.median(conf):.3f}, min={conf.min():.3f}, max={conf.max():.3f}")
for thr in [0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
    print(f"Fraction with confidence ≥ {thr}: {(conf >= thr).mean():.3f}")

# --- 4) Show most-uncertain rows for manual inspection ---
# (lowest confidence, highest entropy, lowest margin)
print("\nLowest confidence (top 10):")
display(df.sort_values("confidence", ascending=True).head(10))

print("\nHighest entropy (top 10):")
display(df.sort_values("entropy", ascending=False).head(10))

print("\nLowest margin (top 10):")
display(df.sort_values("margin", ascending=True).head(10))


Individual modalities

In [ ]:
svm_modalities = final_metrics["svm_final_models_modalities"]
subject_id_column = 'src_subject_id'

pred_modalities = {}
labels_test_modalities = []
labels_test_by_modality = {}

feat_mods = final_metrics.get("svm_feature_names_modalities", [None] * len(modalities))

for i, mod in enumerate(modalities):
    model_mod = svm_modalities[i]
    if model_mod is None:
        print(f"Skipping SVM prediction for modality {mod} due to lack of trained model.")
        pred_modalities[mod] = None
        labels_test_by_modality[mod] = None
        labels_test_modalities.append(None)
        continue

    X_mod = X_test_by_modality[mod]

    # Align to training feature order for this modality model and fail loudly if
    # the validation representation is not the same SVM feature space.
    feat_i = feat_mods[i] if feat_mods is not None and i < len(feat_mods) else None
    if feat_i is not None and isinstance(X_mod, pd.DataFrame):
        missing_i = [c for c in feat_i if c not in X_mod.columns]
        extra_i = [c for c in X_mod.columns if c not in feat_i]
        if missing_i:
            raise ValueError(f"{mod} validation SVM matrix is missing {len(missing_i)} trained features; examples: {missing_i[:10]}")
        if extra_i:
            print(f"{mod}: dropping {len(extra_i)} validation features not used by this SVM; examples: {extra_i[:10]}")
        X_mod_aligned = X_mod.loc[:, feat_i]
    else:
        X_mod_aligned = X_mod

    y_pred, proba, confidence, entropy, margin = svm_predict_with_uncertainty(model_mod, X_mod_aligned)

    out = pd.DataFrame({
        "y_pred": y_pred,
        "confidence": confidence,
        "entropy": entropy,
        "margin": margin,
    })

    if proba is not None:
        proba_df = pd.DataFrame(proba, columns=[f"p_{c}" for c in model_mod.classes_])
        out = pd.concat([out, proba_df], axis=1)

    pred_modalities[mod] = out
    labels_test_by_modality[mod] = y_pred
    labels_test_modalities.append(y_pred)

    counts = pd.Series(y_pred).astype(str).value_counts().sort_index()
    print(f"{mod}: modality SVM predicted classes:", counts.to_dict())
    prob_cols = [c for c in out.columns if str(c).startswith("p_")]
    if prob_cols:
        prob_summary = out[prob_cols].describe(percentiles=[0.05, 0.5, 0.95]).loc[["mean", "5%", "50%", "95%"]]
        print(f"{mod}: probability summary:")
        display(prob_summary)
    if len(counts) < 2:
        print(f"WARNING: {mod} modality SVM predicted a single class. Domain-specific subgroup plots will be skipped for this modality.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use the shared plot pred modality helper.
from Utils import clinical_analysis_plot_pred_modality


# ---- Run for your pred_modalities dict ----
for modality_name, modality_df in pred_modalities.items():
    print(f"\n\n=== Diagnostics for modality: {modality_name} ===")
    clinical_analysis_plot_pred_modality(modality_df, modality_name)


## Visualise test labels

### Differences in original variables - individual labels

In [ ]:

from sklearn.feature_selection import f_classif
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from Utils import display_feature_name

out_dir = os.path.join(plots_dir, "merged_feature_differences")
os.makedirs(out_dir, exist_ok=True)

top_k = 10
sample_name = "validation"
label_scope = "individual_labels"
validation_feature_data = dict_final_test if "dict_final_test" in globals() and isinstance(dict_final_test, dict) else dict_final
validation_cc_data = dict_final_cc_test if "dict_final_cc_test" in globals() and isinstance(dict_final_cc_test, dict) else {}

for mod_num, (modality, df) in enumerate(validation_feature_data.items()):
    print(f"\n=== {sample_name} / {label_scope}: {modality} ===")

    if "labels_test_by_modality" in globals() and isinstance(labels_test_by_modality, dict):
        clusters = labels_test_by_modality.get(modality)
    else:
        clusters = labels_test_modalities[mod_num]
    if clusters is None:
        print(f"Skipping {modality}: no validation individual-label predictions.")
        continue
    clusters = np.asarray(clusters)

    feature_df = df.drop(columns=['src_subject_id'])
    X = feature_df.values
    feature_names = feature_df.columns

    if len(clusters) != len(df):
        raise ValueError(f"{modality}: validation label length ({len(clusters)}) != data rows ({len(df)})")
    if len(pd.unique(clusters)) < 2:
        print(f"Skipping {modality}: fewer than two validation individual labels.")
        continue

    f_vals, _ = f_classif(X, clusters)
    f_df = (
        pd.DataFrame({'feature': feature_names, 'f_value': f_vals})
        .assign(display_feature=lambda d: d['feature'].map(display_feature_name))
        .sort_values('f_value', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x='f_value', y='display_feature', data=f_df.head(top_k), palette=sns.color_palette(n_colors=top_k), ax=ax)
    ax.set_title(f"{modality} — Top {top_k} Discriminative Features (ANOVA F-value; validation individual labels)")
    ax.set_xlabel("F-value")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{sample_name}_{label_scope}_{modality}_top_features_bar.png"))
    plt.show()

    top_features = f_df['feature'].head(top_k).tolist()
    max_cols = 3
    n_rows = int(np.ceil(len(top_features) / max_cols))
    fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
    axes = np.array(axes).reshape(-1)

    cluster_order = [str(x) for x in np.unique(clusters)]
    df_cc = validation_cc_data.get(modality)
    cc_available = df_cc is not None
    if cc_available:
        df_cc = df_cc.copy().reindex(columns=df.columns, fill_value=np.nan)
    group_order = cluster_order + (["CC"] if cc_available else [])
    pal = sns.color_palette(n_colors=len(group_order))

    for i, feat in enumerate(top_features):
        ax = axes[i]
        plot_df = pd.DataFrame({
            'group': pd.Series(clusters).astype(str),
            'value': df[feat].values,
        }).dropna()
        if cc_available and feat in df_cc.columns:
            cc_plot_df = pd.DataFrame({'group': 'CC', 'value': df_cc[feat].values}).dropna()
            plot_df = pd.concat([plot_df, cc_plot_df], ignore_index=True)
        sns.boxplot(data=plot_df, x='group', y='value', order=group_order, palette=pal, ax=ax)
        sns.stripplot(data=plot_df, x='group', y='value', order=group_order, color='black', size=4, jitter=True, alpha=0.5, ax=ax)
        ax.set_title(display_feature_name(feat), fontsize=14)
        ax.set_xlabel("", fontsize=22)
        ax.set_ylabel("", fontsize=22)
        ax.tick_params(axis='both', labelsize=14)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{modality} — Distribution of Top Features by Validation Individual Labels and CC\n(with individual data points)",
        y=1.02, fontsize=18
    )
    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{sample_name}_{label_scope}_{modality}_top_features_with_cc.png"))
    plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from Utils import display_feature_name

# =========================
# PCA aggregated plots per modality + variance explained (first 5 PCs)
# - Respects your existing global theme (NO sns.set_theme here)
# - PC1 by cluster: violin (quartiles) + jitter + median marker + n labels
# - Variance plot: explained variance ratio for PC1..PC5
# - Optional: PC1 loadings (top +/- contributors)
# =========================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

loadings_dir = os.path.join(out_dir, "pc1_loadings/")
os.makedirs(loadings_dir, exist_ok=True)

variance_dir = os.path.join(out_dir, "variance/")
os.makedirs(variance_dir, exist_ok=True)

mod_num = 0
for modality, df in dict_final.items():
    print(f"\n=== PCA Aggregated Plots test sample — Modality: {modality} ===")

    # Extract data and labels
    X = df.drop(columns=['src_subject_id']).values
    clusters = labels_test_modalities[mod_num]
    feature_names = df.drop(columns=['src_subject_id']).columns

    # Stable cluster order (customize if you want a specific ordering)
    cluster_order = np.sort(pd.unique(clusters))

    # --- Standardize ---
    Xz = StandardScaler().fit_transform(X)

    # --- PCA for variance (first 5 components) ---
    n_pcs = min(5, Xz.shape[1])  # cannot exceed number of features
    pca_var = PCA(n_components=n_pcs, random_state=0)
    pca_var.fit(Xz)

    evr = pca_var.explained_variance_ratio_
    cum_evr = np.cumsum(evr)

    # --- Also compute PC scores (at least PC1) ---
    pc_scores = pca_var.transform(Xz)  # shape: (n_samples, n_pcs)
    pc1 = pc_scores[:, 0]
    evr1 = float(evr[0])
    print(f"PC1 EVR: {evr1:.2%}")

    plot_df = pd.DataFrame({"cluster": clusters, "PC1": pc1})

    # -------------------------
    # 1) PC1 by cluster (publication-ready, respects global theme)
    # -------------------------
    fig, ax = plt.subplots(figsize=(10.5, 6.5))


    sns.violinplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        palette=sns.color_palette(n_colors=len(cluster_order)),
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Median marker per cluster
    medians = plot_df.groupby("cluster")["PC1"].median()
    x_positions = np.arange(len(cluster_order))
    ax.scatter(
        x=x_positions,
        y=[medians.loc[c] for c in cluster_order],
        s=180,
        marker="_",
        linewidths=3
    )

    # Annotate n per cluster near the bottom
    counts = plot_df["cluster"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, c in enumerate(cluster_order):
        ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="bottom", fontsize=12)

    ax.set_xlabel("Cluster", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)

    # Light grid for readability; remove if your global theme already handles grids
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"test_{modality}_PC1_violin_pubready.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 2) Variance explained (PC1..PC5): bar + cumulative line
    # -------------------------
    figv, axv = plt.subplots(figsize=(10, 5.5))

    pcs_idx = np.arange(1, n_pcs + 1)
    axv.bar(pcs_idx, evr)  # uses your global matplotlib color cycle
    axv.plot(pcs_idx, cum_evr, marker="o")

    axv.set_xticks(pcs_idx)
    axv.set_xlabel("Principal Component")
    axv.set_ylabel("Explained variance ratio")
    axv.set_title(f"{modality} — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

    axv.set_ylim(0, max(0.25, evr.max() * 1.2))  # keeps plot readable if EVR is small/large
    axv.grid(axis="y", alpha=0.15)
    sns.despine(ax=axv)

    figv.tight_layout()
    clinical_analysis_save_figure_png_pdf(figv, os.path.join(variance_dir, f"test_{modality}_variance_top{n_pcs}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 3) Optional: PC1 feature loadings (top +/- contributors)
    # -------------------------
    loadings = pca_var.components_[0]  # PC1 loadings
    load_df = pd.DataFrame({"feature": feature_names, "loading": loadings})
    load_df["display_feature"] = load_df["feature"].map(display_feature_name)

    top_n = min(15, len(feature_names) // 2) if len(feature_names) >= 2 else 1
    top_pos = load_df.sort_values("loading", ascending=False).head(top_n)
    top_neg = load_df.sort_values("loading", ascending=True).head(top_n)
    load_plot_df = pd.concat([top_neg, top_pos], axis=0)

    fig2, ax2 = plt.subplots(figsize=(10.5, 7.5))
    sns.barplot(data=load_plot_df, x="loading", y="display_feature", ax=ax2)
    ax2.axvline(0, linewidth=1)

    ax2.set_title(f"{modality} — PC1 Feature Loadings (Top ±{top_n})", pad=12)
    ax2.set_xlabel("PC1 loading")
    ax2.set_ylabel("")

    ax2.grid(axis="x", alpha=0.15)
    sns.despine(ax=ax2)

    fig2.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig2, os.path.join(loadings_dir, f"test_{modality}_PC1_loadings_top_pm{top_n}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    mod_num += 1


Different colours

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import Utils
Utils = importlib.reload(Utils)
from Utils import build_group_palette

Utils = importlib.reload(theme)
from theme import CC_COLOR

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# =========================
# PCA aggregated plots per modality: CHR clusters + CC
# - Each modality can have its own manual colors
# - Fits scaler+PCA on CHR only, projects CC into same space
# - Black jittered points always in front of violins
# =========================

# Required inputs:
# final_metrics  -> CHR final metrics dict
# dict_final_cc_test -> output of apply_preprocessing_to_new_data(...): test CC per modality
# plots_dir      -> base directory for plots

out_dir = os.path.join(plots_dir, "merged_feature_pca_chr_vs_cc")
os.makedirs(out_dir, exist_ok=True)

# --------------------------------------------------
# Per-modality colours come from Utils.MODALITY_CLUSTER_PALETTES, the same
# palette already used for the t-SNE/PCA latent-space plots elsewhere in
# this notebook. CC always uses the shared reference colour from
# theme.CC_COLOR (baked into MODALITY_CLUSTER_PALETTES as _THEME_CC).
# --------------------------------------------------

mod_num = 0
for modality, df_chr in dict_final.items():
    print(f"\n=== PCA Aggregated Plots test sample — Modality: {modality} ===")

    # CHR features + labels
    X_chr_df = df_chr.drop(columns=["src_subject_id"]).copy()
    clusters_chr = np.asarray(labels_test_modalities[mod_num])

    # CC features
    df_cc = dict_final_cc_test[modality]
    X_cc_df = df_cc.drop(columns=["src_subject_id"]).copy()

    # Strict feature alignment to CHR
    X_cc_df = X_cc_df.reindex(columns=X_chr_df.columns)

    # Optional diagnostics
    print("CHR shape:", X_chr_df.shape, "CC shape:", X_cc_df.shape)
    print("NaNs in CC feature matrix:", int(X_cc_df.isna().sum().sum()))

    # Convert to arrays
    X_chr = X_chr_df.values
    X_cc = X_cc_df.values

    # Fit scaler + PCA on CHR only
    scaler = StandardScaler()
    X_chr_z = scaler.fit_transform(X_chr)
    X_cc_z = scaler.transform(X_cc)

    n_pcs = min(5, X_chr_z.shape[1])
    pca = PCA(n_components=n_pcs, random_state=0)
    PC_chr = pca.fit_transform(X_chr_z)
    PC_cc = pca.transform(X_cc_z)

    pc1_chr = PC_chr[:, 0]
    pc1_cc = PC_cc[:, 0]

    # Plot dataframe
    plot_chr = pd.DataFrame({
        "group": clusters_chr.astype(str),
        "PC1": pc1_chr,
        "cohort": "CHR"
    })
    plot_cc = pd.DataFrame({
        "group": "CC",
        "PC1": pc1_cc,
        "cohort": "CC"
    })
    plot_df = pd.concat([plot_chr, plot_cc], ignore_index=True)

    cluster_order = sorted(
        plot_chr["group"].unique(),
        key=lambda x: int(x) if str(x).isdigit() else str(x)
    )
    group_order = cluster_order + ["CC"]

    # ---------------------------------------------
    # Pick palette for this modality (shared with the rest of the notebook)
    # ---------------------------------------------
    group_palette = build_group_palette(modality, group_order)

    fig, ax = plt.subplots(figsize=(10.5, 6.5))

    # Violin by group color
    sns.violinplot(
        data=plot_df,
        x="group",
        y="PC1",
        hue="group",
        order=group_order,
        hue_order=group_order,
        palette=group_palette,
        dodge=False,
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        legend=False,
        ax=ax
    )

    # Black points on top
    sns.stripplot(
        data=plot_df,
        x="group",
        y="PC1",
        order=group_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Bring points forward
    for c in ax.collections:
        c.set_zorder(2)
    for l in ax.lines:
        l.set_zorder(3)

    # Median marker
    medians = plot_df.groupby("group")["PC1"].median()
    x_positions = np.arange(len(group_order))
    med_vals = [medians.loc[g] for g in group_order]
    ax.scatter(
        x_positions, med_vals,
        marker="_", s=180, linewidths=3,
        color="black", zorder=4
    )

    # n labels
    counts = plot_df["group"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, g in enumerate(group_order):
        ax.text(i, y_annot, f"n={int(counts.get(g, 0))}",
                ha="center", va="top", fontsize=14)

    ax.set_xlabel("CHR clusters + CC", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)
    ax.tick_params(axis="both", labelsize=14)
    ax.set_title(f"{modality}: CHR cluster PC1 vs CC", fontsize=16)
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    clinical_analysis_save_figure_png_pdf(
        fig,
        os.path.join(out_dir, f"test_{modality}_PC1_CHR_vs_CC.png"),
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

    mod_num += 1


### Differences in original variables - final labels

In [ ]:

from sklearn.feature_selection import f_classif
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from Utils import display_feature_name

out_dir = os.path.join(plots_dir, "merged_feature_differences")
os.makedirs(out_dir, exist_ok=True)

top_k = 10
sample_name = "validation"
label_scope = "final_labels"
validation_feature_data = dict_final_test if "dict_final_test" in globals() and isinstance(dict_final_test, dict) else dict_final
validation_cc_data = dict_final_cc_test if "dict_final_cc_test" in globals() and isinstance(dict_final_cc_test, dict) else {}

for modality, df in validation_feature_data.items():
    print(f"\n=== {sample_name} / {label_scope}: {modality} ===")

    feature_df = df.drop(columns=['src_subject_id'])
    X = feature_df.values
    clusters = np.asarray(labels_test_final)
    feature_names = feature_df.columns

    if len(clusters) != len(df):
        raise ValueError(f"{modality}: validation final label length ({len(clusters)}) != data rows ({len(df)})")
    if len(pd.unique(clusters)) < 2:
        print(f"Skipping {modality}: fewer than two validation final labels.")
        continue

    f_vals, _ = f_classif(X, clusters)
    f_df = (
        pd.DataFrame({'feature': feature_names, 'f_value': f_vals})
        .assign(display_feature=lambda d: d['feature'].map(display_feature_name))
        .sort_values('f_value', ascending=False)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(x='f_value', y='display_feature', data=f_df.head(top_k), palette=sns.color_palette(n_colors=top_k), ax=ax)
    ax.set_title(f"{modality} — Top {top_k} Discriminative Features (ANOVA F-value; validation final labels)")
    ax.set_xlabel("F-value")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{sample_name}_{label_scope}_{modality}_top_features_bar.png"))
    plt.show()

    top_features = f_df['feature'].head(top_k).tolist()
    max_cols = 3
    n_rows = int(np.ceil(len(top_features) / max_cols))
    fig, axes = plt.subplots(n_rows, max_cols, figsize=(max_cols * 6, n_rows * 5))
    axes = np.array(axes).reshape(-1)

    cluster_order = [str(x) for x in np.unique(clusters)]
    df_cc = validation_cc_data.get(modality)
    cc_available = df_cc is not None
    if cc_available:
        df_cc = df_cc.copy().reindex(columns=df.columns, fill_value=np.nan)
    group_order = cluster_order + (["CC"] if cc_available else [])
    pal = sns.color_palette(n_colors=len(group_order))

    for i, feat in enumerate(top_features):
        ax = axes[i]
        plot_df = pd.DataFrame({
            'group': pd.Series(clusters).astype(str),
            'value': df[feat].values,
        }).dropna()
        if cc_available and feat in df_cc.columns:
            cc_plot_df = pd.DataFrame({'group': 'CC', 'value': df_cc[feat].values}).dropna()
            plot_df = pd.concat([plot_df, cc_plot_df], ignore_index=True)
        sns.boxplot(data=plot_df, x='group', y='value', order=group_order, palette=pal, ax=ax)
        sns.stripplot(data=plot_df, x='group', y='value', order=group_order, color='black', size=4, jitter=True, alpha=0.5, ax=ax)
        ax.set_title(display_feature_name(feat), fontsize=14)
        ax.set_xlabel("", fontsize=22)
        ax.set_ylabel("", fontsize=22)
        ax.tick_params(axis='both', labelsize=14)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle(
        f"{modality} — Distribution of Top Features by Validation Final Labels and CC\n(with individual data points)",
        y=1.02, fontsize=18
    )
    plt.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{sample_name}_{label_scope}_{modality}_top_features_with_cc.png"))
    plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from Utils import display_feature_name

# =========================
# PCA aggregated plots per modality + variance explained (first 5 PCs)
# - Respects your existing global theme (NO sns.set_theme here)
# - PC1 by cluster: violin (quartiles) + jitter + median marker + n labels
# - Variance plot: explained variance ratio for PC1..PC5
# - Optional: PC1 loadings (top +/- contributors)
# =========================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

loadings_dir = os.path.join(out_dir, "pc1_loadings/")
os.makedirs(loadings_dir, exist_ok=True)

variance_dir = os.path.join(out_dir, "variance/")
os.makedirs(variance_dir, exist_ok=True)

mod_num = 0
for modality, df in dict_final.items():
    print(f"\n=== PCA Aggregated Plots test sample — Modality: {modality} ===")

    # Extract data and labels
    X = df.drop(columns=['src_subject_id']).values
    clusters = labels_test_final
    feature_names = df.drop(columns=['src_subject_id']).columns

    # Stable cluster order (customize if you want a specific ordering)
    cluster_order = np.sort(pd.unique(clusters))

    # --- Standardize ---
    Xz = StandardScaler().fit_transform(X)

    # --- PCA for variance (first 5 components) ---
    n_pcs = min(5, Xz.shape[1])  # cannot exceed number of features
    pca_var = PCA(n_components=n_pcs, random_state=0)
    pca_var.fit(Xz)

    evr = pca_var.explained_variance_ratio_
    cum_evr = np.cumsum(evr)

    # --- Also compute PC scores (at least PC1) ---
    pc_scores = pca_var.transform(Xz)  # shape: (n_samples, n_pcs)
    pc1 = pc_scores[:, 0]
    evr1 = float(evr[0])
    print(f"PC1 EVR: {evr1:.2%}")

    plot_df = pd.DataFrame({"cluster": clusters, "PC1": pc1})

    # -------------------------
    # 1) PC1 by cluster (publication-ready, respects global theme)
    # -------------------------
    fig, ax = plt.subplots(figsize=(10.5, 6.5))


    sns.violinplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        palette=sns.color_palette(n_colors=len(cluster_order)),
        inner="quartile",
        cut=0,
        bw_adjust=1.0,
        linewidth=1,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x="cluster",
        y="PC1",
        order=cluster_order,
        color="black",
        size=3,
        jitter=0.22,
        alpha=0.35,
        ax=ax
    )

    # Median marker per cluster
    medians = plot_df.groupby("cluster")["PC1"].median()
    x_positions = np.arange(len(cluster_order))
    ax.scatter(
        x=x_positions,
        y=[medians.loc[c] for c in cluster_order],
        s=180,
        marker="_",
        linewidths=3
    )

    # Annotate n per cluster near the bottom
    counts = plot_df["cluster"].value_counts()
    ymin, ymax = ax.get_ylim()
    y_annot = ymin + 0.04 * (ymax - ymin)
    for i, c in enumerate(cluster_order):
        ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="bottom", fontsize=12)

    ax.set_xlabel("Cluster", fontsize=16)
    ax.set_ylabel("PCA Component 1 score", fontsize=16)

    # Light grid for readability; remove if your global theme already handles grids
    ax.grid(axis="y", alpha=0.15)
    sns.despine(ax=ax)

    fig.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, f"{modality}_PC1_violin_pubready.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 2) Variance explained (PC1..PC5): bar + cumulative line
    # -------------------------
    figv, axv = plt.subplots(figsize=(10, 5.5))

    pcs_idx = np.arange(1, n_pcs + 1)
    axv.bar(pcs_idx, evr)  # uses your global matplotlib color cycle
    axv.plot(pcs_idx, cum_evr, marker="o")

    axv.set_xticks(pcs_idx)
    axv.set_xlabel("Principal Component")
    axv.set_ylabel("Explained variance ratio")
    axv.set_title(f"{modality} — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

    axv.set_ylim(0, max(0.25, evr.max() * 1.2))  # keeps plot readable if EVR is small/large
    axv.grid(axis="y", alpha=0.15)
    sns.despine(ax=axv)

    figv.tight_layout()
    clinical_analysis_save_figure_png_pdf(figv, os.path.join(variance_dir, f"{modality}_variance_top{n_pcs}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    # -------------------------
    # 3) Optional: PC1 feature loadings (top +/- contributors)
    # -------------------------
    loadings = pca_var.components_[0]  # PC1 loadings
    load_df = pd.DataFrame({"feature": feature_names, "loading": loadings})
    load_df["display_feature"] = load_df["feature"].map(display_feature_name)

    top_n = min(15, len(feature_names) // 2) if len(feature_names) >= 2 else 1
    top_pos = load_df.sort_values("loading", ascending=False).head(top_n)
    top_neg = load_df.sort_values("loading", ascending=True).head(top_n)
    load_plot_df = pd.concat([top_neg, top_pos], axis=0)

    fig2, ax2 = plt.subplots(figsize=(10.5, 7.5))
    sns.barplot(data=load_plot_df, x="loading", y="display_feature", ax=ax2)
    ax2.axvline(0, linewidth=1)

    ax2.set_title(f"{modality} — PC1 Feature Loadings (Top ±{top_n})", pad=12)
    ax2.set_xlabel("PC1 loading")
    ax2.set_ylabel("")

    ax2.grid(axis="x", alpha=0.15)
    sns.despine(ax=ax2)

    fig2.tight_layout()
    clinical_analysis_save_figure_png_pdf(fig2, os.path.join(loadings_dir, f"{modality}_PC1_loadings_top_pm{top_n}.png"), dpi=300, bbox_inches="tight")
    plt.show()

    mod_num += 1


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# GLOBAL PCA (ALL modalities merged) -> PC1 by final clusters
# - Merges modalities on src_subject_id (inner-join by default)
# - Standardizes all features
# - PCA -> PC1
# - Publication-ready violin (quartiles) + jitter + median + n
# - Also saves a variance plot (top 5 PCs) for the merged space
# ==========================================================

global_out_dir = os.path.join(plots_dir, "global_pca_all_modalities/")
os.makedirs(global_out_dir, exist_ok=True)

# ---- 1) Merge all modalities into one wide dataframe ----
dfs = []
for modality, df in dict_final.items():
    tmp = df.copy()

    # Prefix feature names with modality to avoid collisions
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})

    dfs.append(tmp)

# Inner join across modalities by subject id (keeps subjects present in ALL modalities)
merged = dfs[0]
for d in dfs[1:]:
    merged = merged.merge(d, on="src_subject_id", how="inner")

print(f"[GLOBAL PCA] Subjects after inner-join across modalities: {merged.shape[0]}")
print(f"[GLOBAL PCA] Total merged features: {merged.shape[1] - 1}")

# ---- 2) Align test integrated labels to the merged subject IDs ----
merged_clusters = merged["src_subject_id"].astype(str).map(test_final_labels_by_subject_id).to_numpy()
if np.any(pd.isna(merged_clusters)):
    missing = merged.loc[pd.isna(merged_clusters), "src_subject_id"].head(5).tolist()
    raise ValueError(f"Missing test integrated labels for merged subjects. Examples: {missing}")

# ---- 3) PCA on all features ----
X = merged.drop(columns=["src_subject_id"]).values

# Standardize before PCA
Xz = StandardScaler().fit_transform(X)

# Fit PCA (enough components to report variance for first 5, but at least 2 if possible)
n_pcs = min(5, Xz.shape[1])
pca = PCA(n_components=n_pcs, random_state=0)
scores = pca.fit_transform(Xz)

pc1 = scores[:, 0]
evr = pca.explained_variance_ratio_
cum_evr = np.cumsum(evr)

print(f"[GLOBAL PCA] PC1 EVR: {evr[0]:.2%}")

# ---- 4) Plot PC1 by cluster (global / all modalities) ----
plot_df = pd.DataFrame({"cluster": merged_clusters, "PC1": pc1})
cluster_order = np.sort(pd.unique(plot_df["cluster"]))

fig, ax = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=plot_df,
    x="cluster",
    y="PC1",
    order=cluster_order,
    palette=sns.color_palette(n_colors=len(cluster_order)),
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="cluster",
    y="PC1",
    order=cluster_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.35,
    ax=ax
)

# Median marker per cluster
medians = plot_df.groupby("cluster")["PC1"].median()
x_positions = np.arange(len(cluster_order))
ax.scatter(
    x=x_positions,
    y=[medians.loc[c] for c in cluster_order],
    s=180,
    marker="_",
    linewidths=3
)

# Annotate n per cluster near bottom
counts = plot_df["cluster"].value_counts()
ymin, ymax = ax.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
for i, c in enumerate(cluster_order):
    ax.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="bottom", fontsize=12)

ax.set_xlabel("Cluster", fontsize=16)
ax.set_ylabel("PCA Component 1 score", fontsize=16)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

fig.tight_layout()
clinical_analysis_save_figure_png_pdf(fig, os.path.join(global_out_dir, "GLOBAL_all_modalities_PC1_violin.png"), dpi=300, bbox_inches="tight")
plt.show()

# ---- 5) Variance explained plot (Top PCs) ----
figv, axv = plt.subplots(figsize=(10, 5.5))
pcs_idx = np.arange(1, n_pcs + 1)

axv.bar(pcs_idx, evr)
axv.plot(pcs_idx, cum_evr, marker="o")

axv.set_xticks(pcs_idx)
axv.set_xlabel("Principal Component")
axv.set_ylabel("Explained variance ratio")
axv.set_title(f"GLOBAL (All Modalities) — PCA Variance Explained (Top {n_pcs} PCs)", pad=12)

axv.grid(axis="y", alpha=0.15)
sns.despine(ax=axv)

figv.tight_layout()
clinical_analysis_save_figure_png_pdf(figv, os.path.join(global_out_dir, f"GLOBAL_all_modalities_variance_top{n_pcs}.png"),
             dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# 1) SINGLE PLOT: PC1 (per-modality PCA) distributions by modality (hue=integrated cluster)
# 2) ADDITIONAL "GLOBAL" DIFFERENCE: one shared PC1 computed from ALL features across ALL modalities
#    -> a separate global violin plot + optional printout of EVR
#
# IMPORTANT: Test integrated labels are aligned to src_subject_id via
# test_final_labels_by_subject_id, created from the test predictions.
# ==========================================================

out_dir = os.path.join(plots_dir, "merged_feature_pca/")
os.makedirs(out_dir, exist_ok=True)

# --------------------------
# A) Per-modality PC1 plot (your current single plot)
# --------------------------
rows = []
for modality, df in dict_final.items():
    feature_df = df.drop(columns=["src_subject_id"])
    X = feature_df.values
    clusters = df["src_subject_id"].astype(str).map(test_final_labels_by_subject_id).to_numpy()
    if np.any(pd.isna(clusters)):
        missing = df.loc[pd.isna(clusters), "src_subject_id"].head(5).tolist()
        raise ValueError(f"Missing test integrated labels for modality {modality}. Examples: {missing}")

    Xz = StandardScaler().fit_transform(X)
    pca = PCA(n_components=1, random_state=0).fit(Xz)
    pc1 = pca.transform(Xz)[:, 0]
    evr1 = float(pca.explained_variance_ratio_[0])

    tmp = pd.DataFrame({"modality": modality, "cluster": clusters, "PC1": pc1})
    tmp["PC1_EVR"] = evr1
    rows.append(tmp)

    print(f"{modality}: PC1 EVR={evr1:.2%}")

plot_df = pd.concat(rows, ignore_index=True)

modality_order = list(dict_final.keys())
cluster_order = np.sort(plot_df["cluster"].unique())

fig_w = max(18, 2.2 * len(modality_order))
fig_h = 9
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.violinplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="cluster",
    order=modality_order,
    hue_order=cluster_order,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    dodge=True,
    width=0.65,
    ax=ax
)

sns.stripplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="cluster",
    order=modality_order,
    hue_order=cluster_order,
    dodge=True,
    jitter=0.18,
    size=2.2,
    alpha=0.18,
    color="black",
    ax=ax
)

handles, labels = ax.get_legend_handles_labels()
ax.legend(handles[:len(cluster_order)], labels[:len(cluster_order)], title="Cluster", frameon=False, loc="upper right")

for x in np.arange(0.5, len(modality_order), 1.0):
    ax.axvline(x, linewidth=0.8, alpha=0.25)

ax.set_xlabel("Modality", fontsize=16)
ax.set_ylabel("PCA Component 1 score", fontsize=16)
ax.set_title("PC1 Distributions by Modality (Integrated Cluster Differences)", fontsize=18, pad=14)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=13)
plt.setp(ax.get_yticklabels(), fontsize=13)

fig.tight_layout()
clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, "test_ALL_modalities_PC1_singleplot_violin_big.png"), dpi=300, bbox_inches="tight")
plt.show()


# --------------------------
# B) GLOBAL PCA PC1 across ALL modalities/features (shared PC axis)
# --------------------------

# 1) Merge all modalities into one wide table keyed by src_subject_id
dfs = []
for modality, df in dict_final.items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})  # avoid name collisions
    dfs.append(tmp)

merged = dfs[0]
for d in dfs[1:]:
    merged = merged.merge(d, on="src_subject_id", how="inner")  # subjects present in ALL modalities

print(f"\n[GLOBAL PCA] Subjects after inner-join: {merged.shape[0]}")
print(f"[GLOBAL PCA] Total merged features: {merged.shape[1] - 1}")

# 2) Align integrated test labels to merged subject IDs
merged_clusters = merged["src_subject_id"].astype(str).map(test_final_labels_by_subject_id).to_numpy()
if np.any(pd.isna(merged_clusters)):
    missing = merged.loc[pd.isna(merged_clusters), "src_subject_id"].head(5).tolist()
    raise ValueError(
        "Some merged test subjects are missing labels in test_final_labels_by_subject_id. "
        f"Examples: {missing}"
    )

# 3) Global PCA (PC1)
X_global = merged.drop(columns=["src_subject_id"]).values
Xg_z = StandardScaler().fit_transform(X_global)

pca_global = PCA(n_components=1, random_state=0).fit(Xg_z)
pc1_global = pca_global.transform(Xg_z)[:, 0]
evr1_global = float(pca_global.explained_variance_ratio_[0])
print(f"[GLOBAL PCA] PC1 EVR: {evr1_global:.2%}")

global_df = pd.DataFrame({
    "cluster": merged_clusters,
    "PC1_global": pc1_global
})

global_cluster_order = np.sort(global_df["cluster"].unique())

# 4) Plot global PC1 difference by integrated cluster
fig2, ax2 = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=global_df,
    x="cluster",
    y="PC1_global",
    order=global_cluster_order,
    palette=sns.color_palette(n_colors=len(global_cluster_order)),
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    ax=ax2
)

sns.stripplot(
    data=global_df,
    x="cluster",
    y="PC1_global",
    order=global_cluster_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.30,
    ax=ax2
)

# Median markers
med = global_df.groupby("cluster")["PC1_global"].median()
xpos = np.arange(len(global_cluster_order))
ax2.scatter(
    xpos,
    [med.loc[c] for c in global_cluster_order],
    s=180,
    marker="_",
    linewidths=3
)

# n labels
counts = global_df["cluster"].value_counts()
ymin, ymax = ax2.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
#for i, c in enumerate(global_cluster_order):
#    ax2.text(i, y_annot, f"n={int(counts.loc[c])}", ha="center", va="top", fontsize=16)

ax2.set_xlabel("Integrated cluster", fontsize=20)
ax2.set_ylabel("Global PC1 score (all modalities/features)", fontsize=20)
ax2.set_title(f"Global PC1 Across All Features and Modalities", fontsize=20, pad=14)
ax2.grid(axis="y", alpha=0.15)
ax2.set_xticklabels(ax2.get_xticklabels(), fontsize=16)
ax2.set_yticklabels(ax2.get_yticklabels(), fontsize=16)
sns.despine(ax=ax2)

fig2.tight_layout()
clinical_analysis_save_figure_png_pdf(fig2, os.path.join(out_dir, "GLOBAL_all_modalities_PC1_violin.png"), dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import importlib
import Utils
Utils = importlib.reload(Utils)
from Utils import modality_cluster_palette

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# ==========================================================
# 1) SINGLE PLOT: PC1 (per-modality PCA) distributions by modality
#    - CHR integrated clusters + CC group
# 2) GLOBAL PCA across ALL modalities/features
#    - CHR integrated clusters + CC group
# ==========================================================

out_dir = os.path.join(plots_dir, "merged_feature_pca_chr_vs_cc/")
os.makedirs(out_dir, exist_ok=True)

# --------------------------
# A) Per-modality PC1 plot: CHR clusters + CC
# --------------------------


rows = []

for modality, df_chr in dict_final.items():
    # CHR
    X_chr_df = df_chr.drop(columns=["src_subject_id"]).copy()
    grp_chr_raw = df_chr["src_subject_id"].astype(str).map(test_final_labels_by_subject_id)
    if np.any(pd.isna(grp_chr_raw)):
        missing = df_chr.loc[pd.isna(grp_chr_raw), "src_subject_id"].head(5).tolist()
        raise ValueError(f"{modality}: missing test integrated labels for CHR rows. Examples: {missing}")
    grp_chr = grp_chr_raw.astype(str).to_numpy()

    # CC (already transformed via apply_preprocessing_to_new_data)
    df_cc = dict_final_cc_test[modality]
    X_cc_df = df_cc.drop(columns=["src_subject_id"]).copy()
    X_cc_df = X_cc_df.reindex(columns=X_chr_df.columns)  # strict CHR feature order

    # Fit scaler+PCA on CHR only
    scaler = StandardScaler()
    X_chr_z = scaler.fit_transform(X_chr_df.values)
    X_cc_z = scaler.transform(X_cc_df.values)

    pca = PCA(n_components=1, random_state=0).fit(X_chr_z)
    pc1_chr = pca.transform(X_chr_z)[:, 0]
    pc1_cc = pca.transform(X_cc_z)[:, 0]
    evr1 = float(pca.explained_variance_ratio_[0])

    print(f"{modality}: CHR-fitted PC1 EVR={evr1:.2%}")

    tmp_chr = pd.DataFrame({
        "modality": modality,
        "group": grp_chr,
        "PC1": pc1_chr,
        "cohort": "CHR"
    })
    tmp_cc = pd.DataFrame({
        "modality": modality,
        "group": "CC",
        "PC1": pc1_cc,
        "cohort": "CC"
    })
    tmp = pd.concat([tmp_chr, tmp_cc], ignore_index=True)
    tmp["PC1_EVR_chrfit"] = evr1
    rows.append(tmp)

plot_df = pd.concat(rows, ignore_index=True)

modality_order = list(final_metrics["data"].keys())
chr_labels_all = pd.Series(plot_df.loc[plot_df["cohort"] == "CHR", "group"].astype(str).unique())
chr_cluster_order = sorted(chr_labels_all, key=lambda x: int(x) if str(x).isdigit() else str(x))
group_order = chr_cluster_order + ["CC"]

# Shared palette (same helper used for the per-modality CHR-vs-CC plots above)
group_palette = modality_cluster_palette(group_order)

fig_w = max(18, 2.2 * len(modality_order))
fig_h = 6
fig, ax = plt.subplots(figsize=(fig_w, fig_h))

sns.violinplot(
    data=plot_df,
    x="modality",
    y="PC1",
    hue="group",
    order=modality_order,
    hue_order=group_order,
    palette=group_palette,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    dodge=True,
    width=0.8,
    ax=ax
)

ax.set_xlim(-0.55, len(modality_order) - 0.45)

# --- manual black points centered inside each dodged violin ---
rng = np.random.default_rng(0)

n_hue = len(group_order)
violin_width = 0.8                      # must match sns.violinplot(width=0.65)
sub_width = violin_width / n_hue         # width allotted to each group within a modality
jitter_scale = sub_width * 0.28          # small jitter within each subgroup

for i, modality in enumerate(modality_order):
    for j, group in enumerate(group_order):
        vals = plot_df.loc[
            (plot_df["modality"] == modality) & (plot_df["group"] == group),
            "PC1"
        ].to_numpy()

        if len(vals) == 0:
            continue

        # exact center of this group's violin within this modality
        center = i - violin_width / 2 + (j + 0.5) * sub_width

        # small symmetric jitter around that center
        x = center + rng.uniform(-jitter_scale, jitter_scale, size=len(vals))

        ax.scatter(
            x,
            vals,
            color="black",
            s=10,
            alpha=0.22,
            zorder=3,
            linewidths=0
        )

# Legend cleanup (keep one)
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles[:len(group_order)],
    labels[:len(group_order)],
    title="Group",
    frameon=False,
    loc="upper right"
)

for x in np.arange(0.5, len(modality_order), 1.0):
    ax.axvline(x, linewidth=0.8, alpha=0.25)

ax.set_xlabel("Modality", fontsize=16)
ax.set_ylabel("PC1 score (CHR-fitted PCA)", fontsize=16)
ax.set_title("PC1 Distributions by Modality (CHR integrated clusters + CC)", fontsize=18, pad=14)
ax.tick_params(axis="both", labelsize=14)
ax.grid(axis="y", alpha=0.15)
sns.despine(ax=ax)

plt.setp(ax.get_xticklabels(), rotation=25, ha="right", fontsize=13)
plt.setp(ax.get_yticklabels(), fontsize=13)

fig.tight_layout()
clinical_analysis_save_figure_png_pdf(fig, os.path.join(out_dir, "test_ALL_modalities_PC1_singleplot_violin_CHR_vs_CC.png"), dpi=300, bbox_inches="tight")
plt.show()

# --------------------------
# B) GLOBAL PCA PC1 across ALL modalities/features: CHR clusters + CC
# --------------------------

# CHR merged wide
dfs_chr = []
for modality, df in dict_final.items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})
    dfs_chr.append(tmp)

merged_chr = dfs_chr[0]
for d in dfs_chr[1:]:
    merged_chr = merged_chr.merge(d, on="src_subject_id", how="inner")

print(f"\n[GLOBAL PCA] CHR subjects after inner-join: {merged_chr.shape[0]}")
print(f"[GLOBAL PCA] CHR merged features: {merged_chr.shape[1] - 1}")

# CC merged wide
dfs_cc = []
for modality, df in dict_final_cc_test.items():
    tmp = df.copy()
    feat_cols = [c for c in tmp.columns if c != "src_subject_id"]
    tmp = tmp.rename(columns={c: f"{modality}__{c}" for c in feat_cols})
    dfs_cc.append(tmp)

merged_cc = dfs_cc[0]
for d in dfs_cc[1:]:
    merged_cc = merged_cc.merge(d, on="src_subject_id", how="inner")

print(f"[GLOBAL PCA] CC subjects after inner-join: {merged_cc.shape[0]}")
print(f"[GLOBAL PCA] CC merged features: {merged_cc.shape[1] - 1}")

# Align CHR test labels to merged CHR IDs
chr_clusters_raw = merged_chr["src_subject_id"].astype(str).map(test_final_labels_by_subject_id)
if np.any(pd.isna(chr_clusters_raw)):
    missing = merged_chr.loc[pd.isna(chr_clusters_raw), "src_subject_id"].head(5).tolist()
    raise ValueError(f"Missing test integrated labels for merged CHR IDs. Examples: {missing}")
chr_clusters = chr_clusters_raw.astype(str).to_numpy()

# Strict CC feature alignment to CHR merged features
chr_feature_cols = [c for c in merged_chr.columns if c != "src_subject_id"]
cc_feature_cols = [c for c in merged_cc.columns if c != "src_subject_id"]
missing_in_cc = [c for c in chr_feature_cols if c not in cc_feature_cols]
if missing_in_cc:
    raise ValueError(f"CC missing {len(missing_in_cc)} global features. Example: {missing_in_cc[:10]}")

X_chr_global = merged_chr[chr_feature_cols].values
X_cc_global = merged_cc.reindex(columns=chr_feature_cols).values

# CHR-fitted global PCA
Xg_scaler = StandardScaler()
Xg_chr_z = Xg_scaler.fit_transform(X_chr_global)
Xg_cc_z = Xg_scaler.transform(X_cc_global)

pca_global = PCA(n_components=1, random_state=0).fit(Xg_chr_z)
pc1_chr_global = pca_global.transform(Xg_chr_z)[:, 0]
pc1_cc_global = pca_global.transform(Xg_cc_z)[:, 0]
evr1_global = float(pca_global.explained_variance_ratio_[0])
print(f"[GLOBAL PCA] CHR-fitted PC1 EVR: {evr1_global:.2%}")

global_df = pd.concat([
    pd.DataFrame({"group": chr_clusters, "PC1_global": pc1_chr_global, "cohort": "CHR"}),
    pd.DataFrame({"group": "CC", "PC1_global": pc1_cc_global, "cohort": "CC"})
], ignore_index=True)

global_group_order = chr_cluster_order + ["CC"]

fig2, ax2 = plt.subplots(figsize=(10.5, 6.5))

sns.violinplot(
    data=global_df,
    x="group",
    y="PC1_global",
    hue="group",
    order=global_group_order,
    hue_order=global_group_order,
    palette=group_palette,
    dodge=False,
    inner="quartile",
    cut=0,
    bw_adjust=1.0,
    linewidth=1,
    legend=False,
    ax=ax2
)

sns.stripplot(
    data=global_df,
    x="group",
    y="PC1_global",
    order=global_group_order,
    color="black",
    size=3,
    jitter=0.22,
    alpha=0.30,
    ax=ax2
)

med = global_df.groupby("group")["PC1_global"].median()
xpos = np.arange(len(global_group_order))
ax2.scatter(xpos, [med.loc[g] for g in global_group_order], s=180, marker="_", linewidths=3, color="black", zorder=4)

counts = global_df["group"].value_counts()
ymin, ymax = ax2.get_ylim()
y_annot = ymin + 0.04 * (ymax - ymin)
for i, g in enumerate(global_group_order):
    ax2.text(i, y_annot, f"n={int(counts.get(g, 0))}", ha="center", va="top", fontsize=16)

ax2.set_xlabel("Group", fontsize=20)
ax2.set_ylabel("Global PC1 score (CHR-fitted)", fontsize=20)
ax2.set_title("Global PC1 Across All Features and Modalities (CHR clusters + CC)", fontsize=20, pad=14)
ax2.grid(axis="y", alpha=0.15)
ax2.tick_params(axis="x", labelsize=16)
ax2.tick_params(axis="y", labelsize=16)
sns.despine(ax=ax2)

fig2.tight_layout()
clinical_analysis_save_figure_png_pdf(fig2, os.path.join(out_dir, "test_GLOBAL_all_modalities_PC1_violin_CHR_vs_CC.png"), dpi=300, bbox_inches="tight")
plt.show()


### Differences in categorical variables - individual labels

In [ ]:
# Use the shared add metadata and clusters helper.
from Utils import clinical_analysis_add_metadata_and_clusters_validation_individual_labels

# Use the shared chi square comparison helper.
from Utils import clinical_analysis_chi_square_comparison_validation_individual_labels


mod_num = 0
for modality in dict_final.keys():
    print(f"\n=== Analyzing categorical differences for modality: {modality} ===")

    # Merge cluster labels into full data
    df = clinical_analysis_add_metadata_and_clusters_validation_individual_labels(dict_final[modality], test_data, mod_num)

    # Compare by phenotype (CHR vs CC)
    if 'phenotype' in df.columns:
        clinical_analysis_chi_square_comparison_validation_individual_labels(
            df=df,
            group_col='Cluster',
            label_col='phenotype',
            title_prefix=f"Comparison of CHR vs CC per Subgroup",
        )

    # Compare by site
    if 'Site' in df.columns:
        clinical_analysis_chi_square_comparison_validation_individual_labels(
            df=df,
            group_col='Cluster',
            label_col='Site',
            title_prefix=f"Comparison of Site Distribution per Subgroup",
        )

    # Optional: extend for other categorical variables
    for col in ['sips_bips_scr_lifetime', 'sips_aps_scr_lifetime', 'sips_grd_scr_lifetime']:
        if col in df.columns:
            clinical_analysis_chi_square_comparison_validation_individual_labels(
                df=df,
                group_col='Cluster',
                label_col=col,
                title_prefix=f"Comparison of {col} per Subgroup",
            )
    mod_num = mod_num + 1




### Mapping modalities -> final

In [ ]:
import numpy as np
import pandas as pd

new_test_labels_by_modality = {}

for i, modality in enumerate(modality_names):
    df = dict_final[modality]
    labels = labels_test_modalities[i]

    # Validate alignment
    if len(labels) != len(df):
        raise ValueError(
            f"Length mismatch for {modality}: labels={len(labels)} vs df={len(df)}"
        )

    labels_s = pd.Series(labels, index=df.index)

    uniq = labels_s.dropna().unique()
    if len(uniq) < 2:
        continue

    # Features only (avoid double mean of empty/invalid frames)
    X = df.drop(columns=['src_subject_id'], errors='ignore')
    if X.shape[1] == 0:
        raise ValueError(f"{modality}: no feature columns after dropping src_subject_id")

    # Cluster-specific means
    cluster_means = labels_s.groupby(labels_s).apply(
        lambda s: X.loc[s.index].to_numpy().mean()
    )

    # Choose the "highest mean" cluster as the high_cluster
    high_cluster = cluster_means.sort_values(ascending=False).index[0]

    # Map to severity labels (your modality-specific inversion kept)
    if modality in ("Functioning", "Cognition"):
        # higher score means *lower* severity (per your rule)
        new_labels = np.where(labels_s == high_cluster, "low_severity", "high_severity")
    else:
        new_labels = np.where(labels_s == high_cluster, "high_severity", "low_severity")

    new_test_labels_by_modality[modality] = new_labels.tolist()


In [ ]:
domain_map(
    new_labels_by_modality=new_test_labels_by_modality,
    final_labels=labels_test_final,
    stage_order=["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"],
    final_name="Integrated",
    top_token="high_severity",          # now HIGH is on top
    bottom_token="low_severity",
    invert_final=False,
    color_for_top_final="#38699A",      # top-like final ribbons
    color_for_bottom_final="#B36F9C",   # bottom-like final ribbons
    add_gap_in_final=True,
    gap_weight=20,
    plots_dir=plots_dir,
    save_file_name = "Parcats_by_final_test.pdf"
)

## Discovery-validation domain-map comparison

Compare the observed domain-map paths in the discovery sample with the predicted validation sample.


In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_DOMAIN_MAP_COMPARISON_DIR = os.path.join(plots_dir, "domain_map_discovery_validation_comparison")
os.makedirs(_DOMAIN_MAP_COMPARISON_DIR, exist_ok=True)


discovery_domain_map_frame, discovery_domain_map_paths, domain_map_comparison_stages = clinical_analysis_domain_map_path_table(
    labels_by_modality=new_labels_by_modality,
    final_labels=final_metrics["final_labels"],
    sample_name="discovery",
    stage_order_for_labels=stage_order,
    final_name="Integrated",
)
validation_domain_map_frame, validation_domain_map_paths, validation_domain_map_comparison_stages = clinical_analysis_domain_map_path_table(
    labels_by_modality=new_test_labels_by_modality,
    final_labels=labels_test_final,
    sample_name="validation",
    stage_order_for_labels=stage_order,
    final_name="Integrated",
)
if domain_map_comparison_stages != validation_domain_map_comparison_stages:
    raise ValueError(
        "Discovery and validation domain-map stages differ: "
        f"{domain_map_comparison_stages} vs {validation_domain_map_comparison_stages}"
    )

all_domain_map_paths = pd.concat(
    [discovery_domain_map_paths, validation_domain_map_paths],
    ignore_index=True,
)
all_domain_map_paths.to_csv(
    os.path.join(_DOMAIN_MAP_COMPARISON_DIR, "discovery_validation_domain_map_path_counts_long.csv"),
    index=False,
)

integrated_path_comparison = clinical_analysis_compare_domain_map_paths(
    discovery_domain_map_paths,
    validation_domain_map_paths,
    group_cols=domain_map_comparison_stages + ["Integrated", "integrated_path"],
    label="integrated",
)
domain_path_comparison = clinical_analysis_compare_domain_map_paths(
    discovery_domain_map_paths,
    validation_domain_map_paths,
    group_cols=domain_map_comparison_stages + ["domain_path"],
    label="domain_only",
)

path_overlap_summary = pd.DataFrame([
    {
        "comparison": "domain_only",
        "n_paths_discovery": int(domain_path_comparison["n_discovery"].gt(0).sum()),
        "n_paths_validation": int(domain_path_comparison["n_validation"].gt(0).sum()),
        "n_paths_shared": int(domain_path_comparison["path_present_in"].eq("both").sum()),
        "n_paths_discovery_only": int(domain_path_comparison["path_present_in"].eq("discovery_only").sum()),
        "n_paths_validation_only": int(domain_path_comparison["path_present_in"].eq("validation_only").sum()),
    },
    {
        "comparison": "with_integrated",
        "n_paths_discovery": int(integrated_path_comparison["n_discovery"].gt(0).sum()),
        "n_paths_validation": int(integrated_path_comparison["n_validation"].gt(0).sum()),
        "n_paths_shared": int(integrated_path_comparison["path_present_in"].eq("both").sum()),
        "n_paths_discovery_only": int(integrated_path_comparison["path_present_in"].eq("discovery_only").sum()),
        "n_paths_validation_only": int(integrated_path_comparison["path_present_in"].eq("validation_only").sum()),
    },
])
path_overlap_summary.to_csv(
    os.path.join(_DOMAIN_MAP_COMPARISON_DIR, "domain_map_path_overlap_summary.csv"),
    index=False,
)

print("Domain-map path overlap summary")
display(path_overlap_summary)

print("Largest discovery-validation differences for full paths including Integrated")
display(integrated_path_comparison.head(30))

print("Largest discovery-validation differences for domain-only paths")
display(domain_path_comparison.head(30))

fig_integrated_path_comparison = clinical_analysis_plot_domain_map_path_comparison(
    integrated_path_comparison,
    path_col="integrated_path",
    title="Discovery vs validation domain-map paths including Integrated label",
    filename_prefix="discovery_validation_integrated_domain_map_path_comparison",
    top_n=25,
)
fig_domain_path_comparison = clinical_analysis_plot_domain_map_path_comparison(
    domain_path_comparison,
    path_col="domain_path",
    title="Discovery vs validation domain-only map paths",
    filename_prefix="discovery_validation_domain_only_map_path_comparison",
    top_n=25,
)



In [ ]:
import pandas as pd

stage_order = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition"]

# Pick subject_id vector from one modality (must share row order with labels)
subject_ids_test = dict_final[stage_order[0]]['src_subject_id'].astype(str).reset_index(drop=True)

# Build the path dataframe
df_paths_test = pd.DataFrame({"src_subject_id": subject_ids_test})
for stage in stage_order:
    df_paths_test[stage] = pd.Series(new_test_labels_by_modality[stage]).astype(str).reset_index(drop=True)

df_paths_test["final"] = pd.Series(labels_test_final).astype(str).reset_index(drop=True)

# Sanity checks
N = len(df_paths_test)
assert all(len(new_test_labels_by_modality[s]) == N for s in stage_order), "Label lengths don't match subject_ids length"
assert len(labels_test_final) == N, "Final labels length doesn't match subject_ids length"

df_paths_test


In [ ]:
# Use the shared canonical stream format: Domain=label → ... → final=label.
# This must match the discovery-side stream labels before comparing presence or mappings.
stream_summary_test_result = clinical_analysis_summarize_streams(df_paths_test, stage_order, top_k=100, sample_ids=12)
stream_summary_test = (
    stream_summary_test_result[0]
    if isinstance(stream_summary_test_result, tuple)
    else stream_summary_test_result
)

if not isinstance(stream_summary_test, pd.DataFrame):
    raise TypeError(
        "clinical_analysis_summarize_streams must return a DataFrame or a tuple whose first item is a DataFrame; "
        f"got {type(stream_summary_test).__name__}."
    )

stream_summary_test


### Comparison between discovery and test in cluster mapping

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial.distance import jensenshannon

# Optional (for sankey). If plotly isn't installed, sankey functions will gracefully fail.
try:
    import plotly.graph_objects as go
    _HAS_PLOTLY = True
except Exception:
    _HAS_PLOTLY = False


ARROW_PAT = re.compile(r"\s*(?:→|->)\s*")

# Use the shared parse stream helper.
from Utils import clinical_analysis_parse_stream


# Use the shared infer stage order helper.
from Utils import clinical_analysis_infer_stage_order


# Use the shared clinical_analysis_normalize helper.
from Utils import _normalize


# ------------------------------------------------------------
# 1) Prefix-tree comparison: compare P(next | prefix)
# ------------------------------------------------------------

# Use the shared build prefix next helper.
from Utils import clinical_analysis_build_prefix_next


# Use the shared compare prefix structure helper.
from Utils import clinical_analysis_compare_prefix_structure


# Use the shared plot top prefix differences helper.
from Utils import clinical_analysis_plot_top_prefix_differences


# ------------------------------------------------------------
# 2) Full mapping to final: compare P(final | modalities-prefix)
# ------------------------------------------------------------

# Use the shared final mapping table helper.
from Utils import clinical_analysis_final_mapping_table


# Use the shared compare final mapping helper.
from Utils import clinical_analysis_compare_final_mapping


# Use the shared plot top final mapping shifts helper.
from Utils import clinical_analysis_plot_top_final_mapping_shifts


# ------------------------------------------------------------
# 3) Full stream presence + rank overlap
# ------------------------------------------------------------

# Use the shared stream presence and topk helper.
from Utils import clinical_analysis_stream_presence_and_topk


# ------------------------------------------------------------
# 4) Sankey (full structure) per dataset
# ------------------------------------------------------------

# Use the shared sankey from streams helper.
from Utils import clinical_analysis_sankey_from_streams


# ------------------------------------------------------------
# Master runner
# ------------------------------------------------------------

# Use the shared full structure report helper.
from Utils import clinical_analysis_full_structure_report


# -----------------------------
# Example usage
# -----------------------------
# rep = clinical_analysis_full_structure_report(stream_summary, stream_summary_test, topk=30, final_domain="final")
# rep["presence_metrics"]
# rep["prefix_report"].head(30)  # structural differences by prefix
# clinical_analysis_plot_top_prefix_differences(rep["prefix_report"], top_n=20, min_depth=1)
#
# rep["final_mapping_metrics"]
# rep["final_mapping_compare"].head(30)
# clinical_analysis_plot_top_final_mapping_shifts(rep["final_mapping_compare"], top_n=20)
#
# # Sankey (if plotly installed)
# if _HAS_PLOTLY:
#     figD = clinical_analysis_sankey_from_streams(stream_summary, max_edges=250)
#     figD.update_layout(title_text="Discovery stream structure (Sankey)")
#     figD.show()
#     figT = clinical_analysis_sankey_from_streams(stream_summary_test, max_edges=250)
#     figT.update_layout(title_text="Test stream structure (Sankey)")
#     figT.show()


In [ ]:

# Rebuild both sides here so the comparison cannot reuse stale stream labels
# from an older imported Utils.clinical_analysis_summarize_streams implementation.
stream_summary = clinical_analysis_summarize_streams_for_comparison(df_paths, stage_order, top_k=100, sample_ids=12)
stream_summary_test = clinical_analysis_summarize_streams_for_comparison(df_paths_test, stage_order, top_k=100, sample_ids=12)

rep = clinical_analysis_full_structure_report(stream_summary, stream_summary_test, topk=100, final_domain="final")

rep["presence_metrics"]


In [ ]:
# Where does the pathway grammar differ most?
rep["prefix_report"].head(25)
clinical_analysis_plot_top_prefix_differences(rep["prefix_report"], top_n=25, min_depth=1)


In [ ]:
# Does the multimodal signature map to the same final label?
rep["final_mapping_metrics"]
rep["final_mapping_compare"].head(25)
clinical_analysis_plot_top_final_mapping_shifts(rep["final_mapping_compare"], top_n=25)


In [ ]:
import pandas as pd
import numpy as np

# Use the shared all streams table helper.
from Utils import clinical_analysis_all_streams_table

# Usage
streams_tbl = clinical_analysis_all_streams_table(stream_summary, stream_summary_test)
streams_tbl.head(100)


# Post-Pipeline Paper 1 Analyses

These cells bring the post-pipeline analyses from `PrepareData_demtable_paper1.Rmd` into the clinical notebook: demographic/sample tables, subgroup difference summaries, mixed heatmaps, site/recruitment checks, and included-vs-excluded diagnostics. Outputs are written under `plots_dir/post_pipeline_paper1` so they stay with the pipeline run.


In [ ]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

POST_ANALYSIS_DIR = Path(plots_dir) / "post_pipeline_paper1"
POST_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

LABELS_DIR = Path("path/to/results/study_1/release/Labels")
DISCOVERY_LABELS_FILE = LABELS_DIR / "discovery_labels_clin_multiclust.csv"
TEST_LABELS_FILE = LABELS_DIR / "test_labels_clin_multiclust.csv"
LABELS_DIR.mkdir(parents=True, exist_ok=True)

DICTIONARY_DIR_DIFF = "path/to/project/Feature selection/Complete_dictionary_differences.xlsx"
SUBGROUP_VARS = ["Psychoticism", "Detachment", "Internalising", "Functioning", "Cognition", "final"]
SUBGROUP_DISPLAY_LABELS = {"final": "Integrated"}


discovery_labels = clinical_analysis_load_saved_label_frame(DISCOVERY_LABELS_FILE, "discovery")
test_labels = clinical_analysis_load_saved_label_frame(TEST_LABELS_FILE, "test")

data_for_post = clinical_analysis_resolve_post_analysis_data()
data_for_post["src_subject_id"] = data_for_post["src_subject_id"].astype(str)

# Match the Rmd's `data_finalSample = merged_data`, then sample assignment and filter.
data_final_sample = data_for_post.copy()
data_final_sample["sample"] = np.where(
    data_final_sample["src_subject_id"].isin(discovery_labels["src_subject_id"]),
    "Discovery",
    "Test",
)
data_final_sample = data_final_sample.loc[
    data_final_sample["src_subject_id"].isin(discovery_labels["src_subject_id"]) |
    data_final_sample["src_subject_id"].isin(test_labels["src_subject_id"])
].copy()

# Match the Rmd's inner merge: merge(discovery_labels, merged_data, by='src_subject_id').
Labels_data_disc = discovery_labels.merge(data_for_post, on="src_subject_id", how="inner")
Labels_data_test = test_labels.merge(data_for_post, on="src_subject_id", how="inner")

print("Discovery labels:", discovery_labels.shape, "merged:", Labels_data_disc.shape)
print("Test labels:", test_labels.shape, "merged:", Labels_data_test.shape)
print("Final sample:", data_final_sample.shape)
print("Post-analysis output directory:", POST_ANALYSIS_DIR)


## Shared Helpers

The Rmd used a custom R `demographic_table()` function. The helpers below reproduce the same analysis intent in Python: compare variables across grouping columns, report effect sizes, apply BH/FDR correction, and export notebook-friendly CSV tables plus figure files.


In [ ]:
try:
    from scipy.stats import chi2_contingency as _scipy_chi2_contingency
    from scipy.stats import kruskal as _scipy_kruskal
    _HAS_SCIPY_STATS = True
except Exception:
    _HAS_SCIPY_STATS = False
    _scipy_chi2_contingency = None
    _scipy_kruskal = None

import Utils
Utils = importlib.reload(Utils)
from Utils import *


VARIABLE_LABELS, DICTIONARY_VARIABLE_ORDER = clinical_analysis_load_dictionary_labels()
HEATMAP_EXCLUDED_VARIABLES = {"interview_date", "sex", "chrpsychs_fu_ac1", "chrpsychs_scr_ac3", "chrpsychs_scr_ac4", "chrpsychs_scr_ac6", "chrpsychs_scr_ac7", "chrpsychs_scr_ac2", "chrpsychs_scr_ac5"}
_METADATA_VARIABLE_TABLE_CACHE = None
_DICTIONARY_ALIAS_TABLE_CACHE = {}
_VARIABLE_CANONICAL_LOOKUP_CACHE = None




## Label Difference Summaries


In [ ]:
# Rmd equivalent: subgroup demographic/difference summaries for discovery and test samples.
discovery_difference_summary = clinical_analysis_summarize_group_differences(
    Labels_data_disc,
    comparisons=SUBGROUP_VARS,
    output_prefix="discovery_labels",
)

test_difference_summary = clinical_analysis_summarize_group_differences(
    Labels_data_test,
    comparisons=SUBGROUP_VARS,
    output_prefix="test_labels",
)

print("Discovery difference rows:", len(discovery_difference_summary))
print("Test difference rows:", len(test_difference_summary))


## Mixed Heatmaps of Subgroup Differences


In [ ]:
from Utils import display_feature_name


discovery_heatmap_effect_sizes = clinical_analysis_run_effect_size_domain_heatmap(
    Labels_data_disc,
    discovery_difference_summary,
    sample_label="Discovery",
    filename_prefix="heatmap_group",
)

validation_heatmap_effect_sizes = clinical_analysis_run_effect_size_domain_heatmap(
    Labels_data_test,
    test_difference_summary,
    sample_label="Validation",
    filename_prefix="validation_heatmap_group",
)


## Site and Recruitment Source


In [ ]:

site_recruitment_table = clinical_analysis_site_recruitment_overview_rmd_style(data_for_post)


In [ ]:



site_subgroup_results, site_subgroup_long = clinical_analysis_test_subgroup_by_factor(
    Labels_data_disc,
    factor_col="Site",
    prefix="discovery_site",
    factor_label="site",
)

recruitment_subgroup_results, recruitment_subgroup_long = clinical_analysis_test_subgroup_by_factor(
    Labels_data_disc,
    factor_col="chrrecruit",
    prefix="discovery_recruitment",
    factor_label="recruitment source",
)


## Site Plus Recruitment Checks

The Rmd fits multinomial subgroup models adjusted for site and recruitment source. This notebook keeps that analysis optional: it runs if `statsmodels` is available in the active kernel and otherwise still exports the stratified site-by-subgroup tests within recruitment strata.


In [ ]:



stratified_site_recruitment_tests = clinical_analysis_stratified_site_tests_within_recruitment(Labels_data_disc)
adjusted_site_recruitment_models = clinical_analysis_optional_multinomial_site_recruitment_models(Labels_data_disc)


## Included Versus Excluded CHR Participants


In [ ]:
CHR_data = data_for_post.loc[data_for_post["phenotype"].astype(str).eq("CHR")].copy()
included_ids = set(Labels_data_disc["src_subject_id"].astype(str)) | set(Labels_data_test["src_subject_id"].astype(str))
CHR_data["included"] = np.where(CHR_data["src_subject_id"].astype(str).isin(included_ids), "included", "excluded")

included_vs_excluded_table = clinical_analysis_demographic_summary_table(
    CHR_data,
    comparison="included",
    output_name="dem_table_included_vs_excluded_chr.csv",
)
display(included_vs_excluded_table.head(30))

included_vs_excluded_differences = clinical_analysis_summarize_group_differences(
    CHR_data,
    comparisons=["included"],
    output_prefix="included_vs_excluded_chr",
)

print("Included/excluded CHR counts:")
display(CHR_data["included"].value_counts())


# Longitudinal analyses across months 1-5

It fits baseline-cluster mixed models across all available months and maps follow-up observations back onto baseline cluster centroids for discovery and validation samples.


## Main analyses

In [ ]:
import os
import pandas as pd
import importlib
import Utils
Utils = importlib.reload(Utils)
run_longitudinal_multiclust_report = Utils.run_longitudinal_multiclust_report
display_longitudinal_multiclust_results = Utils.display_longitudinal_multiclust_results
load_longitudinal_multiclust_results = Utils.load_longitudinal_multiclust_results

_is_singleclust_result = isinstance(final_metrics.get("data"), pd.DataFrame) if isinstance(final_metrics, dict) else False
_longitudinal_data_dirs = [
    'path/to/simpleclust_data'
    if _is_singleclust_result
    else 'path/to/multiclust_data'
]
_subject_id_column = subject_id_column if "subject_id_column" in globals() else "src_subject_id"

_longitudinal_prescient_ids = (
    prescient_ids
    if "prescient_ids" in globals()
    else (set(prescient["src_subject_id"]) if "prescient" in globals() else None)
)

_validation_baseline_data = None
if "dict_final_test" in globals() and isinstance(dict_final_test, dict) and dict_final_test:
    _validation_baseline_data = dict_final_test
elif "dict_final" in globals() and isinstance(dict_final, dict) and dict_final and "labels_test_final" in globals():
    _validation_baseline_data = dict_final

_validation_subject_ids = None
if isinstance(_validation_baseline_data, dict) and _validation_baseline_data:
    _validation_subject_ids = {
        modality: df[_subject_id_column].astype(str).tolist()
        for modality, df in _validation_baseline_data.items()
        if _subject_id_column in df.columns
    }
elif "subject_id_list_test" in globals() and "modalities" in globals():
    _validation_subject_ids = {
        modality: ids
        for modality, ids in zip(modalities, subject_id_list_test)
    }

_validation_domain_labels = None
if "new_test_labels_by_modality" in globals() and new_test_labels_by_modality:
    _validation_domain_labels = new_test_labels_by_modality
elif "labels_test_by_modality" in globals() and labels_test_by_modality:
    _validation_domain_labels = {k: v for k, v in labels_test_by_modality.items() if v is not None}
elif "labels_test_modalities" in globals() and "modalities" in globals():
    _validation_domain_labels = {
        modality: labels
        for modality, labels in zip(modalities, labels_test_modalities)
        if labels is not None
    }

_validation_final_labels = labels_test_final if "labels_test_final" in globals() else None

longitudinal_results = run_longitudinal_multiclust_report(
    final_metrics=final_metrics,
    meta=meta,
    plots_dir=plots_dir,
    data_dirs=_longitudinal_data_dirs,
    prescient_ids=_longitudinal_prescient_ids,
    vars_to_keep=vars_to_keep if "vars_to_keep" in globals() else None,
    categorical_columns=cat_vars if "cat_vars" in globals() else None,
    validation_domain_labels=_validation_domain_labels,
    validation_final_labels=_validation_final_labels,
    validation_subject_ids=_validation_subject_ids,
    validation_baseline_data=_validation_baseline_data,
    months=(1, 2, 3, 4, 5),
    subject_id_column=_subject_id_column,
    min_features_per_analysis=2,
    min_features_for_cluster_change=1,
    min_followup_timepoints_per_feature=1,
    min_nonmissing_per_timepoint=8,
    followup_col_threshold=0.5,
    followup_row_threshold=0.5,
    min_group_n=4,
    reuse_existing=True,
)

print("Longitudinal outputs saved under:", longitudinal_results["output_dir"])
print("Month files used:")
for month, path in longitudinal_results["month_files"].items():
    print(f"  month {month}: {path}")
print("Analyses completed:", len(longitudinal_results["analyses"]))
if longitudinal_results["analysis_summary"].empty:
    print("No mixed-model or cluster-change analyses were completed. Check longitudinal_results['preprocessing_report'] for preprocessing failures.")
else:
    display(longitudinal_results["analysis_summary"])


# Sankey image embedding can time out on cloud-synced output files; saved files remain available in the output directory.
display_longitudinal_multiclust_results(longitudinal_results, max_analyses=20, show_sankey=False)


In [ ]:
_preprocessing_report = longitudinal_results["preprocessing_report"]
_sort_cols = [
    col for col in ["sample", "month", "modality"]
    if col in _preprocessing_report.columns
]
_preprocessing_report_view = (
    _preprocessing_report.sort_values(_sort_cols)
    if _sort_cols else _preprocessing_report
)
display(_preprocessing_report_view.head(30))

if "status" in _preprocessing_report.columns:
    display(
        _preprocessing_report
        .groupby([c for c in ["sample", "status"] if c in _preprocessing_report.columns])
        .size()
        .reset_index(name="n_rows")
    )


In [ ]:
# Display saved longitudinal results without rerunning the analyses.
import importlib
import Utils
importlib.reload(Utils)
from Utils import load_longitudinal_multiclust_results, display_longitudinal_multiclust_results

_existing_longitudinal_results = load_longitudinal_multiclust_results(
    os.path.join(plots_dir, "longitudinal_all_timepoints")
)
display_longitudinal_multiclust_results(
    _existing_longitudinal_results,
    max_analyses=10,
    show_sankey=False,  # Avoid notebook timeouts while embedding saved Sankey/image files. 
)


## Longitudinal Domain-Map Path Groups

This tests longitudinal feature trajectories using each unique baseline path through the domain-map axes as the grouping variable. It runs two variants: domain paths only, and domain paths plus the integrated/final cluster. Outputs are written to `longitudinal_domain_map_path_groups`.



In [ ]:
import os
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2

import Utils
Utils = importlib.reload(Utils)

_DOMAIN_MAP_GROUPS_DIR = os.path.join(plots_dir, "longitudinal_domain_map_path_groups")
os.makedirs(_DOMAIN_MAP_GROUPS_DIR, exist_ok=True)

_subject_id_column = subject_id_column if "subject_id_column" in globals() else "src_subject_id"
_domain_stage_order = [stage for stage in stage_order if stage in new_labels_by_modality]


_source_longitudinal_results = clinical_analysis_get_domain_map_source_longitudinal_results()

discovery_subject_ids_domain_map = final_metrics["data"][_domain_stage_order[0]][_subject_id_column].reset_index(drop=True)

_validation_subject_ids_domain_map = None
if "new_test_labels_by_modality" in globals() and new_test_labels_by_modality:
    if "subject_ids_test" in globals():
        _validation_subject_ids_domain_map = pd.Series(subject_ids_test).reset_index(drop=True)
    elif isinstance(globals().get("_validation_subject_ids"), dict) and globals().get("_validation_subject_ids"):
        _validation_subject_ids_domain_map = pd.Series(next(iter(_validation_subject_ids.values()))).reset_index(drop=True)
    else:
        _validation_subject_ids_domain_map = pd.Series(range(len(next(iter(new_test_labels_by_modality.values())))))

_domain_map_group_variants = [
    {
        "variant": "domains_only",
        "label": "Domain paths only",
        "include_final_in_path": False,
    },
    {
        "variant": "domains_plus_integrated",
        "label": "Domain paths plus integrated cluster",
        "include_final_in_path": True,
    },
]

_domain_map_variant_label_frames = {}
for variant_config in _domain_map_group_variants:
    variant_name = variant_config["variant"]
    include_final = bool(variant_config["include_final_in_path"])
    variant_label_frames = {}

    discovery_labels_variant, discovery_counts_variant = clinical_analysis_domain_map_path_label_frame(
        labels_by_modality=new_labels_by_modality,
        subject_ids=discovery_subject_ids_domain_map,
        final_labels=final_metrics.get("final_labels"),
        sample_name=f"discovery_{variant_name}",
        stage_order_for_labels=_domain_stage_order,
        min_group_n=4,
        include_final_in_path=include_final,
    )
    variant_label_frames["discovery"] = discovery_labels_variant

    if _validation_subject_ids_domain_map is not None:
        validation_labels_variant, validation_counts_variant = clinical_analysis_domain_map_path_label_frame(
            labels_by_modality=new_test_labels_by_modality,
            subject_ids=_validation_subject_ids_domain_map,
            final_labels=labels_test_final if "labels_test_final" in globals() else None,
            sample_name=f"validation_{variant_name}",
            stage_order_for_labels=[stage for stage in _domain_stage_order if stage in new_test_labels_by_modality],
            min_group_n=4,
            include_final_in_path=include_final,
        )
        variant_label_frames["validation"] = validation_labels_variant

    _domain_map_variant_label_frames[variant_name] = variant_label_frames

_domain_map_summary_rows = []
_domain_map_mixedlm_summaries = {}
for variant_config in _domain_map_group_variants:
    variant_name = variant_config["variant"]
    variant_label = variant_config["label"]
    variant_label_frames = _domain_map_variant_label_frames.get(variant_name, {})

    for (sample_name, analysis_name), analysis_results in _source_longitudinal_results.get("analyses", {}).items():
        if sample_name not in variant_label_frames:
            continue
        mixed = analysis_results.get("mixedlm", {})
        long_df = mixed.get("long_df", pd.DataFrame())
        if not isinstance(long_df, pd.DataFrame) or long_df.empty:
            print(f"Skipping {variant_name}/{sample_name}/{analysis_name}: no long-format data available.")
            continue
        safe_analysis_name = str(analysis_name).replace(os.sep, "_")
        analysis_out = os.path.join(_DOMAIN_MAP_GROUPS_DIR, variant_name, sample_name, safe_analysis_name)
        analysis_output = clinical_analysis_run_domain_map_mixed_models_from_long_df(
            long_df=long_df,
            labels_df=variant_label_frames[sample_name],
            output_dir=analysis_out,
            analysis_name=f"{sample_name}_{safe_analysis_name}_{variant_name}",
            min_group_n=4,
            top_n_plot=12,
            max_plot_groups=None,
        )
        summary = analysis_output.get("summary", pd.DataFrame())
        _domain_map_mixedlm_summaries[(variant_name, sample_name, analysis_name)] = analysis_output
        _domain_map_summary_rows.append({
            "variant": variant_name,
            "variant_label": variant_label,
            "includes_integrated_in_path": bool(variant_config["include_final_in_path"]),
            "sample": sample_name,
            "analysis": analysis_name,
            "n_rows": int(len(summary)),
            "n_ok": int(summary["status"].eq("ok").sum()) if "status" in summary.columns else 0,
            "output_dir": analysis_out,
            "summary_path": os.path.join(analysis_out, f"{sample_name}_{safe_analysis_name}_{variant_name}_domain_map_mixedlm_summary.csv"),
            "top_features_plot": os.path.join(analysis_out, f"{sample_name}_{safe_analysis_name}_{variant_name}_domain_map_mixedlm_top_features.png"),
            "mean_drift_summary": analysis_output.get("mean_drift_summary_path", ""),
            "mean_drift_raw_plot": analysis_output.get("mean_drift_raw_plot_path", ""),
            "mean_drift_change_plot": analysis_output.get("mean_drift_change_plot_path", ""),
        })

longitudinal_domain_map_group_summary = pd.DataFrame(_domain_map_summary_rows)
longitudinal_domain_map_group_summary.to_csv(
    os.path.join(_DOMAIN_MAP_GROUPS_DIR, "longitudinal_domain_map_group_analysis_summary.csv"),
    index=False,
)
print("Domain-map longitudinal outputs saved under:", _DOMAIN_MAP_GROUPS_DIR)
display(longitudinal_domain_map_group_summary)




## Longitudinal grouping-scheme comparison

Compare the mixed-model outcome patterns across three grouping schemes: individual domain subgroups, integrated/final subgroups, and domain-map paths. This section reads saved mixed-model summary CSVs and does not rerun longitudinal models.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

_GROUPING_COMPARISON_DIR = os.path.join(_DOMAIN_MAP_GROUPS_DIR, "grouping_scheme_comparison")
os.makedirs(_GROUPING_COMPARISON_DIR, exist_ok=True)
_LONGITUDINAL_ALL_DIR = os.path.join(plots_dir, "longitudinal_all_timepoints")
_DOMAIN_MAP_PATH_DIR = os.path.join(_DOMAIN_MAP_GROUPS_DIR, "domains_only")
_LONGITUDINAL_ANALYSES = ["Internalising", "Functioning", "Detachment", "Psychoticism", "integrated"]


longitudinal_grouping_scheme_results = clinical_analysis_load_longitudinal_grouping_scheme_results()
longitudinal_grouping_scheme_results.to_csv(
    os.path.join(_GROUPING_COMPARISON_DIR, "longitudinal_grouping_scheme_feature_results.csv"),
    index=False,
)

longitudinal_grouping_scheme_summary = (
    longitudinal_grouping_scheme_results
    .groupby(["grouping_scheme", "sample", "analysis", "feature_domain"], dropna=False)
    .agg(
        n_features=("feature", "nunique"),
        n_ok=("ok", "sum"),
        n_group_sig=("group_sig", "sum"),
        n_time_sig=("time_sig", "sum"),
        n_interaction_sig=("interaction_sig", "sum"),
        median_interaction_q=("interaction_q", "median"),
        min_interaction_q=("interaction_q", "min"),
        median_n_groups=("n_groups", "median"),
    )
    .reset_index()
)
longitudinal_grouping_scheme_summary["interaction_sig_rate"] = (
    longitudinal_grouping_scheme_summary["n_interaction_sig"]
    / longitudinal_grouping_scheme_summary["n_ok"].replace(0, np.nan)
)
longitudinal_grouping_scheme_summary.to_csv(
    os.path.join(_GROUPING_COMPARISON_DIR, "longitudinal_grouping_scheme_summary.csv"),
    index=False,
)

_feature_comparison_rows = []
for keys, group in longitudinal_grouping_scheme_results.groupby(["sample", "feature_domain", "feature"], dropna=False):
    row = dict(zip(["sample", "feature_domain", "feature"], keys))
    for grouping_scheme, scheme_group in group.groupby("grouping_scheme"):
        best = scheme_group.sort_values("interaction_q", na_position="last").iloc[0]
        row[f"{grouping_scheme}_analysis"] = best["analysis"]
        row[f"{grouping_scheme}_n_groups"] = best["n_groups"]
        row[f"{grouping_scheme}_group_q"] = best["group_q"]
        row[f"{grouping_scheme}_time_q"] = best["time_q"]
        row[f"{grouping_scheme}_interaction_q"] = best["interaction_q"]
        row[f"{grouping_scheme}_interaction_sig"] = bool(best["interaction_sig"])
    _feature_comparison_rows.append(row)
longitudinal_grouping_scheme_feature_comparison = pd.DataFrame(_feature_comparison_rows)
longitudinal_grouping_scheme_feature_comparison.to_csv(
    os.path.join(_GROUPING_COMPARISON_DIR, "longitudinal_grouping_scheme_feature_comparison.csv"),
    index=False,
)

_replication_rows = []
for keys, group in longitudinal_grouping_scheme_results.groupby(["grouping_scheme", "feature_domain", "feature"], dropna=False):
    grouping_scheme, feature_domain, feature = keys
    discovery = group.loc[group["sample"].eq("discovery")].sort_values("interaction_q", na_position="last")
    validation = group.loc[group["sample"].eq("validation")].sort_values("interaction_q", na_position="last")
    if discovery.empty or validation.empty:
        continue
    discovery_row = discovery.iloc[0]
    validation_row = validation.iloc[0]
    _replication_rows.append({
        "grouping_scheme": grouping_scheme,
        "feature_domain": feature_domain,
        "feature": feature,
        "discovery_analysis": discovery_row["analysis"],
        "validation_analysis": validation_row["analysis"],
        "discovery_interaction_q": discovery_row["interaction_q"],
        "validation_interaction_q": validation_row["interaction_q"],
        "discovery_interaction_sig": bool(discovery_row["interaction_sig"]),
        "validation_interaction_sig": bool(validation_row["interaction_sig"]),
        "replicated_interaction": bool(discovery_row["interaction_sig"] and validation_row["interaction_sig"]),
    })
longitudinal_grouping_scheme_replication = pd.DataFrame(_replication_rows)
longitudinal_grouping_scheme_replication.to_csv(
    os.path.join(_GROUPING_COMPARISON_DIR, "longitudinal_grouping_scheme_replication.csv"),
    index=False,
)

longitudinal_grouping_scheme_replication_summary = (
    longitudinal_grouping_scheme_replication
    .groupby(["grouping_scheme", "feature_domain"], dropna=False)
    .agg(
        n_features_compared=("feature", "nunique"),
        n_discovery_interactions=("discovery_interaction_sig", "sum"),
        n_validation_interactions=("validation_interaction_sig", "sum"),
        n_replicated_interactions=("replicated_interaction", "sum"),
    )
    .reset_index()
)
longitudinal_grouping_scheme_replication_summary["replicated_interaction_rate"] = (
    longitudinal_grouping_scheme_replication_summary["n_replicated_interactions"]
    / longitudinal_grouping_scheme_replication_summary["n_features_compared"].replace(0, np.nan)
)
longitudinal_grouping_scheme_replication_summary.to_csv(
    os.path.join(_GROUPING_COMPARISON_DIR, "longitudinal_grouping_scheme_replication_summary.csv"),
    index=False,
)

print("Grouping-scheme comparison outputs saved under:", _GROUPING_COMPARISON_DIR)
print("\nInteraction summary by grouping scheme:")
display(
    longitudinal_grouping_scheme_summary[[
        "grouping_scheme", "sample", "analysis", "n_ok", "n_interaction_sig",
        "interaction_sig_rate", "median_n_groups",
    ]].sort_values(["sample", "analysis", "grouping_scheme"])
)

print("\nDiscovery-validation replication summary:")
display(
    longitudinal_grouping_scheme_replication_summary.sort_values([
        "feature_domain", "grouping_scheme",
    ])
)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
plot_data = longitudinal_grouping_scheme_replication_summary.copy()
plot_data["grouping_scheme"] = pd.Categorical(
    plot_data["grouping_scheme"],
    categories=["individual_domain_subgroups", "integrated_subgroups", "domain_map_paths"],
    ordered=True,
)
sns.barplot(
    data=plot_data,
    x="feature_domain",
    y="n_replicated_interactions",
    hue="grouping_scheme",
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("Replicated group-by-time features")
ax.set_title("Longitudinal mixed-model findings by grouping scheme")
ax.tick_params(axis="x", rotation=25)
ax.legend(title="Grouping scheme", frameon=False)
sns.despine(ax=ax)
fig.tight_layout()
comparison_plot = os.path.join(_GROUPING_COMPARISON_DIR, "longitudinal_grouping_scheme_replicated_interactions.png")
Utils._save_longitudinal_matplotlib_image(fig, comparison_plot, dpi=300, bbox_inches="tight")
plt.close(fig)

# Compact feature matrix: useful for judging whether domain-map paths add unique longitudinal signal.
print("\nFeature-level comparison: replicated interactions by grouping scheme")
replicated_feature_matrix = (
    longitudinal_grouping_scheme_replication
    .pivot_table(
        index=["feature_domain", "feature"],
        columns="grouping_scheme",
        values="replicated_interaction",
        aggfunc="max",
        fill_value=False,
    )
    .reset_index()
)
for col in ["individual_domain_subgroups", "integrated_subgroups", "domain_map_paths"]:
    if col not in replicated_feature_matrix.columns:
        replicated_feature_matrix[col] = False
replicated_feature_matrix.to_csv(
    os.path.join(_GROUPING_COMPARISON_DIR, "longitudinal_grouping_scheme_replicated_feature_matrix.csv"),
    index=False,
)
display(replicated_feature_matrix)

# Check conversion with domain maps

## Import and merge conversion data

In [ ]:
conv_prescient = pd.read_csv("path/to/restricted_data/conversion/Prescient-Conversion-11-May-2026.csv")
conv_pronet = pd.read_csv("path/to/restricted_data/conversion/ProNETPsychosisRiskO-ConversoinFloatingFo_DATA_2026-04-27_1629 1.csv")

In [ ]:
# Prescient rows are already a list of converted subjects.
prescient_conversion_subjects = set(conv_prescient["subjectkey"])

# ProNET contains conversion-form rows; use consensus_outcome == 1 as confirmed conversion.
# If you want every ProNET conversion-form row instead, use set(conv_pronet["chric_record_id"]).
pronet_conversion_subjects_all_forms = set(conv_pronet["chric_record_id"])
pronet_conversion_subjects_confirmed = set(
    conv_pronet.loc[
        pd.to_numeric(conv_pronet["chrconv_consensus_outcome"], errors="coerce").eq(1),
        "chric_record_id",
    ]
)

print(f"Raw Prescient conversion rows: {len(prescient_conversion_subjects)}")
print(f"Raw ProNET conversion-form rows: {len(pronet_conversion_subjects_all_forms)}")
print(f"Raw ProNET confirmed conversion rows: {len(pronet_conversion_subjects_confirmed)}")


In [ ]:



subject_ids_discovery_conversion = final_metrics["data"][stage_order[0]]["src_subject_id"].reset_index(drop=True)
fig_conversion_discovery, conversion_labels_discovery = clinical_analysis_plot_conversion_domain_map(
    sample_name="discovery",
    labels_by_modality=new_labels_by_modality,
    subject_ids=subject_ids_discovery_conversion,
    conversion_subjects=prescient_conversion_subjects,
    save_file_name="Parcats_by_conversion_discovery.pdf",
)

subject_ids_validation_conversion = subject_ids_test if "subject_ids_test" in globals() else dict_final[stage_order[0]]["src_subject_id"].reset_index(drop=True)
fig_conversion_validation, conversion_labels_validation = clinical_analysis_plot_conversion_domain_map(
    sample_name="validation",
    labels_by_modality=new_test_labels_by_modality,
    subject_ids=subject_ids_validation_conversion,
    conversion_subjects=pronet_conversion_subjects_confirmed,
    save_file_name="Parcats_by_conversion_validation.pdf",
)


## Predict conversion from clinical profile mapping


In [ ]:
from sklearn.metrics import (
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    roc_auc_score,
)


svm_conversion_discovery_df = clinical_analysis_build_profile_predictor_df(
    labels_by_modality=new_labels_by_modality,
    subject_ids=subject_ids_discovery_conversion,
    conversion_labels=conversion_labels_discovery,
)
svm_conversion_validation_df = clinical_analysis_build_profile_predictor_df(
    labels_by_modality=new_test_labels_by_modality,
    subject_ids=subject_ids_validation_conversion,
    conversion_labels=conversion_labels_validation,
)

categorical_predictors = stage_order + ["clinical_profile"]
numeric_predictors = ["n_high_severity_domains"]


X_discovery = clinical_analysis_encode_profile_predictors(svm_conversion_discovery_df)
y_discovery = svm_conversion_discovery_df["converted_to_psychosis"]
X_validation = clinical_analysis_encode_profile_predictors(
    svm_conversion_validation_df,
    reference_columns=list(X_discovery.columns),
)
y_validation = svm_conversion_validation_df["converted_to_psychosis"]

print("Discovery conversion class counts:")
print(y_discovery.value_counts())
print("\nValidation conversion class counts:")
print(y_validation.value_counts())

if y_discovery.nunique() < 2:
    raise ValueError("Discovery conversion labels contain fewer than two classes; cannot train conversion SVM.")

svm_conversion_results, conversion_svm = SVM_nested_cv(X_discovery, y_discovery)
svm_conversion_cv_summary = pd.DataFrame({
    "mean": svm_conversion_results["mean_metrics"],
    "std": svm_conversion_results["std_metrics"],
})
print("\nClinical-profile discovery nested-CV summary:")
display(svm_conversion_cv_summary)

validation_pred = conversion_svm.predict(X_validation)
yes_class_index = list(conversion_svm.classes_).index("yes")
validation_score = conversion_svm.predict_proba(X_validation)[:, yes_class_index]

svm_conversion_validation_predictions = svm_conversion_validation_df[["src_subject_id"] + stage_order + ["clinical_profile", "converted_to_psychosis"]].copy()
svm_conversion_validation_predictions["predicted_conversion"] = validation_pred
svm_conversion_validation_predictions["p_conversion"] = validation_score

svm_conversion_validation_metrics = {
    "balanced_accuracy": balanced_accuracy_score(y_validation, validation_pred),
    "sensitivity_yes": recall_score(y_validation, validation_pred, pos_label="yes", zero_division=0),
    "precision_yes": precision_score(y_validation, validation_pred, pos_label="yes", zero_division=0),
    "majority_baseline_balanced_accuracy": balanced_accuracy_score(
        y_validation,
        np.repeat(y_discovery.value_counts().idxmax(), len(y_validation)),
    ),
}
if y_validation.nunique() == 2:
    svm_conversion_validation_metrics["roc_auc"] = roc_auc_score((y_validation == "yes").astype(int), validation_score)

svm_conversion_validation_metrics = pd.Series(svm_conversion_validation_metrics, name="validation_metric")
svm_conversion_validation_report = pd.DataFrame(
    classification_report(y_validation, validation_pred, output_dict=True, zero_division=0)
).T
svm_conversion_validation_confusion = pd.DataFrame(
    confusion_matrix(y_validation, validation_pred, labels=["no", "yes"]),
    index=["true_no", "true_yes"],
    columns=["pred_no", "pred_yes"],
)

print("\nValidation metrics:")
display(svm_conversion_validation_metrics.to_frame())
print("\nValidation confusion matrix:")
display(svm_conversion_validation_confusion)
print("\nValidation classification report:")
display(svm_conversion_validation_report)

svm_conversion_outfile = os.path.join(plots_dir, "SVM_conversion_from_profile_validation_predictions.csv")
svm_conversion_validation_predictions.to_csv(svm_conversion_outfile, index=False)
print("Saved validation predictions to:", svm_conversion_outfile)


## Predict conversion from preprocessed clinical variables


In [ ]:


svm_preprocessed_discovery_df = clinical_analysis_build_preprocessed_conversion_feature_df(
    data_by_modality=final_metrics["data"],
    subject_ids=subject_ids_discovery_conversion,
    conversion_labels=conversion_labels_discovery,
    sample_name="discovery",
)
svm_preprocessed_validation_df = clinical_analysis_build_preprocessed_conversion_feature_df(
    data_by_modality=dict_final,
    subject_ids=subject_ids_validation_conversion,
    conversion_labels=conversion_labels_validation,
    sample_name="validation",
)

preprocessed_feature_cols = [
    col for col in svm_preprocessed_discovery_df.columns
    if col not in {"src_subject_id", "converted_to_psychosis"}
]
missing_validation_cols = sorted(set(preprocessed_feature_cols) - set(svm_preprocessed_validation_df.columns))
extra_validation_cols = sorted(set(svm_preprocessed_validation_df.columns) - set(preprocessed_feature_cols) - {"src_subject_id", "converted_to_psychosis"})
if missing_validation_cols:
    raise ValueError(f"Validation is missing preprocessed feature columns, examples: {missing_validation_cols[:10]}")
if extra_validation_cols:
    print(f"Validation has {len(extra_validation_cols)} extra preprocessed columns not used by discovery training.")

X_preprocessed_discovery = svm_preprocessed_discovery_df[preprocessed_feature_cols]
y_preprocessed_discovery = svm_preprocessed_discovery_df["converted_to_psychosis"]
X_preprocessed_validation = svm_preprocessed_validation_df[preprocessed_feature_cols]
y_preprocessed_validation = svm_preprocessed_validation_df["converted_to_psychosis"]

print("Preprocessed-variable SVM feature matrix shapes:")
print("discovery:", X_preprocessed_discovery.shape)
print("validation:", X_preprocessed_validation.shape)
print("\nDiscovery conversion class counts:")
print(y_preprocessed_discovery.value_counts())
print("\nValidation conversion class counts:")
print(y_preprocessed_validation.value_counts())

if y_preprocessed_discovery.nunique() < 2:
    raise ValueError("Discovery conversion labels contain fewer than two classes; cannot train preprocessed-variable conversion SVM.")

svm_preprocessed_results, preprocessed_conversion_svm = SVM_nested_cv(
    X_preprocessed_discovery,
    y_preprocessed_discovery,
)
svm_preprocessed_conversion_cv_summary = pd.DataFrame({
    "mean": svm_preprocessed_results["mean_metrics"],
    "std": svm_preprocessed_results["std_metrics"],
})
print("\nPreprocessed-variable discovery nested-CV summary:")
display(svm_preprocessed_conversion_cv_summary)

preprocessed_oof = svm_preprocessed_results["oof_uncertainty"].copy()
svm_preprocessed_discovery_converter_metrics = pd.Series(
    {
        "sensitivity_yes": recall_score(
            preprocessed_oof["y_true"],
            preprocessed_oof["y_pred"],
            pos_label="yes",
            zero_division=0,
        ),
        "precision_yes": precision_score(
            preprocessed_oof["y_true"],
            preprocessed_oof["y_pred"],
            pos_label="yes",
            zero_division=0,
        ),
    },
    name="discovery_oof_metric",
)
print("\nPreprocessed-variable discovery converter metrics from out-of-fold predictions:")
display(svm_preprocessed_discovery_converter_metrics.to_frame())

preprocessed_validation_pred = preprocessed_conversion_svm.predict(X_preprocessed_validation)
yes_class_index = list(preprocessed_conversion_svm.classes_).index("yes")
preprocessed_validation_score = preprocessed_conversion_svm.predict_proba(X_preprocessed_validation)[:, yes_class_index]

svm_preprocessed_conversion_validation_predictions = svm_preprocessed_validation_df[["src_subject_id", "converted_to_psychosis"]].copy()
svm_preprocessed_conversion_validation_predictions["predicted_conversion"] = preprocessed_validation_pred
svm_preprocessed_conversion_validation_predictions["p_conversion"] = preprocessed_validation_score

svm_preprocessed_conversion_validation_metrics = {
    "balanced_accuracy": balanced_accuracy_score(y_preprocessed_validation, preprocessed_validation_pred),
    "sensitivity_yes": recall_score(y_preprocessed_validation, preprocessed_validation_pred, pos_label="yes", zero_division=0),
    "precision_yes": precision_score(y_preprocessed_validation, preprocessed_validation_pred, pos_label="yes", zero_division=0),
    "majority_baseline_balanced_accuracy": balanced_accuracy_score(
        y_preprocessed_validation,
        np.repeat(y_preprocessed_discovery.value_counts().idxmax(), len(y_preprocessed_validation)),
    ),
}
if y_preprocessed_validation.nunique() == 2:
    svm_preprocessed_conversion_validation_metrics["roc_auc"] = roc_auc_score(
        (y_preprocessed_validation == "yes").astype(int),
        preprocessed_validation_score,
    )

svm_preprocessed_conversion_validation_metrics = pd.Series(
    svm_preprocessed_conversion_validation_metrics,
    name="validation_metric",
)
svm_preprocessed_conversion_validation_report = pd.DataFrame(
    classification_report(y_preprocessed_validation, preprocessed_validation_pred, output_dict=True, zero_division=0)
).T
svm_preprocessed_conversion_validation_confusion = pd.DataFrame(
    confusion_matrix(y_preprocessed_validation, preprocessed_validation_pred, labels=["no", "yes"]),
    index=["true_no", "true_yes"],
    columns=["pred_no", "pred_yes"],
)

print("\nPreprocessed-variable validation metrics:")
display(svm_preprocessed_conversion_validation_metrics.to_frame())
print("\nPreprocessed-variable validation confusion matrix:")
display(svm_preprocessed_conversion_validation_confusion)
print("\nPreprocessed-variable validation classification report:")
display(svm_preprocessed_conversion_validation_report)

svm_preprocessed_conversion_outfile = os.path.join(plots_dir, "SVM_conversion_from_preprocessed_clinical_variables_validation_predictions.csv")
svm_preprocessed_conversion_validation_predictions.to_csv(svm_preprocessed_conversion_outfile, index=False)
print("Saved preprocessed-variable validation predictions to:", svm_preprocessed_conversion_outfile)


# Save dataframe with subject IDs and labels

In [ ]:
# Save dataframe with subject ids and all labels discovery

## Create dataframe with subjects ids and all labels for discovery
subject_ids_disc = final_metrics['data'][stage_order[0]]['src_subject_id'].astype(str).reset_index(drop=True)
df_disc_labels = pd.DataFrame({"src_subject_id": subject_ids_disc})
for stage in stage_order:
    df_disc_labels[stage] = pd.Series(new_labels_by_modality[stage]).astype(str).reset_index(drop=True)
df_disc_labels["final"] = pd.Series(final_metrics["final_labels"]).astype(str).reset_index(drop=True) 

## Create dataframe with subjects ids and all labels for test
subject_ids_test = dict_final[stage_order[0]]['src_subject_id'].astype(str).reset_index(drop=True)
df_test_labels = pd.DataFrame({"src_subject_id": subject_ids_test})
for stage in stage_order:
    df_test_labels[stage] = pd.Series(new_test_labels_by_modality[stage]).astype(str).reset_index(drop=True)
df_test_labels["final"] = pd.Series(labels_test_final).astype(str).reset_index(drop=True) 

## Save to CSV files
df_disc_labels.to_csv("path/to/results/study_1/release/Labels/discovery_labels_clin_multiclust.csv", index=False)
df_test_labels.to_csv("path/to/results/study_1/release/Labels/test_labels_clin_multiclust.csv", index=False)
